In [12]:
!pip install -q \
utilsforecast \
statsforecast \
holidays \
hyperopt \
category_encoders \
sktime \
shap \
lightgbm \
catboost \
xgboost \
plotly \
tqdm

# Загрузка библиотек

In [13]:
# Базовые библиотеки и утилиты
import numpy as np
import pandas as pd
import datetime
import os
import time
import time as time_module
import warnings
from copy import deepcopy
from bisect import bisect_left
from itertools import product
from typing import Union, Optional

# Настройки pandas
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.max_columns', None)

# Отображение в Jupyter
from IPython.display import display
%matplotlib inline

# Визуализация
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
from utilsforecast.plotting import plot_series

# Работа с календарём
import holidays

# Прогресс-бары
from tqdm import tqdm

# SciPy
import scipy as sp
from scipy import stats
from scipy.stats import ttest_1samp, shapiro, ks_2samp, zscore
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# Statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.datasets import longley
from statsmodels.formula.api import ols
from statsmodels.graphics import tsaplots
from statsmodels.graphics.gofplots import qqplot
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.tsa.stattools import acf, adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.ar_model import ar_select_order
from statsmodels.tsa.arima.model import ARIMA

# StatsForecast
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, _TS

# Scikit-learn: модели, препроцессинг, метрики
from sklearn.cluster import DBSCAN
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, mean_absolute_error, mean_squared_error, precision_score,
    r2_score, recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import (
    GridSearchCV, ParameterGrid, RandomizedSearchCV,
    TimeSeriesSplit, train_test_split
)
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Boosting модели
from catboost import CatBoostRegressor
import catboost as cb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier, XGBRegressor
import xgboost as xgb

# Гиперпараметрический поиск
from hyperopt import fmin, tpe, Trials, STATUS_OK, hp

# Кодировщики категориальных признаков
import category_encoders as ce

# Дополнительные утилиты
from sktime.utils.plotting import plot_correlations

# SHAP
import shap

In [14]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/kaggle/input/datasets/jamesslax/train-2/train_2.csv


In [15]:
df = pd.read_csv('/kaggle/input/datasets/jamesslax/train-2/train_2.csv')

In [16]:
df.head()

,new_id,Год,Месяц,Среднее количество промо товаров в чеке,Среднее количество товаров в чеке,Среднее количество отмен,Рабочие часы в день,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,Регион,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м),Количество касс,Флаг алкогольной лицензии,РТО
0,0,2024,1,1.080,6.030,147.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,75147744.850
1,0,2023,1,1.320,6.040,162.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,74914754.220
2,0,2025,1,0.820,6.000,145.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,87125506.920
3,0,2025,2,0.900,6.000,118.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,82659801.630
4,0,2024,2,1.250,6.060,154.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,74209339.110


# Функции создания признаков
# Подготовка

In [17]:
rename_cols = {
    "new_id": "new_id",
    "Год": "year",
    "Месяц": "month",
    "Среднее количество промо товаров в чеке": "avg_promo_items_per_receipt",
    "Среднее количество товаров в чеке": "avg_items_per_receipt",
    "Среднее количество отмен": "avg_cancellations",
    "Рабочие часы в день": "working_hours_per_day",
    "Дата открытия, категориальный": "opening_date_category",
    "Торговая площадь, категориальный": "trading_area_category",
    "Населенный пункт": "settlement",
    "Регион": "region",
    "Численность населения": "population_size",
    "Количество домохозяйств": "number_of_households",
    "Трафик пеший, в час": "pedestrian_traffic_per_hour",
    "Трафик авто, в час": "car_traffic_per_hour",
    "Маркетплейсы, доставки, постаматы (100 м)": "marketplaces_deliveries_parcel_lockers_100m",
    "Медицинские уч. и аптеки (300 м)": "medical_institutions_and_pharmacies_300m",
    "Школы (300 м)": "schools_300m",
    "Остановки (300 м)": "public_transport_stops_300m",
    "Продуктовые магазины (500 м)": "grocery_stores_500m",
    "Пятерочки (500 м)": "pyaterochka_stores_500m",
    "Количество касс": "number_of_cash_registers",
    "Флаг алкогольной лицензии": "alcohol_license_flag",
    "РТО": "pto",
}

df = df.rename(columns=rename_cols)

# Функция для создания признаков и сплита (без кросс-валидации)

In [18]:
from typing import Literal, Optional, Sequence

import gc
import warnings
import numpy as np
import pandas as pd


def make_panel_time_split(
    df: pd.DataFrame,
    *,
    group_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    split_mode: Literal["train_val", "train_val_test", "train_test"] = "train_val",
    val_time_idx: Optional[int] = None,
    val_year: Optional[int] = None,
    val_month: Optional[int] = None,
    test_time_idx: Optional[int] = None,
    test_year: Optional[int] = None,
    test_month: Optional[int] = None,
    min_year: Optional[int] = None,
    create_time_col: bool = True,
    create_test_if_missing: bool = True,
    test_target_value=np.nan,
    future_unknown_cols: Optional[Sequence[str]] = None,
    drop_target_na_from_train: bool = True,
    require_known_val_target: bool = True,
    check_duplicates: bool = True,
    validate_time_col: bool = True,
    expected_test_size: Optional[int] = None,
    copy: bool = True,
    verbose: bool = True,
):
    """
    Leakage-safe time split для panel monthly forecasting.

    split_mode:
    - "train_val": train = months before val, val = selected validation month;
    - "train_val_test": train + val + artificial/existing test month;
    - "train_test": train = all known history before test, test = artificial/existing test month.

    Если test-месяца нет в df, он создаётся из последней известной строки каждого магазина.
    Важно: если дальше строятся лаги/rolling, test-строки должны быть созданы ДО feature engineering.
    """
    if split_mode not in {"train_val", "train_val_test", "train_test"}:
        raise ValueError("split_mode должен быть 'train_val', 'train_val_test' или 'train_test'.")

    if copy:
        df = df.copy()

    def _check_time_args(prefix, time_idx, year, month):
        if time_idx is not None and (year is not None or month is not None):
            raise ValueError(
                f"Для {prefix} передай либо {prefix}_time_idx, либо пару "
                f"{prefix}_year + {prefix}_month, но не оба варианта."
            )
        if (year is None) ^ (month is None):
            raise ValueError(
                f"Для {prefix} нужно передавать year и month вместе. "
                f"Получено: {prefix}_year={year}, {prefix}_month={month}."
            )

    _check_time_args("val", val_time_idx, val_year, val_month)
    _check_time_args("test", test_time_idx, test_year, test_month)

    if split_mode == "train_test" and (
        val_time_idx is not None or val_year is not None or val_month is not None
    ):
        raise ValueError("В split_mode='train_test' validation-месяц не используется.")

    required_cols = {group_col, target_col}
    has_time_col = time_col in df.columns
    has_year_month = year_col in df.columns and month_col in df.columns

    if not has_time_col:
        if create_time_col:
            required_cols.update({year_col, month_col})
        else:
            required_cols.add(time_col)

    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"В df отсутствуют обязательные колонки: {sorted(missing)}")

    if has_year_month:
        if df[year_col].isna().any() or df[month_col].isna().any():
            raise ValueError(f"В {year_col}/{month_col} есть пропуски.")
        df[year_col] = pd.to_numeric(df[year_col], errors="raise").astype(int)
        df[month_col] = pd.to_numeric(df[month_col], errors="raise").astype(int)
        bad_months = sorted(set(df[month_col].unique()) - set(range(1, 13)))
        if bad_months:
            raise ValueError(f"В колонке '{month_col}' некорректные месяцы: {bad_months}.")

    if has_time_col:
        if df[time_col].isna().any():
            raise ValueError(f"В колонке '{time_col}' есть пропуски.")
        df[time_col] = pd.to_numeric(df[time_col], errors="raise").astype(int)

        if min_year is None and has_year_month:
            inferred = df[year_col] - ((df[time_col] - (df[month_col] - 1)) // 12)
            unique = sorted(inferred.unique())
            if len(unique) == 1:
                min_year = int(unique[0])
            else:
                raise ValueError(
                    f"Не удалось однозначно восстановить min_year из {time_col}. "
                    f"Передай min_year явно. Найдено: {unique[:10]}"
                )

        if validate_time_col and has_year_month and min_year is not None:
            expected = ((df[year_col] - int(min_year)) * 12 + (df[month_col] - 1)).astype(int)
            bad = df[time_col].astype(int) != expected
            if bad.any():
                examples = df.loc[bad, [group_col, year_col, month_col, time_col]].head(10)
                raise ValueError(
                    f"'{time_col}' не совпадает с расчётом по min_year={min_year}. "
                    f"Примеры:\n{examples}"
                )
    else:
        if not has_year_month:
            raise ValueError(f"Нельзя создать '{time_col}', потому что нет year/month.")
        if min_year is None:
            min_year = int(df[year_col].min())
        df[time_col] = ((df[year_col] - int(min_year)) * 12 + (df[month_col] - 1)).astype(int)

    known_target_times = sorted(df.loc[df[target_col].notna(), time_col].unique())
    if not known_target_times:
        raise ValueError(f"В '{target_col}' нет известных значений.")
    max_known_time_idx = int(known_target_times[-1])

    def _resolve(label, time_idx, year, month, default=None):
        if time_idx is not None:
            return int(time_idx)
        if year is not None and month is not None:
            if min_year is None:
                raise ValueError(f"Для расчёта {label}_time_idx нужен min_year.")
            month = int(month)
            if not 1 <= month <= 12:
                raise ValueError(f"{label}_month должен быть от 1 до 12.")
            return (int(year) - int(min_year)) * 12 + (month - 1)
        return None if default is None else int(default)

    if split_mode in {"train_val", "train_val_test"}:
        val_time_idx = _resolve("val", val_time_idx, val_year, val_month, max_known_time_idx)
    else:
        val_time_idx = None

    if split_mode in {"train_val_test", "train_test"}:
        test_time_idx = _resolve("test", test_time_idx, test_year, test_month, max_known_time_idx + 1)
    else:
        test_time_idx = None

    if split_mode == "train_val_test" and not (val_time_idx < test_time_idx):
        raise ValueError(
            f"Ожидается val_time_idx < test_time_idx, получено "
            f"{val_time_idx=} и {test_time_idx=}."
        )

    df = df.sort_values([group_col, time_col]).reset_index(drop=True)

    if check_duplicates:
        dup_mask = df.duplicated([group_col, time_col], keep=False)
        if dup_mask.any():
            examples = df.loc[dup_mask, [group_col, time_col]].sort_values([group_col, time_col]).head(10)
            raise ValueError(
                f"Есть дубли по [{group_col}, {time_col}]. "
                f"Строк-дублей: {int(dup_mask.sum())}. Примеры:\n{examples}"
            )

    test = None
    if split_mode in {"train_val_test", "train_test"}:
        existing = df[df[time_col] == test_time_idx].copy()
        if not existing.empty:
            test = existing.copy()
        else:
            if not create_test_if_missing:
                raise ValueError(f"В df нет test_time_idx={test_time_idx}.")
            hist = df[(df[time_col] < test_time_idx) & (df[target_col].notna())].copy()
            if hist.empty:
                raise ValueError(f"Нет исторических строк до test_time_idx={test_time_idx}.")
            test = (
                hist.sort_values([group_col, time_col])
                .groupby(group_col, sort=False)
                .tail(1)
                .copy()
            )
            test[time_col] = int(test_time_idx)
            if has_year_month and min_year is not None:
                test[year_col] = int(min_year) + int(test_time_idx) // 12
                test[month_col] = int(test_time_idx) % 12 + 1
            test[target_col] = test_target_value

            if future_unknown_cols is not None:
                for col in future_unknown_cols:
                    if col in test.columns:
                        test[col] = np.nan
                    else:
                        warnings.warn(f"Колонка '{col}' из future_unknown_cols отсутствует в df.")

        test = test.sort_values([group_col, time_col]).reset_index(drop=True)
        if expected_test_size is not None and len(test) != int(expected_test_size):
            raise ValueError(
                f"Размер test не совпадает с expected_test_size: "
                f"получено {len(test)}, ожидалось {expected_test_size}."
            )

    if split_mode == "train_test":
        train_mask = df[time_col] < test_time_idx
    else:
        train_mask = df[time_col] < val_time_idx

    if drop_target_na_from_train:
        train_mask = train_mask & df[target_col].notna()

    train = df.loc[train_mask].copy()
    if train.empty:
        raise ValueError("train получился пустым.")

    val = None
    if split_mode in {"train_val", "train_val_test"}:
        val = df[df[time_col] == val_time_idx].copy()
        if val.empty:
            available = sorted(df[time_col].unique())
            raise ValueError(
                f"val пустой для val_time_idx={val_time_idx}. "
                f"Доступные time_idx: {available[:5]} ... {available[-5:]}"
            )
        if require_known_val_target and val[target_col].isna().any():
            n_nan = int(val[target_col].isna().sum())
            raise ValueError(
                f"В validation есть NaN в '{target_col}': {n_nan}. "
                "Скорее всего, validation попал на искусственный test-месяц."
            )
        val = val.sort_values([group_col, time_col]).reset_index(drop=True)

    train = train.sort_values([group_col, time_col]).reset_index(drop=True)

    if verbose:
        print("split_mode:", split_mode)
        print("min_year:", min_year)
        print("max_known_time_idx:", max_known_time_idx)
        if val_time_idx is not None:
            print("val_time_idx:", val_time_idx)
        if test_time_idx is not None:
            print("test_time_idx:", test_time_idx)
        print("train shape:", train.shape)
        if val is not None:
            print("val shape:", val.shape)
        if test is not None:
            print("test shape:", test.shape)

    if split_mode == "train_val":
        return train, val
    if split_mode == "train_test":
        return train, test
    return train, val, test




# ============================================================
# Base feature generator from uploaded file
# ============================================================

import gc
import numpy as np
import pandas as pd


def generate_retail_forecast_features_leakfree(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame | None = None,
    test_df: pd.DataFrame | None = None,
    *,
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    min_year: int = 2023,
    make_val_from_train: bool = False,
    val_time_idx: int | None = None,
    use_val_as_test_history: bool = True,
    add_calendar_features: bool = True,
    add_static_features: bool = True,
    add_target_history: bool = True,
    add_category_target_history: bool = True,
    add_non_target_monthly_history: bool = True,
    include_current_receipt_features: bool = False,
    current_monthly_cols: tuple[str, ...] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    current_monthly_derived_cols: tuple[str, ...] = (
        "promo_items_share",
        "promo_x_items",
        "items_x_working_hours",
        "items_x_working_days",
        "alcohol_x_items",
    ),
    working_hours_future_known: bool = False,
    target_lags: tuple[int, ...] = (1, 2, 3, 6, 12, 13),
    target_rolling_windows: tuple[int, ...] = (3, 12),
    target_rolling_min_periods: int = 2,
    target_history_scale: str = "raw",   # raw | log | both
    keep_raw_pto_lag_1_anchor_for_log: bool = True,
    add_log_target_dynamics: bool = True,
    group_specs: list | None = None,
    group_agg_stat: str = "mean",
    group_lag: int = 1,
    add_group_lag_feature: bool = True,
    add_group_relative_ratio: bool = True,
    add_group_relative_diff: bool = False,
    monthly_feature_config: dict | None = None,
    monthly_history_scale: str = "raw",  # raw | log | both
    monthly_rolling_min_periods: int = 2,
    add_monthly_cross_lags: bool = True,
    monthly_cross_lags: tuple[int, ...] = (1, 12),
    monthly_cross_feature_types: tuple[str, ...] = ("promo_share",),
    drop_current_monthly_cols: bool = True,
    drop_current_monthly_derived_cols: bool = True,
    drop_year_col: bool = True,
    drop_obvious_duplicates: bool = True,
    extra_drop_cols: list | tuple | None = None,
    fill_history_na: bool = False,
    fill_value: float = 0.0,
    downcast_float_features: bool = True,
    copy: bool = True,
    verbose: bool = True,
) -> dict:
    """
    Единый leak-free feature-pipeline для panel monthly forecasting.

    Основной контракт:
    - val/test target не используется для лагов и group target history;
    - текущие monthly-cols из val/test не используются для лагов;
    - лаги считаются через time-aware merge по time_col - lag, а не через shift строк;
    - rolling-признаки считаются только по прошлым наблюдениям внутри history;
    - для test можно использовать train+val как историю, если val уже является известным прошлым.
    """

    if target_history_scale not in {"raw", "log", "both"}:
        raise ValueError("target_history_scale должен быть 'raw', 'log' или 'both'.")
    if monthly_history_scale not in {"raw", "log", "both"}:
        raise ValueError("monthly_history_scale должен быть 'raw', 'log' или 'both'.")
    if group_agg_stat not in {"mean", "median"}:
        raise ValueError("group_agg_stat должен быть 'mean' или 'median'.")

    if copy:
        train_df = train_df.copy()
        val_df = val_df.copy() if val_df is not None else None
        test_df = test_df.copy() if test_df is not None else None

    # ============================================================
    # Helpers
    # ============================================================

    def _safe_divide(numerator, denominator, fill=np.nan):
        numerator = numerator if isinstance(numerator, pd.Series) else pd.Series(numerator)
        denominator = denominator if isinstance(denominator, pd.Series) else pd.Series(denominator)
        result = numerator / denominator.replace(0, np.nan)
        result = result.replace([np.inf, -np.inf], np.nan)
        if fill is not None and not (isinstance(fill, float) and np.isnan(fill)):
            result = result.fillna(fill)
        return result

    def _col(df, name, default=0.0):
        if name in df.columns:
            return df[name].fillna(default)
        return pd.Series(default, index=df.index)

    def _ensure_time_year_month(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
        df = df.copy()

        has_year_month = year_col in df.columns and month_col in df.columns
        has_time = time_col in df.columns

        if not has_year_month and not has_time:
            raise ValueError(
                f"В {name} нет ни пары '{year_col}'+'{month_col}', ни '{time_col}'."
            )

        if has_year_month:
            if df[year_col].isna().any() or df[month_col].isna().any():
                raise ValueError(f"В {name} есть NA в {year_col}/{month_col}.")
            df[year_col] = df[year_col].astype(int)
            df[month_col] = df[month_col].astype(int)
            bad_months = sorted(set(df[month_col].unique()) - set(range(1, 13)))
            if bad_months:
                raise ValueError(f"В {name} некорректные месяцы: {bad_months}.")
            # Важно: всегда пересчитываем глобальную шкалу, чтобы не было local-min-year.
            df[time_col] = ((df[year_col] - min_year) * 12 + (df[month_col] - 1)).astype(int)
        else:
            if df[time_col].isna().any():
                raise ValueError(f"В {name} есть NA в {time_col}.")
            df[time_col] = df[time_col].astype(int)
            df[year_col] = (min_year + df[time_col] // 12).astype(int)
            df[month_col] = (df[time_col] % 12 + 1).astype(int)

        return df

    def _validate_required(df: pd.DataFrame, required: set[str], name: str):
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"В {name} нет обязательных колонок: {missing}")

    def _validate_unique_panel(df: pd.DataFrame, name: str):
        if {id_col, time_col}.issubset(df.columns):
            duplicated = df.duplicated([id_col, time_col]).sum()
            if duplicated > 0:
                raise ValueError(
                    f"В {name} найдено {duplicated} дублей по [{id_col}, {time_col}]. "
                    "Для лагов ожидается одна строка на магазин-месяц."
                )

    def _part_concat(parts: dict[str, pd.DataFrame]) -> pd.DataFrame:
        out = []
        for part_name, df_part in parts.items():
            p = df_part.copy()
            p["__part__"] = part_name
            p["__row_order__"] = np.arange(len(p))
            out.append(p)
        return pd.concat(out, axis=0, ignore_index=True, sort=False)

    def _split_parts(combined: pd.DataFrame, part_names: list[str]) -> dict[str, pd.DataFrame]:
        result = {}
        for part_name in part_names:
            part = (
                combined[combined["__part__"] == part_name]
                .sort_values("__row_order__")
                .drop(columns=["__part__", "__row_order__"], errors="ignore")
                .reset_index(drop=True)
            )
            result[part_name] = part
        return result

    def _drop_preexisting_generated_cols(df: pd.DataFrame) -> pd.DataFrame:
        prefixes = [
            f"{target_col}_lag_", f"log_{target_col}_lag_", f"{target_col}_roll_",
            f"{target_col}_diff_", f"{target_col}_ratio_", f"{target_col}_seasonal_",
            f"log_{target_col}_diff_", f"{target_col}_lag_1_to_", f"{target_col}_lag_12_to_",
            "promo_items_share_lag_", "promo_x_items_lag_", "items_x_working_hours_lag_",
            "cancellations_per_item_lag_", "cancellations_per_working_hour_lag_",
            "region_", "settlement_",
        ]
        monthly_prefixes = []
        for c in current_monthly_cols:
            monthly_prefixes.extend([
                f"{c}_lag_", f"log_{c}_lag_", f"{c}_roll_", f"{c}_diff_", f"{c}_ratio_",
            ])

        explicit = {
            "season", "is_winter", "is_spring", "is_autumn", "is_quarter_start_month",
            "is_quarter_end_month", "is_halfyear_start_month", "is_halfyear_end_month",
            "is_year_start_month", "is_year_end_month", "month_in_halfyear",
            "month_name", "month_sin", "month_cos", "month_sin_2", "month_cos_2",
            "prev_month", "n_days_in_month", "n_weekend_days", "n_may_holiday_days",
            "n_single_public_holiday_days", "n_november_holiday_days",
            "n_december_additional_holiday_days", "n_holiday_days", "n_additional_holiday_days",
            "n_working_days", "n_additional_working_days", "is_leap_year", "holiday_month_type",
            "is_middle_of_quarter", "is_pre_new_year_month", "is_pre_may_holidays_month",
            "is_back_to_school_month", "is_business_activity_low_month", "is_business_activity_high_month",
            "total_traffic_per_hour", "pedestrian_traffic_share", "car_to_pedestrian_traffic_ratio",
            "log_population_size", "log_number_of_households", "log_pedestrian_traffic_per_hour",
            "log_car_traffic_per_hour", "is_moscow", "is_million_plus_city", "population_city_type",
            "has_school_nearby", "has_transport_nearby", "has_marketplace_nearby",
            "has_medical_nearby", "has_grocery_competition", "has_pyaterochka_nearby",
            "is_high_grocery_competition", "is_high_pyaterochka_density",
            "non_pyaterochka_grocery_stores_500m", "pyaterochka_share_among_grocery",
            "non_pyaterochka_competition_share", "grocery_per_1000_people",
            "pyaterochka_per_1000_people", "competition_per_1000_households",
            "pyaterochka_per_1000_households", "people_per_household",
            "households_per_1000_people", "traffic_per_1000_people", "traffic_per_1000_households",
            "pedestrian_traffic_per_1000_people", "social_infra_score", "commercial_infra_score",
            "poi_score", "poi_per_1000_households", "public_transport_share_in_social_infra",
            "medical_share_in_social_infra", "cash_register_hours_capacity", "traffic_per_cash_register",
            "traffic_per_working_hour", "households_per_cash_register", "population_per_cash_register",
            "cash_registers_per_total_traffic", "promo_items_share", "promo_x_items",
            "items_x_working_hours", "items_x_working_days", "alcohol_x_capacity",
            "alcohol_x_total_traffic", "alcohol_x_items", "traffic_x_total_poi",
            "traffic_x_grocery_competition", "traffic_x_pyaterochka_competition",
            "pedestrian_traffic_x_transport", "schools_x_back_to_school",
            "settlement_store_count", "region_store_count", "region_trading_area_store_count",
            "region_opening_age_store_count",
        }

        drop_cols = []
        for c in df.columns:
            if c in explicit:
                drop_cols.append(c)
            elif any(c.startswith(p) for p in prefixes + monthly_prefixes):
                # Не удаляем сырые region/settlement, только generated mean/count; это грубый фильтр ниже уточняем.
                if c in {"region", "settlement"}:
                    continue
                # region_* может быть исходной колонкой только маловероятно; для безопасности удаляем только *_mean/count.
                if c.startswith(("region_", "settlement_")) and not (c.endswith("_mean") or c.endswith("_count")):
                    continue
                drop_cols.append(c)
            elif f"_{target_col}_{group_agg_stat}_lag_" in c:
                drop_cols.append(c)

        return df.drop(columns=sorted(set(drop_cols)), errors="ignore")

    # ============================================================
    # Calendar and static feature blocks
    # ============================================================

    def _add_calendar(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        years = sorted(df[year_col].dropna().astype(int).unique())
        if not years:
            return df

        def _holiday_calendar():
            holidays = {
                # 2023
                "2023-01-01": "New_Year_Holidays", "2023-01-02": "New_Year_Holidays",
                "2023-01-03": "New_Year_Holidays", "2023-01-04": "New_Year_Holidays",
                "2023-01-05": "New_Year_Holidays", "2023-01-06": "New_Year_Holidays",
                "2023-01-07": "Christmas_Day", "2023-01-08": "New_Year_Holidays",
                "2023-02-23": "Defender_of_the_Fatherland_Day",
                "2023-03-08": "International_Womens_Day",
                "2023-05-01": "Spring_and_Labour_Holiday", "2023-05-09": "Victory_Day",
                "2023-06-12": "Day_of_Russia", "2023-11-04": "National_Unity_Day",
                "2023-02-24": "Additional_Holiday", "2023-05-08": "Additional_Holiday",
                "2023-11-06": "Additional_Holiday",
                # 2024
                "2024-01-01": "New_Year_Holidays", "2024-01-02": "New_Year_Holidays",
                "2024-01-03": "New_Year_Holidays", "2024-01-04": "New_Year_Holidays",
                "2024-01-05": "New_Year_Holidays", "2024-01-06": "New_Year_Holidays",
                "2024-01-07": "Christmas_Day", "2024-01-08": "New_Year_Holidays",
                "2024-02-23": "Defender_of_the_Fatherland_Day",
                "2024-03-08": "International_Womens_Day",
                "2024-05-01": "Spring_and_Labour_Holiday", "2024-05-09": "Victory_Day",
                "2024-06-12": "Day_of_Russia", "2024-11-04": "National_Unity_Day",
                "2024-04-29": "Additional_Holiday", "2024-04-30": "Additional_Holiday",
                "2024-05-10": "Additional_Holiday", "2024-12-30": "Additional_Holiday",
                "2024-12-31": "Additional_Holiday",
                # 2025
                "2025-01-01": "New_Year_Holidays", "2025-01-02": "New_Year_Holidays",
                "2025-01-03": "New_Year_Holidays", "2025-01-04": "New_Year_Holidays",
                "2025-01-05": "New_Year_Holidays", "2025-01-06": "New_Year_Holidays",
                "2025-01-07": "Christmas_Day", "2025-01-08": "New_Year_Holidays",
                "2025-02-23": "Defender_of_the_Fatherland_Day",
                "2025-03-08": "International_Womens_Day",
                "2025-05-01": "Spring_and_Labour_Holiday", "2025-05-09": "Victory_Day",
                "2025-06-12": "Day_of_Russia", "2025-11-04": "National_Unity_Day",
                "2025-05-02": "Additional_Holiday", "2025-05-08": "Additional_Holiday",
                "2025-06-13": "Additional_Holiday", "2025-11-03": "Additional_Holiday",
                "2025-12-31": "Additional_Holiday",
            }
            additional_working_days = {"2024-04-27", "2024-11-02", "2024-12-28", "2025-11-01"}
            return (
                {pd.to_datetime(k): v for k, v in holidays.items()},
                {pd.to_datetime(x) for x in additional_working_days},
            )

        holidays, additional_working_days = _holiday_calendar()

        df["season"] = df[month_col].map({
            12: "Winter", 1: "Winter", 2: "Winter",
            3: "Spring", 4: "Spring", 5: "Spring",
            6: "Summer", 7: "Summer", 8: "Summer",
            9: "Autumn", 10: "Autumn", 11: "Autumn",
        })
        df["is_winter"] = (df["season"] == "Winter").astype(int)
        df["is_spring"] = (df["season"] == "Spring").astype(int)
        df["is_autumn"] = (df["season"] == "Autumn").astype(int)
        df["is_quarter_start_month"] = df[month_col].isin([1, 4, 7, 10]).astype(int)
        df["is_quarter_end_month"] = df[month_col].isin([3, 6, 9, 12]).astype(int)
        df["is_halfyear_start_month"] = df[month_col].isin([1, 7]).astype(int)
        df["is_halfyear_end_month"] = df[month_col].isin([6, 12]).astype(int)
        df["is_year_start_month"] = (df[month_col] == 1).astype(int)
        df["is_year_end_month"] = (df[month_col] == 12).astype(int)
        df["month_in_halfyear"] = ((df[month_col] - 1) % 6 + 1).astype(int)
        df["month_name"] = df[month_col].map({
            1: "January", 2: "February", 3: "March", 4: "April", 5: "May", 6: "June",
            7: "July", 8: "August", 9: "September", 10: "October", 11: "November", 12: "December",
        })
        df["month_sin"] = np.sin(2 * np.pi * df[month_col] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df[month_col] / 12)
        df["month_sin_2"] = np.sin(2 * np.pi * 2 * df[month_col] / 12)
        df["month_cos_2"] = np.cos(2 * np.pi * 2 * df[month_col] / 12)
        df["prev_month"] = df[month_col].where(df[month_col] > 1, 13) - 1

        calendar = pd.DataFrame({
            "date": pd.date_range(f"{min(years)}-01-01", f"{max(years)}-12-31", freq="D")
        })
        calendar["year"] = calendar["date"].dt.year
        calendar["month"] = calendar["date"].dt.month
        calendar["weekday"] = calendar["date"].dt.dayofweek
        calendar["is_weekend"] = calendar["weekday"].isin([5, 6]).astype(int)
        calendar["holiday_name"] = calendar["date"].map(holidays)
        calendar["is_holiday"] = calendar["holiday_name"].notna().astype(int)
        calendar["is_additional_working_day"] = calendar["date"].isin(additional_working_days).astype(int)
        calendar["is_nonworking_day"] = (((calendar["is_weekend"] == 1) | (calendar["is_holiday"] == 1)) & (calendar["is_additional_working_day"] == 0)).astype(int)
        calendar["is_working_day"] = 1 - calendar["is_nonworking_day"]
        calendar["is_may_holiday"] = calendar["holiday_name"].isin(["Spring_and_Labour_Holiday", "Victory_Day"]).astype(int)
        calendar["is_single_public_holiday"] = calendar["holiday_name"].isin([
            "Defender_of_the_Fatherland_Day", "International_Womens_Day", "Day_of_Russia", "National_Unity_Day"
        ]).astype(int)
        calendar["is_additional_holiday"] = (calendar["holiday_name"] == "Additional_Holiday").astype(int)
        calendar["is_november_holiday"] = (calendar["holiday_name"] == "National_Unity_Day").astype(int)
        calendar["is_december_additional_holiday"] = ((calendar["month"] == 12) & (calendar["holiday_name"] == "Additional_Holiday")).astype(int)

        month_calendar = (
            calendar.groupby(["year", "month"], as_index=False)
            .agg(
                n_days_in_month=("date", "count"),
                n_weekend_days=("is_weekend", "sum"),
                n_may_holiday_days=("is_may_holiday", "sum"),
                n_single_public_holiday_days=("is_single_public_holiday", "sum"),
                n_november_holiday_days=("is_november_holiday", "sum"),
                n_december_additional_holiday_days=("is_december_additional_holiday", "sum"),
                n_holiday_days=("is_holiday", "sum"),
                n_additional_holiday_days=("is_additional_holiday", "sum"),
                n_working_days=("is_working_day", "sum"),
                n_additional_working_days=("is_additional_working_day", "sum"),
            )
        )
        month_calendar["is_leap_year"] = (
            ((month_calendar["year"] % 4 == 0) & ((month_calendar["year"] % 100 != 0) | (month_calendar["year"] % 400 == 0)))
        ).astype(int)
        month_calendar["holiday_month_type"] = month_calendar["month"].map(
            lambda m: "new_year_holidays" if m == 1 else
            "may_holidays" if m == 5 else
            "single_public_holiday" if m in [2, 3, 6, 11] else
            "pre_new_year" if m == 12 else
            "no_major_holiday"
        )

        df = df.merge(
            month_calendar,
            left_on=[year_col, month_col],
            right_on=["year", "month"],
            how="left",
            suffixes=("", "_calendar"),
        ).drop(columns=["year_calendar", "month_calendar"], errors="ignore")

        df["is_middle_of_quarter"] = (((df[month_col] - 1) % 3 + 1) == 2).astype(int)
        df["is_pre_new_year_month"] = df[month_col].isin([11, 12]).astype(int)
        df["is_pre_may_holidays_month"] = (df[month_col] == 4).astype(int)
        df["is_back_to_school_month"] = df[month_col].isin([8, 9]).astype(int)
        df["is_business_activity_low_month"] = df[month_col].isin([1, 5, 7, 8]).astype(int)
        df["is_business_activity_high_month"] = df[month_col].isin([3, 4, 9, 10, 11]).astype(int)

        return df

    def _add_static(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Traffic
        df["total_traffic_per_hour"] = _col(df, "pedestrian_traffic_per_hour") + _col(df, "car_traffic_per_hour")
        df["pedestrian_traffic_share"] = _safe_divide(_col(df, "pedestrian_traffic_per_hour"), df["total_traffic_per_hour"], fill=0.0)
        df["car_to_pedestrian_traffic_ratio"] = _safe_divide(_col(df, "car_traffic_per_hour"), _col(df, "pedestrian_traffic_per_hour"), fill=0.0)

        for c in ["population_size", "number_of_households", "pedestrian_traffic_per_hour", "car_traffic_per_hour"]:
            if c in df.columns:
                df[f"log_{c}"] = np.log1p(df[c].clip(lower=0).fillna(0))

        if "settlement" in df.columns:
            df["is_moscow"] = df["settlement"].astype(str).str.strip().str.lower().eq("москва г").astype(int)
        else:
            df["is_moscow"] = 0

        df["is_million_plus_city"] = (_col(df, "population_size") >= 1_000_000).astype(int)

        def _city_type(pop, is_moscow):
            if is_moscow == 1:
                return "moscow"
            if pd.isna(pop) or pop <= 0:
                return "unknown_or_zero"
            if pop < 10_000:
                return "small_settlement"
            if pop < 50_000:
                return "small_town"
            if pop < 250_000:
                return "medium_city"
            if pop < 1_000_000:
                return "large_city"
            return "million_plus_city"

        df["population_city_type"] = [_city_type(p, m) for p, m in zip(_col(df, "population_size"), df["is_moscow"])]

        for src, dst in [
            ("schools_300m", "has_school_nearby"),
            ("public_transport_stops_300m", "has_transport_nearby"),
            ("marketplaces_deliveries_parcel_lockers_100m", "has_marketplace_nearby"),
            ("medical_institutions_and_pharmacies_300m", "has_medical_nearby"),
            ("grocery_stores_500m", "has_grocery_competition"),
            ("pyaterochka_stores_500m", "has_pyaterochka_nearby"),
        ]:
            df[dst] = (_col(df, src) > 0).astype(int)

        df["is_high_grocery_competition"] = (_col(df, "grocery_stores_500m") >= 5).astype(int)
        df["is_high_pyaterochka_density"] = (_col(df, "pyaterochka_stores_500m") >= 3).astype(int)

        df["non_pyaterochka_grocery_stores_500m"] = (_col(df, "grocery_stores_500m") - _col(df, "pyaterochka_stores_500m")).clip(lower=0)
        df["pyaterochka_share_among_grocery"] = _safe_divide(_col(df, "pyaterochka_stores_500m"), _col(df, "grocery_stores_500m"), fill=0.0)
        df["non_pyaterochka_competition_share"] = _safe_divide(df["non_pyaterochka_grocery_stores_500m"], _col(df, "grocery_stores_500m"), fill=0.0)
        df["grocery_per_1000_people"] = _safe_divide(_col(df, "grocery_stores_500m") * 1000, _col(df, "population_size"), fill=0.0)
        df["pyaterochka_per_1000_people"] = _safe_divide(_col(df, "pyaterochka_stores_500m") * 1000, _col(df, "population_size"), fill=0.0)
        df["competition_per_1000_households"] = _safe_divide(_col(df, "grocery_stores_500m") * 1000, _col(df, "number_of_households"), fill=0.0)
        df["pyaterochka_per_1000_households"] = _safe_divide(_col(df, "pyaterochka_stores_500m") * 1000, _col(df, "number_of_households"), fill=0.0)

        df["people_per_household"] = _safe_divide(_col(df, "population_size"), _col(df, "number_of_households"), fill=0.0)
        df["households_per_1000_people"] = _safe_divide(_col(df, "number_of_households") * 1000, _col(df, "population_size"), fill=0.0)

        df["traffic_per_1000_people"] = _safe_divide(df["total_traffic_per_hour"] * 1000, _col(df, "population_size"), fill=0.0)
        df["traffic_per_1000_households"] = _safe_divide(df["total_traffic_per_hour"] * 1000, _col(df, "number_of_households"), fill=0.0)
        df["pedestrian_traffic_per_1000_people"] = _safe_divide(_col(df, "pedestrian_traffic_per_hour") * 1000, _col(df, "population_size"), fill=0.0)

        df["social_infra_score"] = _col(df, "medical_institutions_and_pharmacies_300m") + _col(df, "schools_300m") + _col(df, "public_transport_stops_300m")
        df["commercial_infra_score"] = _col(df, "marketplaces_deliveries_parcel_lockers_100m") + _col(df, "grocery_stores_500m")
        df["poi_score"] = (
            _col(df, "marketplaces_deliveries_parcel_lockers_100m")
            + _col(df, "medical_institutions_and_pharmacies_300m")
            + _col(df, "schools_300m")
            + _col(df, "public_transport_stops_300m")
            + _col(df, "grocery_stores_500m")
        )
        df["poi_per_1000_households"] = _safe_divide(df["poi_score"] * 1000, _col(df, "number_of_households"), fill=0.0)
        df["public_transport_share_in_social_infra"] = _safe_divide(_col(df, "public_transport_stops_300m"), df["social_infra_score"], fill=0.0)
        df["medical_share_in_social_infra"] = _safe_divide(_col(df, "medical_institutions_and_pharmacies_300m"), df["social_infra_score"], fill=0.0)

        df["cash_register_hours_capacity"] = _col(df, "number_of_cash_registers") * _col(df, "working_hours_per_day")
        df["traffic_per_cash_register"] = _safe_divide(df["total_traffic_per_hour"], _col(df, "number_of_cash_registers"), fill=0.0)
        df["traffic_per_working_hour"] = _safe_divide(df["total_traffic_per_hour"], _col(df, "working_hours_per_day"), fill=0.0)
        df["households_per_cash_register"] = _safe_divide(_col(df, "number_of_households"), _col(df, "number_of_cash_registers"), fill=0.0)
        df["population_per_cash_register"] = _safe_divide(_col(df, "population_size"), _col(df, "number_of_cash_registers"), fill=0.0)
        df["cash_registers_per_total_traffic"] = _safe_divide(_col(df, "number_of_cash_registers"), df["total_traffic_per_hour"], fill=0.0)

        if include_current_receipt_features:
            if {"avg_promo_items_per_receipt", "avg_items_per_receipt"}.issubset(df.columns):
                df["promo_items_share"] = _safe_divide(_col(df, "avg_promo_items_per_receipt"), _col(df, "avg_items_per_receipt"), fill=0.0)
            if {"avg_items_per_receipt", "working_hours_per_day"}.issubset(df.columns):
                df["items_x_working_hours"] = _col(df, "avg_items_per_receipt") * _col(df, "working_hours_per_day")

        df["alcohol_x_capacity"] = _col(df, "alcohol_license_flag") * df["cash_register_hours_capacity"]
        df["alcohol_x_total_traffic"] = _col(df, "alcohol_license_flag") * df["total_traffic_per_hour"]
        if include_current_receipt_features and "avg_items_per_receipt" in df.columns:
            df["alcohol_x_items"] = _col(df, "alcohol_license_flag") * _col(df, "avg_items_per_receipt")

        df["traffic_x_total_poi"] = df["total_traffic_per_hour"] * df["poi_score"]
        df["traffic_x_grocery_competition"] = df["total_traffic_per_hour"] * _col(df, "grocery_stores_500m")
        df["traffic_x_pyaterochka_competition"] = df["total_traffic_per_hour"] * _col(df, "pyaterochka_stores_500m")
        df["pedestrian_traffic_x_transport"] = _col(df, "pedestrian_traffic_per_hour") * _col(df, "public_transport_stops_300m")
        if "is_back_to_school_month" in df.columns:
            df["schools_x_back_to_school"] = _col(df, "schools_300m") * _col(df, "is_back_to_school_month")

        # Store-level counts / exogenous group means. They use only exogenous known columns, not target.
        if id_col in df.columns:
            static_cols = [
                id_col, "settlement", "region", "opening_date_category", "trading_area_category",
                "total_traffic_per_hour", "grocery_stores_500m", "pyaterochka_stores_500m",
                "number_of_cash_registers", "poi_score",
            ]
            static_cols = [c for c in static_cols if c in df.columns]
            store_static = df[static_cols].drop_duplicates(subset=[id_col]).copy()

            for group_cols, new_name in [
                (["settlement"], "settlement_store_count"),
                (["region"], "region_store_count"),
                (["region", "trading_area_category"], "region_trading_area_store_count"),
                (["region", "opening_date_category"], "region_opening_age_store_count"),
            ]:
                if all(c in store_static.columns for c in group_cols):
                    cnt = store_static.groupby(group_cols, dropna=False)[id_col].nunique().rename(new_name).reset_index()
                    df = df.merge(cnt, on=group_cols, how="left")

            exog_cols = [c for c in [
                "total_traffic_per_hour", "grocery_stores_500m", "pyaterochka_stores_500m", "number_of_cash_registers", "poi_score"
            ] if c in store_static.columns]

            if exog_cols and "region" in store_static.columns:
                agg = store_static.groupby("region", dropna=False)[exog_cols].mean().add_prefix("region_").add_suffix("_mean").reset_index()
                df = df.merge(agg, on="region", how="left")
            if exog_cols and "settlement" in store_static.columns:
                agg = store_static.groupby("settlement", dropna=False)[exog_cols].mean().add_prefix("settlement_").add_suffix("_mean").reset_index()
                df = df.merge(agg, on="settlement", how="left")

        return df

    def _prepare_base_parts(parts: dict[str, pd.DataFrame]) -> tuple[dict[str, pd.DataFrame], dict]:
        checked = {}
        for name, p in parts.items():
            p = _ensure_time_year_month(p, name)
            _validate_required(p, {id_col, time_col}, name)
            checked[name] = p

        combined = _part_concat(checked)
        combined = _drop_preexisting_generated_cols(combined)
        combined = _ensure_time_year_month(combined, "combined")

        if add_calendar_features:
            combined = _add_calendar(combined)
            combined = _ensure_time_year_month(combined, "combined_after_calendar")

        if add_static_features:
            combined = _add_static(combined)

        parts_out = _split_parts(combined, list(parts.keys()))
        return parts_out, {"base_shape": combined.shape}

    # ============================================================
    # History feature blocks
    # ============================================================

    def _combine_history_future(history_df: pd.DataFrame, future_df: pd.DataFrame, future_name: str):
        history_df = history_df.copy()
        future_df = future_df.copy()
        history_df["__part__"] = "history"
        future_df["__part__"] = future_name
        history_df["__row_order__"] = np.arange(len(history_df))
        future_df["__row_order__"] = np.arange(len(future_df))
        combined = pd.concat([history_df, future_df], axis=0, ignore_index=True, sort=False)
        combined = _ensure_time_year_month(combined, "history_future_combined")
        return combined

    def _restore_pair(combined: pd.DataFrame, future_name: str):
        history_features = (
            combined[combined["__part__"] == "history"]
            .sort_values("__row_order__")
            .drop(columns=["__part__", "__row_order__"], errors="ignore")
            .reset_index(drop=True)
        )
        future_features = (
            combined[combined["__part__"] == future_name]
            .sort_values("__row_order__")
            .drop(columns=["__part__", "__row_order__"], errors="ignore")
            .reset_index(drop=True)
        )
        return history_features, future_features

    def _merge_lag_feature(combined, history, source_col, lag, feature_name, by_cols):
        if source_col not in history.columns:
            return combined, False
        lag_df = history[by_cols + [time_col, source_col]].copy()
        lag_df = lag_df.dropna(subset=[source_col])
        lag_df[time_col] = lag_df[time_col].astype(int) + int(lag)
        lag_df = lag_df.rename(columns={source_col: feature_name})
        lag_df = lag_df.drop_duplicates(subset=by_cols + [time_col], keep="last")
        combined = combined.merge(lag_df[by_cols + [time_col, feature_name]], on=by_cols + [time_col], how="left")
        return combined, True

    def _add_target_history_pair(history_df, future_df, future_name):
        combined = _combine_history_future(history_df, future_df, future_name)
        history = combined[combined["__part__"] == "history"].copy()
        _validate_required(history, {id_col, time_col, target_col}, "history_for_target_lags")
        _validate_unique_panel(history, "history_for_target_lags")

        created = []
        for lag in target_lags:
            fname = f"{target_col}_lag_{lag}"
            combined, ok = _merge_lag_feature(combined, history, target_col, lag, fname, [id_col])
            if ok:
                created.append(fname)
                if target_history_scale in {"log", "both"}:
                    log_name = f"log_{fname}"
                    combined[log_name] = np.log1p(combined[fname].clip(lower=0))
                    created.append(log_name)

        # Rolling features are based only on past history values; future target is ignored even if present.
        if target_rolling_windows:
            tmp = combined[[id_col, time_col, "__part__", target_col]].copy()
            tmp["__target_for_roll__"] = np.where(tmp["__part__"].eq("history"), tmp[target_col], np.nan)
            tmp = tmp.sort_values([id_col, time_col], kind="mergesort")
            shifted = tmp.groupby(id_col, sort=False)["__target_for_roll__"].shift(1)
            for window in target_rolling_windows:
                rg = shifted.groupby(tmp[id_col], sort=False)
                mean_col = f"{target_col}_roll_mean_{window}"
                std_col = f"{target_col}_roll_std_{window}"
                cv_col = f"{target_col}_roll_cv_{window}"
                tmp[mean_col] = rg.rolling(window, min_periods=target_rolling_min_periods).mean().reset_index(level=0, drop=True)
                tmp[std_col] = rg.rolling(window, min_periods=target_rolling_min_periods).std().reset_index(level=0, drop=True)
                tmp[cv_col] = _safe_divide(tmp[std_col], tmp[mean_col], fill=np.nan)
                created.extend([mean_col, std_col, cv_col])
            tmp = tmp.sort_index()
            for c in created:
                if c in tmp.columns and c not in combined.columns:
                    combined[c] = tmp[c]

        def lag_col(lag): return f"{target_col}_lag_{lag}"

        for a, b in [(1, 2), (1, 3), (1, 6), (1, 12), (3, 6)]:
            if {lag_col(a), lag_col(b)}.issubset(combined.columns):
                c = f"{target_col}_diff_lag_{a}_{b}"
                combined[c] = combined[lag_col(a)] - combined[lag_col(b)]
                created.append(c)

        for a, b in [(1, 2), (1, 3), (1, 6), (1, 12), (3, 6), (6, 12)]:
            if {lag_col(a), lag_col(b)}.issubset(combined.columns):
                c = f"{target_col}_ratio_lag_{a}_{b}"
                combined[c] = _safe_divide(combined[lag_col(a)], combined[lag_col(b)], fill=np.nan)
                created.append(c)

        for window in target_rolling_windows:
            roll_mean = f"{target_col}_roll_mean_{window}"
            if {lag_col(1), roll_mean}.issubset(combined.columns):
                c = f"{target_col}_lag_1_to_roll_mean_{window}"
                combined[c] = _safe_divide(combined[lag_col(1)], combined[roll_mean], fill=np.nan)
                created.append(c)

        if {lag_col(12), f"{target_col}_roll_mean_12"}.issubset(combined.columns):
            c = f"{target_col}_lag_12_to_roll_mean_12"
            combined[c] = _safe_divide(combined[lag_col(12)], combined[f"{target_col}_roll_mean_12"], fill=np.nan)
            created.append(c)

        if {lag_col(12), lag_col(13)}.issubset(combined.columns):
            c1 = f"{target_col}_seasonal_diff_12_13"
            c2 = f"{target_col}_seasonal_ratio_12_13"
            combined[c1] = combined[lag_col(12)] - combined[lag_col(13)]
            combined[c2] = _safe_divide(combined[lag_col(12)], combined[lag_col(13)], fill=np.nan)
            created.extend([c1, c2])

        if target_history_scale in {"log", "both"} and add_log_target_dynamics:
            for a, b in [(1, 2), (1, 3), (1, 6), (1, 12), (3, 6), (6, 12)]:
                la, lb = f"log_{lag_col(a)}", f"log_{lag_col(b)}"
                if {la, lb}.issubset(combined.columns):
                    c = f"log_{target_col}_diff_lag_{a}_{b}"
                    combined[c] = combined[la] - combined[lb]
                    created.append(c)

        created = sorted(set(c for c in created if c in combined.columns))
        combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)
        if fill_history_na:
            combined[created] = combined[created].fillna(fill_value)
        return (*_restore_pair(combined, future_name), created)

    def _add_category_history_pair(history_df, future_df, future_name):
        combined = _combine_history_future(history_df, future_df, future_name)
        history = combined[combined["__part__"] == "history"].copy()
        _validate_required(history, {id_col, time_col, target_col}, "history_for_category_lags")

        specs = group_specs
        if specs is None:
            specs = [
                "region",
                "settlement",
                ["region", "trading_area_category"],
                ["region", "opening_date_category"],
                ["settlement", "trading_area_category"],
            ]
        norm_specs = []
        for spec in specs:
            spec = [spec] if isinstance(spec, str) else list(spec)
            if all(c in combined.columns for c in spec):
                norm_specs.append(spec)
        if not norm_specs:
            return (*_restore_pair(combined, future_name), [])

        created = []
        internal_store_lag = None
        store_lag_1 = f"{target_col}_lag_1"
        if store_lag_1 not in combined.columns:
            internal_store_lag = f"__{target_col}_store_lag_1__"
            combined, _ = _merge_lag_feature(combined, history, target_col, 1, internal_store_lag, [id_col])
            store_lag_1 = internal_store_lag

        for group_cols in norm_specs:
            prefix = "__".join(group_cols)
            agg_col = f"__{prefix}_{target_col}_{group_agg_stat}__"
            monthly_agg = (
                history.groupby(group_cols + [time_col], dropna=False)[target_col]
                .agg(group_agg_stat)
                .reset_index()
                .rename(columns={target_col: agg_col})
            )
            monthly_agg[time_col] = monthly_agg[time_col].astype(int) + int(group_lag)
            group_lag_col = f"{prefix}_{target_col}_{group_agg_stat}_lag_{group_lag}"
            monthly_agg = monthly_agg.rename(columns={agg_col: group_lag_col})
            combined = combined.merge(monthly_agg[group_cols + [time_col, group_lag_col]], on=group_cols + [time_col], how="left")

            if add_group_lag_feature:
                created.append(group_lag_col)

            if add_group_relative_ratio:
                c = f"{target_col}_lag_1_to_{prefix}_{target_col}_{group_agg_stat}_lag_{group_lag}"
                combined[c] = _safe_divide(combined[store_lag_1], combined[group_lag_col], fill=np.nan)
                created.append(c)

            if add_group_relative_diff:
                c = f"{target_col}_lag_1_minus_{prefix}_{target_col}_{group_agg_stat}_lag_{group_lag}"
                combined[c] = combined[store_lag_1] - combined[group_lag_col]
                created.append(c)

        if internal_store_lag is not None:
            combined = combined.drop(columns=[internal_store_lag], errors="ignore")

        created = sorted(set(c for c in created if c in combined.columns))
        combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)
        if fill_history_na:
            combined[created] = combined[created].fillna(fill_value)
        return (*_restore_pair(combined, future_name), created)

    def _add_monthly_history_pair(history_df, future_df, future_name):
        combined = _combine_history_future(history_df, future_df, future_name)
        history = combined[combined["__part__"] == "history"].copy()
        _validate_required(history, {id_col, time_col}, "history_for_monthly_lags")
        _validate_unique_panel(history, "history_for_monthly_lags")

        existing_monthly = [c for c in current_monthly_cols if c in history.columns]
        if not existing_monthly:
            return (*_restore_pair(combined, future_name), [])

        if monthly_feature_config is None:
            cfg_all = {
                "avg_promo_items_per_receipt": {
                    "lags": (1, 3, 12), "rolling_windows": (3, 12),
                    "rolling_stats": ("std", "cv"), "diff_pairs": ((1, 12),), "ratio_pairs": ((1, 12),),
                },
                "avg_items_per_receipt": {
                    "lags": (1, 3, 12), "rolling_windows": (3, 12),
                    "rolling_stats": ("std", "cv"), "diff_pairs": ((1, 12),), "ratio_pairs": ((1, 12),),
                },
                "avg_cancellations": {
                    "lags": (1, 3, 12), "rolling_windows": (3, 12),
                    "rolling_stats": ("mean", "std", "cv"), "diff_pairs": ((1, 12),), "ratio_pairs": ((1, 12),),
                },
                "working_hours_per_day": {
                    "lags": (1,), "rolling_windows": (), "rolling_stats": (), "diff_pairs": (), "ratio_pairs": (),
                },
            }
        else:
            cfg_all = monthly_feature_config

        created = []

        for colname in existing_monthly:
            cfg = cfg_all.get(colname, {
                "lags": (1, 3, 12), "rolling_windows": (3, 12),
                "rolling_stats": ("std", "cv"), "diff_pairs": ((1, 12),), "ratio_pairs": ((1, 12),),
            })

            # Time-aware lags.
            for lag in tuple(cfg.get("lags", ())):
                c = f"{colname}_lag_{lag}"
                combined, ok = _merge_lag_feature(combined, history, colname, lag, c, [id_col])
                if ok:
                    created.append(c)
                    if monthly_history_scale in {"log", "both"}:
                        lc = f"log_{colname}_lag_{lag}"
                        combined[lc] = np.log1p(combined[c].clip(lower=0))
                        created.append(lc)

            # Rolling over past observed history.
            windows = tuple(cfg.get("rolling_windows", ()))
            stats = tuple(cfg.get("rolling_stats", ()))
            if windows and stats:
                tmp = combined[[id_col, time_col, "__part__", colname]].copy()
                tmp["__x__"] = np.where(tmp["__part__"].eq("history"), tmp[colname], np.nan)
                tmp = tmp.sort_values([id_col, time_col], kind="mergesort")
                shifted = tmp.groupby(id_col, sort=False)["__x__"].shift(1)
                for window in windows:
                    rg = shifted.groupby(tmp[id_col], sort=False)
                    mean_s = std_s = min_s = max_s = None
                    if "mean" in stats or "cv" in stats:
                        mean_s = rg.rolling(window, min_periods=monthly_rolling_min_periods).mean().reset_index(level=0, drop=True)
                        if "mean" in stats:
                            c = f"{colname}_roll_mean_{window}"
                            tmp[c] = mean_s
                            created.append(c)
                    if "std" in stats or "cv" in stats:
                        std_s = rg.rolling(window, min_periods=monthly_rolling_min_periods).std().reset_index(level=0, drop=True)
                        if "std" in stats:
                            c = f"{colname}_roll_std_{window}"
                            tmp[c] = std_s
                            created.append(c)
                    if "min" in stats or "range" in stats:
                        min_s = rg.rolling(window, min_periods=monthly_rolling_min_periods).min().reset_index(level=0, drop=True)
                        if "min" in stats:
                            c = f"{colname}_roll_min_{window}"
                            tmp[c] = min_s
                            created.append(c)
                    if "max" in stats or "range" in stats:
                        max_s = rg.rolling(window, min_periods=monthly_rolling_min_periods).max().reset_index(level=0, drop=True)
                        if "max" in stats:
                            c = f"{colname}_roll_max_{window}"
                            tmp[c] = max_s
                            created.append(c)
                    if "range" in stats:
                        c = f"{colname}_roll_range_{window}"
                        tmp[c] = max_s - min_s
                        created.append(c)
                    if "cv" in stats:
                        c = f"{colname}_roll_cv_{window}"
                        tmp[c] = _safe_divide(std_s, mean_s, fill=np.nan)
                        created.append(c)
                tmp = tmp.sort_index()
                for c in created:
                    if c in tmp.columns and c not in combined.columns:
                        combined[c] = tmp[c]

            for a, b in tuple(cfg.get("diff_pairs", ())):
                ca, cb = f"{colname}_lag_{a}", f"{colname}_lag_{b}"
                if {ca, cb}.issubset(combined.columns):
                    c = f"{colname}_diff_lag_{a}_{b}"
                    combined[c] = combined[ca] - combined[cb]
                    created.append(c)

            for a, b in tuple(cfg.get("ratio_pairs", ())):
                ca, cb = f"{colname}_lag_{a}", f"{colname}_lag_{b}"
                if {ca, cb}.issubset(combined.columns):
                    c = f"{colname}_ratio_lag_{a}_{b}"
                    combined[c] = _safe_divide(combined[ca], combined[cb], fill=np.nan)
                    created.append(c)

        if add_monthly_cross_lags:
            for lag in monthly_cross_lags:
                promo = f"avg_promo_items_per_receipt_lag_{lag}"
                items = f"avg_items_per_receipt_lag_{lag}"
                canc = f"avg_cancellations_lag_{lag}"
                hours = f"working_hours_per_day_lag_{lag}"

                if "promo_share" in monthly_cross_feature_types and {promo, items}.issubset(combined.columns):
                    c = f"promo_items_share_lag_{lag}"
                    combined[c] = _safe_divide(combined[promo], combined[items], fill=np.nan)
                    created.append(c)
                if "promo_x_items" in monthly_cross_feature_types and {promo, items}.issubset(combined.columns):
                    c = f"promo_x_items_lag_{lag}"
                    combined[c] = combined[promo] * combined[items]
                    created.append(c)
                if "items_x_hours" in monthly_cross_feature_types and {items, hours}.issubset(combined.columns):
                    c = f"items_x_working_hours_lag_{lag}"
                    combined[c] = combined[items] * combined[hours]
                    created.append(c)
                if "cancellations_per_item" in monthly_cross_feature_types and {canc, items}.issubset(combined.columns):
                    c = f"cancellations_per_item_lag_{lag}"
                    combined[c] = _safe_divide(combined[canc], combined[items], fill=np.nan)
                    created.append(c)
                if "cancellations_per_hour" in monthly_cross_feature_types and {canc, hours}.issubset(combined.columns):
                    c = f"cancellations_per_working_hour_lag_{lag}"
                    combined[c] = _safe_divide(combined[canc], combined[hours], fill=np.nan)
                    created.append(c)

        created = sorted(set(c for c in created if c in combined.columns))
        combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)
        if fill_history_na:
            combined[created] = combined[created].fillna(fill_value)

        if drop_current_monthly_cols:
            combined = combined.drop(columns=list(existing_monthly), errors="ignore")
        if drop_current_monthly_derived_cols:
            combined = combined.drop(columns=list(current_monthly_derived_cols), errors="ignore")

        return (*_restore_pair(combined, future_name), created)

    # ============================================================
    # Final cleanup
    # ============================================================

    def _future_unknown_cols():
        cols = []
        if drop_current_monthly_cols:
            cols.extend(list(current_monthly_cols))
        if drop_current_monthly_derived_cols:
            cols.extend(list(current_monthly_derived_cols))
        if not working_hours_future_known:
            cols.extend([
                "working_hours_per_day",
                "cash_register_hours_capacity",
                "traffic_per_working_hour",
                "alcohol_x_capacity",
                "items_x_working_hours",
            ])
        if drop_year_col:
            cols.append(year_col)
        return sorted(set(cols))

    def _obvious_duplicate_cols():
        cols = [
            f"{target_col}_roll_mean_3", f"{target_col}_roll_mean_12",
            f"{target_col}_lag_13", f"{target_col}_diff_lag_3_6", f"{target_col}_ratio_lag_3_6",
            "avg_items_per_receipt_diff_lag_1_12",
            "avg_cancellations_roll_mean_3", "avg_cancellations_roll_mean_12",
            "cancellations_per_item_lag_1", "cancellations_per_item_lag_3", "cancellations_per_item_lag_12",
            "settlement_store_count", "total_traffic_per_hour", "non_pyaterochka_grocery_stores_500m",
        ]

        if target_history_scale == "raw":
            cols.extend([f"log_{target_col}_lag_{l}" for l in target_lags])
            cols.extend([c for c in []])
        elif target_history_scale == "log":
            raw_lags = [f"{target_col}_lag_{l}" for l in target_lags if l != 1]
            if not keep_raw_pto_lag_1_anchor_for_log:
                raw_lags.append(f"{target_col}_lag_1")
            cols.extend(raw_lags)
            cols.extend([
                f"{target_col}_diff_lag_1_2", f"{target_col}_diff_lag_1_3",
                f"{target_col}_diff_lag_1_6", f"{target_col}_diff_lag_1_12",
                f"{target_col}_diff_lag_3_6", f"{target_col}_ratio_lag_1_2",
                f"{target_col}_ratio_lag_1_3", f"{target_col}_ratio_lag_1_6",
                f"{target_col}_ratio_lag_1_12", f"{target_col}_ratio_lag_3_6",
                f"{target_col}_ratio_lag_6_12", f"{target_col}_seasonal_diff_12_13",
                f"{target_col}_seasonal_ratio_12_13",
            ])

        if monthly_history_scale == "raw":
            for c in current_monthly_cols:
                cols.extend([f"log_{c}_lag_1", f"log_{c}_lag_3", f"log_{c}_lag_12"])
        elif monthly_history_scale == "log":
            for c in current_monthly_cols:
                cols.extend([f"{c}_lag_3", f"{c}_lag_12"])

        if extra_drop_cols is not None:
            cols.extend(list(extra_drop_cols))
        return sorted(set(cols))

    def _apply_final_clean(train_part, future_part):
        train_part = train_part.copy()
        future_part = future_part.copy()
        drop_cols = _future_unknown_cols()
        if drop_obvious_duplicates:
            drop_cols.extend(_obvious_duplicate_cols())
        drop_cols = sorted(set(c for c in drop_cols if c in train_part.columns or c in future_part.columns))
        train_part = train_part.drop(columns=drop_cols, errors="ignore")
        future_part = future_part.drop(columns=drop_cols, errors="ignore")

        # Replace infinities only; do not fill all lag NaNs unless requested.
        for df_ in (train_part, future_part):
            num_cols = df_.select_dtypes(include=[np.number]).columns
            df_[num_cols] = df_[num_cols].replace([np.inf, -np.inf], np.nan)
            if fill_history_na:
                df_[num_cols] = df_[num_cols].fillna(fill_value)

        # Align columns in stable order.
        ordered_cols = list(train_part.columns)
        for c in future_part.columns:
            if c not in ordered_cols:
                ordered_cols.append(c)
        for c in ordered_cols:
            if c not in train_part.columns:
                train_part[c] = np.nan
            if c not in future_part.columns:
                future_part[c] = np.nan
        train_part = train_part[ordered_cols]
        future_part = future_part[ordered_cols]
        return train_part, future_part, drop_cols

    def _downcast(df):
        if not downcast_float_features:
            return df
        df = df.copy()
        float_cols = [c for c in df.select_dtypes(include=["float64"]).columns if c != target_col]
        if float_cols:
            df[float_cols] = df[float_cols].astype("float32")
        return df

    def _generate_pair(history_base, future_base, future_name):
        info = {}
        h, f = history_base.copy(), future_base.copy()

        if add_target_history:
            h, f, created = _add_target_history_pair(h, f, future_name)
            info["target_history_features"] = created
        else:
            info["target_history_features"] = []

        if add_category_target_history:
            h, f, created = _add_category_history_pair(h, f, future_name)
            info["category_target_history_features"] = created
        else:
            info["category_target_history_features"] = []

        if add_non_target_monthly_history:
            h, f, created = _add_monthly_history_pair(h, f, future_name)
            info["non_target_monthly_history_features"] = created
        else:
            info["non_target_monthly_history_features"] = []
            if drop_current_monthly_cols:
                h = h.drop(columns=[c for c in current_monthly_cols if c in h.columns], errors="ignore")
                f = f.drop(columns=[c for c in current_monthly_cols if c in f.columns], errors="ignore")
            if drop_current_monthly_derived_cols:
                h = h.drop(columns=[c for c in current_monthly_derived_cols if c in h.columns], errors="ignore")
                f = f.drop(columns=[c for c in current_monthly_derived_cols if c in f.columns], errors="ignore")

        h, f, dropped = _apply_final_clean(h, f)
        info["dropped_final_cols"] = dropped
        h = _downcast(h)
        f = _downcast(f)
        return h, f, info

    # ============================================================
    # Main flow
    # ============================================================

    train_df = _ensure_time_year_month(train_df, "train_df")
    _validate_required(train_df, {id_col, time_col, target_col}, "train_df")
    _validate_unique_panel(train_df, "train_df")

    if make_val_from_train:
        if val_df is not None:
            raise ValueError("Нельзя одновременно передать val_df и make_val_from_train=True.")
        if val_time_idx is None:
            val_time_idx = int(train_df[time_col].max())
        val_df = train_df[train_df[time_col] == val_time_idx].copy()
        train_df = train_df[train_df[time_col] < val_time_idx].copy()
        if verbose:
            print(f"Auto val_time_idx: {val_time_idx}")
            print("Auto train shape:", train_df.shape)
            print("Auto val shape:", val_df.shape)

    if val_df is not None:
        val_df = _ensure_time_year_month(val_df, "val_df")
        _validate_required(val_df, {id_col, time_col}, "val_df")
        _validate_unique_panel(val_df, "val_df")

    if test_df is not None:
        test_df = _ensure_time_year_month(test_df, "test_df")
        _validate_required(test_df, {id_col, time_col}, "test_df")
        _validate_unique_panel(test_df, "test_df")
        if target_col not in test_df.columns:
            test_df[target_col] = np.nan

    output = {
        "train_features": None,
        "val_features": None,
        "test_train_features": None,
        "test_features": None,
        "feature_cols": None,
        "categorical_cols": None,
        "feature_info": {},
    }

    if val_df is not None:
        base_parts, base_info = _prepare_base_parts({"train": train_df, "val": val_df})
        train_features, val_features, val_info = _generate_pair(base_parts["train"], base_parts["val"], "val")
        output["train_features"] = train_features
        output["val_features"] = val_features
        output["feature_info"]["val"] = {**base_info, **val_info}
        if verbose:
            print("Generated train_features:", train_features.shape)
            print("Generated val_features:", val_features.shape)

    if test_df is not None:
        if val_df is not None and use_val_as_test_history:
            test_history = pd.concat([train_df, val_df], axis=0, ignore_index=True, sort=False)
        else:
            test_history = train_df.copy()
        test_history = _ensure_time_year_month(test_history, "test_history")
        _validate_unique_panel(test_history, "test_history")

        base_parts, base_info = _prepare_base_parts({"train": test_history, "test": test_df})
        test_train_features, test_features, test_info = _generate_pair(base_parts["train"], base_parts["test"], "test")
        output["test_train_features"] = test_train_features
        output["test_features"] = test_features
        output["feature_info"]["test"] = {**base_info, **test_info}
        if verbose:
            print("Generated test_train_features:", test_train_features.shape)
            print("Generated test_features:", test_features.shape)

    # Select columns for model.
    ref = output["test_train_features"] if output["test_train_features"] is not None else output["train_features"]
    if ref is not None:
        exclude = {target_col}
        feature_cols = [c for c in ref.columns if c not in exclude]
        categorical_cols = [
            c for c in feature_cols
            if c in ref.columns and (pd.api.types.is_object_dtype(ref[c]) or pd.api.types.is_categorical_dtype(ref[c]))
        ]
        output["feature_cols"] = feature_cols
        output["categorical_cols"] = categorical_cols

    gc.collect()
    return output


def _ensure_time_year_month_external(
    df: pd.DataFrame,
    *,
    year_col: str,
    month_col: str,
    time_col: str,
    min_year: int,
    name: str = "df",
) -> pd.DataFrame:
    """Внешний helper для wrapper/enhancer: гарантирует year/month/time."""
    out = df.copy()
    has_year_month = year_col in out.columns and month_col in out.columns
    has_time = time_col in out.columns

    if not has_year_month and not has_time:
        raise ValueError(f"В {name} нет ни year/month, ни time_col='{time_col}'.")

    if has_year_month:
        out[year_col] = pd.to_numeric(out[year_col], errors="raise").astype(int)
        out[month_col] = pd.to_numeric(out[month_col], errors="raise").astype(int)
        out[time_col] = ((out[year_col] - int(min_year)) * 12 + (out[month_col] - 1)).astype(int)
    else:
        out[time_col] = pd.to_numeric(out[time_col], errors="raise").astype(int)
        out[year_col] = (int(min_year) + out[time_col] // 12).astype(int)
        out[month_col] = (out[time_col] % 12 + 1).astype(int)

    return out


def _calendar_month_from_time(time_s: pd.Series, min_year: int) -> pd.Series:
    return (time_s.astype(int) % 12 + 1).astype(int)


def _year_from_time(time_s: pd.Series, min_year: int) -> pd.Series:
    return (int(min_year) + time_s.astype(int) // 12).astype(int)


def _days_in_month_from_year_month(year_s: pd.Series, month_s: pd.Series) -> pd.Series:
    dt = pd.to_datetime(
        {
            "year": pd.to_numeric(year_s, errors="coerce").astype("Int64"),
            "month": pd.to_numeric(month_s, errors="coerce").astype("Int64"),
            "day": 1,
        },
        errors="coerce",
    )
    return dt.dt.days_in_month.astype("float32")


def _safe_divide_external(a, b, fill=np.nan):
    a = a if isinstance(a, pd.Series) else pd.Series(a)
    b = b if isinstance(b, pd.Series) else pd.Series(b)
    out = a.astype(float) / b.replace(0, np.nan).astype(float)
    out = out.replace([np.inf, -np.inf], np.nan)
    if fill is not None and not (isinstance(fill, float) and np.isnan(fill)):
        out = out.fillna(fill)
    return out


def _safe_log_external(x, eps: float = 1e-9):
    return np.log(np.clip(pd.Series(x).astype(float), eps, None))



def _assign_lag1_rto_size_bin(
    df: pd.DataFrame,
    *,
    time_col: str,
    lag1_col: str,
    out_col: str = "lag1_rto_size_bin",
    n_bins: int = 5,
) -> pd.Series:
    """
    Cross-sectional size bin by pto_lag_1 inside each forecast month.
    Uses only lagged target values, therefore it is safe for future rows.
    """
    result = pd.Series(np.nan, index=df.index, dtype="object")

    if lag1_col not in df.columns:
        return result

    for _, idx in df.groupby(time_col, sort=False).groups.items():
        s = pd.to_numeric(df.loc[idx, lag1_col], errors="coerce")
        valid = s.notna()
        if valid.sum() == 0:
            continue
        if valid.sum() == 1 or s[valid].nunique(dropna=True) == 1:
            result.loc[s[valid].index] = "q3"
            continue
        q = int(min(n_bins, valid.sum(), s[valid].nunique(dropna=True)))
        if q < 2:
            result.loc[s[valid].index] = "q3"
            continue
        try:
            codes = pd.qcut(
                s[valid].rank(method="first"),
                q=q,
                labels=False,
                duplicates="drop",
            )
            result.loc[s[valid].index] = [f"q{int(x) + 1}" if pd.notna(x) else np.nan for x in codes]
        except ValueError:
            result.loc[s[valid].index] = "q3"

    return result


def _expanding_median_factor_from_history(
    combined: pd.DataFrame,
    observed: pd.DataFrame,
    *,
    group_cols: list[str],
    time_col: str,
    obs_col: str,
    out_col: str,
) -> pd.Series:
    """
    For every row in combined returns median(obs_col) over historical observations
    with the same group key and strictly earlier time_col.

    Implementation detail: query rows are sorted before observation rows at the
    same time index, so same-month target is never used for the same row/month.
    """
    keys = list(group_cols) + [time_col]
    result = pd.Series(np.nan, index=combined.index, dtype="float64")

    if not group_cols or obs_col not in observed.columns:
        return result
    if any(c not in combined.columns for c in keys) or any(c not in observed.columns for c in keys):
        return result

    obs = observed[keys + [obs_col]].dropna(subset=[obs_col]).copy()
    if obs.empty:
        return result

    obs_gt = (
        obs.groupby(keys, dropna=False, as_index=False)[obs_col]
        .median()
        .rename(columns={obs_col: "__obs_factor__"})
    )
    if obs_gt.empty:
        return result

    queries = combined[keys].copy()
    queries["__query_index__"] = combined.index
    queries["__obs_factor__"] = np.nan
    queries["__is_query__"] = 1
    queries["__order__"] = 0

    obs_gt["__query_index__"] = -1
    obs_gt["__is_query__"] = 0
    obs_gt["__order__"] = 1

    stack = pd.concat([queries, obs_gt], ignore_index=True, sort=False)
    stack = stack.sort_values(group_cols + [time_col, "__order__"], kind="mergesort")

    stack[out_col] = (
        stack.groupby(group_cols, dropna=False)["__obs_factor__"]
        .transform(lambda s: s.expanding(min_periods=1).median())
    )

    q = stack[stack["__is_query__"].eq(1)][["__query_index__", out_col]].copy()
    q = q.drop_duplicates("__query_index__", keep="last")
    result.loc[q["__query_index__"].astype(int).values] = q[out_col].values
    return result


def _add_expanding_calendar_heuristic_features(
    combined: pd.DataFrame,
    *,
    id_col: str,
    time_col: str,
    target_col: str,
    min_year: int,
    eps: float = 1e-9,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Adds the old expanding calendar heuristic block without target leakage.

    Exposed features:
    - lag1_rto_size_bin;
    - same_month_calendar_factor;
    - same_month_region_calendar_factor;
    - same_month_population_type_calendar_factor;
    - same_month_lag1_size_calendar_factor;
    - calendar_growth_multiplier;
    - calendar_heuristic_*_prediction;
    - log_calendar_heuristic_prediction;
    - log_pto_lag_1_to_calendar_heuristic;
    - log_pto_lag_1_to_day_scaled_lag1.

    Internal observed_calendar_factor is deliberately not returned as a model
    feature, because it uses current target on history rows.
    """
    out = combined.copy()
    created: list[str] = []

    def mark(col: str):
        if col in out.columns and col not in created:
            created.append(col)

    pto_lag_1 = f"{target_col}_lag_1"
    required = {"__part__", time_col, target_col, pto_lag_1, "day_scaled_lag1_prediction", "month_day_ratio_to_previous_month"}
    if not required.issubset(out.columns):
        return out, created

    if "calendar_month_num" not in out.columns:
        out["calendar_month_num"] = _calendar_month_from_time(out[time_col], min_year)
        mark("calendar_month_num")

    out["lag1_rto_size_bin"] = _assign_lag1_rto_size_bin(
        out,
        time_col=time_col,
        lag1_col=pto_lag_1,
        out_col="lag1_rto_size_bin",
        n_bins=5,
    )
    mark("lag1_rto_size_bin")

    # Useful log-ratio against the pure day-scaled anchor.
    if f"log_{target_col}_lag_1" not in out.columns:
        out[f"log_{target_col}_lag_1"] = _safe_log_external(out[pto_lag_1], eps=eps)
        mark(f"log_{target_col}_lag_1")
    if "log_day_scaled_lag1_prediction" not in out.columns:
        out["log_day_scaled_lag1_prediction"] = _safe_log_external(out["day_scaled_lag1_prediction"], eps=eps)
        mark("log_day_scaled_lag1_prediction")

    out["log_pto_lag_1_to_day_scaled_lag1"] = (
        out[f"log_{target_col}_lag_1"] - out["log_day_scaled_lag1_prediction"]
    )
    mark("log_pto_lag_1_to_day_scaled_lag1")

    hist_mask = out["__part__"].eq("history") & out[target_col].notna()
    observed = out.loc[hist_mask].copy()
    observed["__observed_calendar_factor__"] = _safe_divide_external(
        observed[target_col],
        observed["day_scaled_lag1_prediction"],
        fill=np.nan,
    )
    observed["__observed_calendar_factor__"] = observed["__observed_calendar_factor__"].replace(
        [np.inf, -np.inf], np.nan
    )
    observed = observed.dropna(subset=["__observed_calendar_factor__"])

    if observed.empty:
        return out, created

    factor_specs = [
        (["calendar_month_num"], "same_month_calendar_factor", None),
    ]
    if "region" in out.columns:
        factor_specs.append((["calendar_month_num", "region"], "same_month_region_calendar_factor", "same_month_calendar_factor"))
    if "population_city_type" in out.columns:
        factor_specs.append((["calendar_month_num", "population_city_type"], "same_month_population_type_calendar_factor", "same_month_calendar_factor"))
    if "lag1_rto_size_bin" in out.columns:
        factor_specs.append((["calendar_month_num", "lag1_rto_size_bin"], "same_month_lag1_size_calendar_factor", "same_month_calendar_factor"))

    for group_cols, factor_col, fallback_col in factor_specs:
        if any(c not in out.columns for c in group_cols):
            continue
        factor = _expanding_median_factor_from_history(
            out,
            observed,
            group_cols=group_cols,
            time_col=time_col,
            obs_col="__observed_calendar_factor__",
            out_col=factor_col,
        )
        out[factor_col] = factor
        if fallback_col is not None and fallback_col in out.columns:
            out[factor_col] = out[factor_col].fillna(out[fallback_col])
        mark(factor_col)

    if "same_month_calendar_factor" in out.columns:
        out["calendar_growth_multiplier"] = (
            out["month_day_ratio_to_previous_month"] * out["same_month_calendar_factor"]
        )
        out["calendar_heuristic_prediction"] = out[pto_lag_1] * out["calendar_growth_multiplier"]
        out["log_calendar_heuristic_prediction"] = _safe_log_external(
            out["calendar_heuristic_prediction"], eps=eps
        )
        out["log_pto_lag_1_to_calendar_heuristic"] = (
            out[f"log_{target_col}_lag_1"] - out["log_calendar_heuristic_prediction"]
        )
        mark("calendar_growth_multiplier")
        mark("calendar_heuristic_prediction")
        mark("log_calendar_heuristic_prediction")
        mark("log_pto_lag_1_to_calendar_heuristic")

    pred_specs = [
        ("same_month_region_calendar_factor", "calendar_heuristic_region_prediction"),
        ("same_month_population_type_calendar_factor", "calendar_heuristic_population_type_prediction"),
        ("same_month_lag1_size_calendar_factor", "calendar_heuristic_lag1_size_prediction"),
    ]
    for factor_col, pred_col in pred_specs:
        if factor_col not in out.columns:
            continue
        out[pred_col] = out[pto_lag_1] * out["month_day_ratio_to_previous_month"] * out[factor_col]
        out[f"log_{pred_col}"] = _safe_log_external(out[pred_col], eps=eps)
        mark(pred_col)
        mark(f"log_{pred_col}")

    created = sorted(set(c for c in created if c in out.columns))
    out[created] = out[created].replace([np.inf, -np.inf], np.nan)
    return out, created

def _add_integrated_anchor_features_pair(
    history_features: pd.DataFrame,
    future_features: pd.DataFrame,
    history_raw: pd.DataFrame,
    future_raw: pd.DataFrame,
    *,
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    min_year: int = 2023,
    target_lags: tuple[int, ...] = (1, 2, 3, 6, 12, 13),
    add_calendar_anchor_features: bool = True,
    add_expanding_calendar_heuristic_features: bool = True,
    add_daily_anchor_features: bool = True,
    add_transition_anchor_features: bool = True,
    add_march_2024_features: bool = True,
    add_hierarchy_share_features: bool = True,
    add_operational_lag_features: bool = True,
    group_cols_for_transition: list | None = None,
    lagged_monthly_cols: tuple[str, ...] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    clip_growth_low: float = 0.70,
    clip_growth_high: float = 1.45,
    eps: float = 1e-9,
    fill_history_na: bool = False,
    fill_value: float = 0.0,
    downcast_float_features: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """
    Добавляет идеи из build_lag1_features_for_pipeline,
    build_anchor_features_for_pipeline и build_march_hierarchy_ops_features_for_pipeline
    к уже собранным leak-free features.

    Важная логика:
    - лаги target и monthly-cols строятся time-aware merge'ем из history_raw;
    - current target future_features не используется;
    - мартовские коэффициенты 2024 считаются только по history_raw;
    - expanding calendar heuristic считается только по history rows и строго прошлым time_idx;
    - мартовские коэффициенты доступны только для строк после марта 2024, чтобы не
      протаскивать target марта 2024 в более ранние train-строки.
    """
    h_feat = history_features.copy()
    f_feat = future_features.copy()

    h_feat["__part__"] = "history"
    f_feat["__part__"] = "future"
    h_feat["__row_order__"] = np.arange(len(h_feat))
    f_feat["__row_order__"] = np.arange(len(f_feat))

    combined = pd.concat([h_feat, f_feat], ignore_index=True, sort=False)
    combined = _ensure_time_year_month_external(
        combined,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        min_year=min_year,
        name="combined_features",
    )

    history_raw = _ensure_time_year_month_external(
        history_raw,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        min_year=min_year,
        name="history_raw",
    )
    future_raw = _ensure_time_year_month_external(
        future_raw,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        min_year=min_year,
        name="future_raw",
    )

    if id_col not in combined.columns:
        raise ValueError(f"В features нет id_col='{id_col}'.")
    if target_col not in history_raw.columns:
        raise ValueError(f"В history_raw нет target_col='{target_col}'.")

    created: list[str] = []

    def _mark(col):
        if col in combined.columns and col not in created:
            created.append(col)

    def _merge_history_lag(source_history, source_col, lag, feature_name, by_cols):
        nonlocal combined
        if source_col not in source_history.columns:
            return False

        lag_df = source_history[by_cols + [time_col, source_col]].copy()
        lag_df = lag_df.dropna(subset=[source_col])
        lag_df[time_col] = lag_df[time_col].astype(int) + int(lag)
        lag_df = lag_df.rename(columns={source_col: feature_name})
        lag_df = lag_df.drop_duplicates(subset=by_cols + [time_col], keep="last")

        combined = combined.drop(columns=[feature_name], errors="ignore")
        combined = combined.merge(
            lag_df[by_cols + [time_col, feature_name]],
            on=by_cols + [time_col],
            how="left",
        )
        _mark(feature_name)
        return True

    # ------------------------------------------------------------
    # 1. Calendar day features + target lags/daily lags
    # ------------------------------------------------------------
    combined["calendar_month_num"] = _calendar_month_from_time(combined[time_col], min_year)
    combined["calendar_year"] = _year_from_time(combined[time_col], min_year)
    combined["days_in_month"] = _days_in_month_from_year_month(
        combined["calendar_year"], combined["calendar_month_num"]
    )
    _mark("calendar_month_num")
    _mark("days_in_month")

    for lag in target_lags:
        lag = int(lag)
        pto_lag_col = f"{target_col}_lag_{lag}"
        _merge_history_lag(history_raw, target_col, lag, pto_lag_col, [id_col])

        lag_time = combined[time_col].astype(int) - lag
        lag_year = _year_from_time(lag_time, min_year)
        lag_month = _calendar_month_from_time(lag_time, min_year)
        days_lag_col = f"days_lag_{lag}"
        combined[days_lag_col] = _days_in_month_from_year_month(lag_year, lag_month)
        _mark(days_lag_col)

        daily_col = f"daily_{target_col}_lag_{lag}"
        combined[daily_col] = _safe_divide_external(
            combined[pto_lag_col],
            combined[days_lag_col],
            fill=np.nan,
        )
        _mark(daily_col)

        log_pto_col = f"log_{target_col}_lag_{lag}"
        log_daily_col = f"log_daily_{target_col}_lag_{lag}"
        combined[log_pto_col] = _safe_log_external(combined[pto_lag_col], eps=eps)
        combined[log_daily_col] = _safe_log_external(combined[daily_col], eps=eps)
        _mark(log_pto_col)
        _mark(log_daily_col)

    if "days_lag_1" in combined.columns:
        combined["days_in_prev_month"] = combined["days_lag_1"]
        _mark("days_in_prev_month")

    if add_calendar_anchor_features and {f"{target_col}_lag_1", "days_in_month", "days_in_prev_month"}.issubset(combined.columns):
        combined["month_day_ratio_to_previous_month"] = _safe_divide_external(
            combined["days_in_month"],
            combined["days_in_prev_month"],
            fill=np.nan,
        )
        combined["day_scaled_lag1_prediction"] = (
            combined[f"{target_col}_lag_1"] * combined["month_day_ratio_to_previous_month"]
        )
        combined["log_day_scaled_lag1_prediction"] = _safe_log_external(
            combined["day_scaled_lag1_prediction"], eps=eps
        )
        _mark("month_day_ratio_to_previous_month")
        _mark("day_scaled_lag1_prediction")
        _mark("log_day_scaled_lag1_prediction")

        for lag in [2, 3, 6, 12]:
            daily_col = f"daily_{target_col}_lag_{lag}"
            pred_col = f"day_scaled_lag{lag}_prediction"
            if daily_col in combined.columns:
                combined[pred_col] = combined[daily_col] * combined["days_in_month"]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

    if add_calendar_anchor_features and add_expanding_calendar_heuristic_features:
        combined, calendar_created = _add_expanding_calendar_heuristic_features(
            combined,
            id_col=id_col,
            time_col=time_col,
            target_col=target_col,
            min_year=min_year,
            eps=eps,
        )
        for c in calendar_created:
            _mark(c)

    # ------------------------------------------------------------
    # 2. Recent daily baselines + trend anchors + same-transition anchors
    # ------------------------------------------------------------
    if add_daily_anchor_features:
        daily_1 = f"daily_{target_col}_lag_1"
        daily_2 = f"daily_{target_col}_lag_2"
        daily_3 = f"daily_{target_col}_lag_3"

        if {daily_1, daily_2}.issubset(combined.columns):
            combined["recent_daily_mean_2"] = combined[[daily_1, daily_2]].mean(axis=1)
            combined["recent_daily_mean_2_prediction"] = combined["recent_daily_mean_2"] * combined["days_in_month"]
            combined["log_recent_daily_mean_2_prediction"] = _safe_log_external(
                combined["recent_daily_mean_2_prediction"], eps=eps
            )
            _mark("recent_daily_mean_2")
            _mark("recent_daily_mean_2_prediction")
            _mark("log_recent_daily_mean_2_prediction")

        if {daily_1, daily_2, daily_3}.issubset(combined.columns):
            recent_cols = [daily_1, daily_2, daily_3]
            combined["recent_daily_mean_3"] = combined[recent_cols].mean(axis=1)
            combined["recent_daily_median_3"] = combined[recent_cols].median(axis=1)
            combined["recent_daily_weighted_123"] = (
                0.60 * combined[daily_1]
                + 0.25 * combined[daily_2]
                + 0.15 * combined[daily_3]
            )

            for col in ["recent_daily_mean_3", "recent_daily_median_3", "recent_daily_weighted_123"]:
                pred_col = f"{col}_prediction"
                combined[pred_col] = combined[col] * combined["days_in_month"]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(col)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

            combined["daily_growth_1_2"] = _safe_divide_external(combined[daily_1], combined[daily_2], fill=np.nan)
            combined["daily_growth_1_3"] = _safe_divide_external(combined[daily_1], combined[daily_3], fill=np.nan)
            combined["daily_growth_1_2_clipped"] = combined["daily_growth_1_2"].clip(0.70, 1.35)
            combined["daily_growth_1_3_clipped"] = combined["daily_growth_1_3"].clip(0.70, 1.35)
            _mark("daily_growth_1_2")
            _mark("daily_growth_1_3")
            _mark("daily_growth_1_2_clipped")
            _mark("daily_growth_1_3_clipped")

            if "day_scaled_lag1_prediction" in combined.columns:
                combined["trend_day_scaled_lag1_prediction"] = (
                    combined["day_scaled_lag1_prediction"] * combined["daily_growth_1_2_clipped"]
                )
                combined["trend_day_scaled_lag1_prediction_smooth"] = (
                    combined["day_scaled_lag1_prediction"] * np.sqrt(combined["daily_growth_1_2_clipped"])
                )
                combined["log_trend_day_scaled_lag1_prediction"] = _safe_log_external(
                    combined["trend_day_scaled_lag1_prediction"], eps=eps
                )
                combined["log_trend_day_scaled_lag1_prediction_smooth"] = _safe_log_external(
                    combined["trend_day_scaled_lag1_prediction_smooth"], eps=eps
                )
                _mark("trend_day_scaled_lag1_prediction")
                _mark("trend_day_scaled_lag1_prediction_smooth")
                _mark("log_trend_day_scaled_lag1_prediction")
                _mark("log_trend_day_scaled_lag1_prediction_smooth")

    if add_transition_anchor_features:
        pto12, pto13 = f"{target_col}_lag_12", f"{target_col}_lag_13"
        daily1, daily12, daily13 = (
            f"daily_{target_col}_lag_1",
            f"daily_{target_col}_lag_12",
            f"daily_{target_col}_lag_13",
        )

        if {pto12, pto13, f"{target_col}_lag_1"}.issubset(combined.columns):
            combined["same_transition_last_year_factor"] = _safe_divide_external(
                combined[pto12], combined[pto13], fill=np.nan
            )
            combined["same_transition_last_year_factor_clipped"] = (
                combined["same_transition_last_year_factor"].clip(0.75, 1.45)
            )
            combined["same_transition_last_year_prediction"] = (
                combined[f"{target_col}_lag_1"] * combined["same_transition_last_year_factor_clipped"]
            )
            combined["log_same_transition_last_year_prediction"] = _safe_log_external(
                combined["same_transition_last_year_prediction"], eps=eps
            )
            _mark("same_transition_last_year_factor")
            _mark("same_transition_last_year_factor_clipped")
            _mark("same_transition_last_year_prediction")
            _mark("log_same_transition_last_year_prediction")

        if {daily1, daily12, daily13, "days_in_month"}.issubset(combined.columns):
            combined["daily_same_transition_last_year_factor"] = _safe_divide_external(
                combined[daily12], combined[daily13], fill=np.nan
            )
            combined["daily_same_transition_last_year_factor_clipped"] = (
                combined["daily_same_transition_last_year_factor"].clip(0.75, 1.35)
            )
            combined["daily_same_transition_last_year_prediction"] = (
                combined[daily1]
                * combined["daily_same_transition_last_year_factor_clipped"]
                * combined["days_in_month"]
            )
            combined["log_daily_same_transition_last_year_prediction"] = _safe_log_external(
                combined["daily_same_transition_last_year_prediction"], eps=eps
            )
            _mark("daily_same_transition_last_year_factor")
            _mark("daily_same_transition_last_year_factor_clipped")
            _mark("daily_same_transition_last_year_prediction")
            _mark("log_daily_same_transition_last_year_prediction")

            combined["global_daily_transition_factor"] = (
                combined.groupby(time_col, dropna=False)["daily_same_transition_last_year_factor_clipped"]
                .transform("median")
            )
            combined["global_transition_prediction"] = (
                combined[daily1] * combined["global_daily_transition_factor"] * combined["days_in_month"]
            )
            combined["log_global_transition_prediction"] = _safe_log_external(
                combined["global_transition_prediction"], eps=eps
            )
            _mark("global_daily_transition_factor")
            _mark("global_transition_prediction")
            _mark("log_global_transition_prediction")

            if group_cols_for_transition is None:
                group_cols_for_transition = []
                for col in ["region", "trading_area_category", "opening_date_category", "settlement"]:
                    if col in combined.columns:
                        group_cols_for_transition.append([col])
                if {"region", "trading_area_category"}.issubset(combined.columns):
                    group_cols_for_transition.append(["region", "trading_area_category"])
                if {"region", "opening_date_category"}.issubset(combined.columns):
                    group_cols_for_transition.append(["region", "opening_date_category"])

            for group_cols in group_cols_for_transition:
                existing = [c for c in group_cols if c in combined.columns]
                if not existing:
                    continue
                group_name = "__".join(existing)
                factor_col = f"{group_name}_daily_transition_factor"
                pred_col = f"{group_name}_transition_prediction"
                combined[factor_col] = (
                    combined.groupby([time_col] + existing, dropna=False)["daily_same_transition_last_year_factor_clipped"]
                    .transform("median")
                )
                combined[pred_col] = combined[daily1] * combined[factor_col] * combined["days_in_month"]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(factor_col)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

        # Blends
        blend_sources = {
            "blend_day_scaled_and_last_year_prediction": (
                "day_scaled_lag1_prediction",
                "daily_same_transition_last_year_prediction",
                0.50,
                0.50,
            ),
            "blend_day_scaled_and_global_prediction": (
                "day_scaled_lag1_prediction",
                "global_transition_prediction",
                0.50,
                0.50,
            ),
            "blend_day_scaled_and_recent_mean_prediction": (
                "day_scaled_lag1_prediction",
                "recent_daily_mean_3_prediction",
                0.60,
                0.40,
            ),
        }
        for out_col, (c1, c2, w1, w2) in blend_sources.items():
            if {c1, c2}.issubset(combined.columns):
                combined[out_col] = w1 * combined[c1] + w2 * combined[c2]
                combined[f"log_{out_col}"] = _safe_log_external(combined[out_col], eps=eps)
                _mark(out_col)
                _mark(f"log_{out_col}")

        ratio_pairs = {
            "day_scaled_to_lag1_ratio": ("day_scaled_lag1_prediction", f"{target_col}_lag_1"),
            "last_year_anchor_to_day_scaled_ratio": ("daily_same_transition_last_year_prediction", "day_scaled_lag1_prediction"),
            "global_anchor_to_day_scaled_ratio": ("global_transition_prediction", "day_scaled_lag1_prediction"),
            "recent_mean_anchor_to_day_scaled_ratio": ("recent_daily_mean_3_prediction", "day_scaled_lag1_prediction"),
        }
        for out_col, (a, b) in ratio_pairs.items():
            if {a, b}.issubset(combined.columns):
                combined[out_col] = _safe_divide_external(combined[a], combined[b], fill=np.nan)
                _mark(out_col)

        log_diff_pairs = [(1, 2), (1, 3), (1, 6), (1, 12), (2, 3), (3, 6), (12, 13)]
        for left, right in log_diff_pairs:
            a, b = f"{target_col}_lag_{left}", f"{target_col}_lag_{right}"
            if {a, b}.issubset(combined.columns):
                out_col = f"log_{target_col}_diff_lag_{left}_{right}"
                combined[out_col] = _safe_log_external(combined[a], eps=eps) - _safe_log_external(combined[b], eps=eps)
                _mark(out_col)

            da, db = f"daily_{target_col}_lag_{left}", f"daily_{target_col}_lag_{right}"
            if {da, db}.issubset(combined.columns):
                out_col = f"log_daily_{target_col}_diff_lag_{left}_{right}"
                combined[out_col] = _safe_log_external(combined[da], eps=eps) - _safe_log_external(combined[db], eps=eps)
                _mark(out_col)

    # ------------------------------------------------------------
    # 3. March 2024 growth coefficients
    # ------------------------------------------------------------
    if add_march_2024_features:
        march_2024_idx = (2024 - int(min_year)) * 12 + (3 - 1)
        after_march_2024 = combined[time_col].astype(int) > march_2024_idx

        hist = history_raw.copy()
        hist["calendar_month_num"] = _calendar_month_from_time(hist[time_col], min_year)
        hist["calendar_year"] = _year_from_time(hist[time_col], min_year)

        hist_2024 = hist[
            (hist["calendar_year"].astype(int) == 2024)
            & (hist["calendar_month_num"].isin([2, 3]))
            & hist[target_col].notna()
        ].copy()

        if not hist_2024.empty:
            store_pivot = (
                hist_2024.pivot_table(
                    index=id_col,
                    columns="calendar_month_num",
                    values=target_col,
                    aggfunc="first",
                )
                .rename(columns={2: "pto_feb_2024", 3: "pto_mar_2024"})
                .reset_index()
            )
            if {"pto_feb_2024", "pto_mar_2024"}.issubset(store_pivot.columns):
                store_pivot["store_feb_mar_growth_2024"] = _safe_divide_external(
                    store_pivot["pto_mar_2024"], store_pivot["pto_feb_2024"], fill=np.nan
                ).clip(clip_growth_low, clip_growth_high)
                combined = combined.drop(columns=["store_feb_mar_growth_2024"], errors="ignore")
                combined = combined.merge(
                    store_pivot[[id_col, "store_feb_mar_growth_2024"]],
                    on=id_col,
                    how="left",
                )
                combined.loc[~after_march_2024, "store_feb_mar_growth_2024"] = np.nan
                _mark("store_feb_mar_growth_2024")

            def _add_group_growth(group_col, out_col):
                nonlocal combined
                if group_col not in hist_2024.columns or group_col not in combined.columns:
                    return
                group_agg = (
                    hist_2024.groupby([group_col, "calendar_month_num"], dropna=False)[target_col]
                    .sum()
                    .reset_index()
                )
                pivot = (
                    group_agg.pivot_table(
                        index=group_col,
                        columns="calendar_month_num",
                        values=target_col,
                        aggfunc="first",
                    )
                    .rename(columns={2: "pto_feb_2024", 3: "pto_mar_2024"})
                    .reset_index()
                )
                if {"pto_feb_2024", "pto_mar_2024"}.issubset(pivot.columns):
                    pivot[out_col] = _safe_divide_external(
                        pivot["pto_mar_2024"], pivot["pto_feb_2024"], fill=np.nan
                    ).clip(clip_growth_low, clip_growth_high)
                    combined = combined.drop(columns=[out_col], errors="ignore")
                    combined = combined.merge(pivot[[group_col, out_col]], on=group_col, how="left")
                    combined.loc[~after_march_2024, out_col] = np.nan
                    _mark(out_col)

            _add_group_growth("region", "region_feb_mar_growth_2024")
            _add_group_growth("settlement", "settlement_feb_mar_growth_2024")
            _add_group_growth("trading_area_category", "trading_area_feb_mar_growth_2024")

            feb_sum = hist_2024.loc[hist_2024["calendar_month_num"] == 2, target_col].sum()
            mar_sum = hist_2024.loc[hist_2024["calendar_month_num"] == 3, target_col].sum()
            global_growth = np.nan
            if pd.notna(feb_sum) and feb_sum > 0:
                global_growth = np.clip(mar_sum / feb_sum, clip_growth_low, clip_growth_high)

            combined["global_feb_mar_growth_2024"] = np.where(after_march_2024, global_growth, np.nan)
            _mark("global_feb_mar_growth_2024")

            growth_cols = [
                "store_feb_mar_growth_2024",
                "region_feb_mar_growth_2024",
                "settlement_feb_mar_growth_2024",
                "trading_area_category_feb_mar_growth_2024",  # backward-safe typo guard
                "trading_area_feb_mar_growth_2024",
                "global_feb_mar_growth_2024",
            ]
            # unify expected column name from the prompt
            if "trading_area_category_feb_mar_growth_2024" in combined.columns and "trading_area_feb_mar_growth_2024" not in combined.columns:
                combined["trading_area_feb_mar_growth_2024"] = combined["trading_area_category_feb_mar_growth_2024"]
                _mark("trading_area_feb_mar_growth_2024")

            for col in [
                "store_feb_mar_growth_2024",
                "region_feb_mar_growth_2024",
                "settlement_feb_mar_growth_2024",
                "trading_area_feb_mar_growth_2024",
                "global_feb_mar_growth_2024",
            ]:
                if col not in combined.columns or f"{target_col}_lag_1" not in combined.columns:
                    continue
                combined[col] = combined[col].fillna(combined["global_feb_mar_growth_2024"])
                pred_col = f"lag1_x_{col}"
                combined[pred_col] = combined[f"{target_col}_lag_1"] * combined[col]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

            if {"day_scaled_lag1_prediction", "lag1_x_store_feb_mar_growth_2024"}.issubset(combined.columns):
                combined["blend_day_scaled_and_store_march_growth"] = (
                    0.50 * combined["day_scaled_lag1_prediction"]
                    + 0.50 * combined["lag1_x_store_feb_mar_growth_2024"]
                )
                combined["log_blend_day_scaled_and_store_march_growth"] = _safe_log_external(
                    combined["blend_day_scaled_and_store_march_growth"], eps=eps
                )
                _mark("blend_day_scaled_and_store_march_growth")
                _mark("log_blend_day_scaled_and_store_march_growth")

            if {"day_scaled_lag1_prediction", "lag1_x_region_feb_mar_growth_2024"}.issubset(combined.columns):
                combined["blend_day_scaled_and_region_march_growth"] = (
                    0.50 * combined["day_scaled_lag1_prediction"]
                    + 0.50 * combined["lag1_x_region_feb_mar_growth_2024"]
                )
                combined["log_blend_day_scaled_and_region_march_growth"] = _safe_log_external(
                    combined["blend_day_scaled_and_region_march_growth"], eps=eps
                )
                _mark("blend_day_scaled_and_region_march_growth")
                _mark("log_blend_day_scaled_and_region_march_growth")

    # ------------------------------------------------------------
    # 4. Hierarchical share features
    # ------------------------------------------------------------
    if add_hierarchy_share_features:
        pto_lag_1 = f"{target_col}_lag_1"
        if pto_lag_1 in combined.columns:
            if "region" in combined.columns:
                combined["region_pto_lag1_mean"] = (
                    combined.groupby([time_col, "region"], dropna=False)[pto_lag_1].transform("mean")
                )
                combined["store_share_region_lag1"] = _safe_divide_external(
                    combined[pto_lag_1], combined["region_pto_lag1_mean"], fill=np.nan
                )
                _mark("region_pto_lag1_mean")
                _mark("store_share_region_lag1")

            if "settlement" in combined.columns:
                combined["settlement_pto_lag1_mean"] = (
                    combined.groupby([time_col, "settlement"], dropna=False)[pto_lag_1].transform("mean")
                )
                combined["store_share_settlement_lag1"] = _safe_divide_external(
                    combined[pto_lag_1], combined["settlement_pto_lag1_mean"], fill=np.nan
                )
                _mark("settlement_pto_lag1_mean")
                _mark("store_share_settlement_lag1")

            if "trading_area_category" in combined.columns:
                combined["area_category_pto_lag1_mean"] = (
                    combined.groupby([time_col, "trading_area_category"], dropna=False)[pto_lag_1].transform("mean")
                )
                combined["store_share_area_category_lag1"] = _safe_divide_external(
                    combined[pto_lag_1], combined["area_category_pto_lag1_mean"], fill=np.nan
                )
                _mark("area_category_pto_lag1_mean")
                _mark("store_share_area_category_lag1")

            if "region" in combined.columns:
                for lag in [2, 3]:
                    lag_col = f"{target_col}_lag_{lag}"
                    if lag_col not in combined.columns:
                        continue
                    mean_col = f"region_pto_lag{lag}_mean"
                    share_col = f"store_share_region_lag{lag}"
                    combined[mean_col] = (
                        combined.groupby([time_col, "region"], dropna=False)[lag_col].transform("mean")
                    )
                    combined[share_col] = _safe_divide_external(combined[lag_col], combined[mean_col], fill=np.nan)
                    _mark(mean_col)
                    _mark(share_col)

                share_cols = [c for c in ["store_share_region_lag1", "store_share_region_lag2", "store_share_region_lag3"] if c in combined.columns]
                if share_cols:
                    combined["store_share_region_mean_3"] = combined[share_cols].mean(axis=1)
                    _mark("store_share_region_mean_3")
                if {"store_share_region_lag1", "store_share_region_lag2"}.issubset(combined.columns):
                    combined["store_share_region_change_1_2"] = _safe_divide_external(
                        combined["store_share_region_lag1"], combined["store_share_region_lag2"], fill=np.nan
                    )
                    combined["store_share_region_change_1_2_clipped"] = (
                        combined["store_share_region_change_1_2"].clip(0.70, 1.35)
                    )
                    _mark("store_share_region_change_1_2")
                    _mark("store_share_region_change_1_2_clipped")

            if "settlement" in combined.columns:
                for lag in [2, 3]:
                    lag_col = f"{target_col}_lag_{lag}"
                    if lag_col not in combined.columns:
                        continue
                    mean_col = f"settlement_pto_lag{lag}_mean"
                    share_col = f"store_share_settlement_lag{lag}"
                    combined[mean_col] = (
                        combined.groupby([time_col, "settlement"], dropna=False)[lag_col].transform("mean")
                    )
                    combined[share_col] = _safe_divide_external(combined[lag_col], combined[mean_col], fill=np.nan)
                    _mark(mean_col)
                    _mark(share_col)

                share_cols = [
                    c for c in ["store_share_settlement_lag1", "store_share_settlement_lag2", "store_share_settlement_lag3"]
                    if c in combined.columns
                ]
                if share_cols:
                    combined["store_share_settlement_mean_3"] = combined[share_cols].mean(axis=1)
                    _mark("store_share_settlement_mean_3")

            if {"store_share_region_mean_3", "lag1_x_region_feb_mar_growth_2024"}.issubset(combined.columns):
                combined["region_anchor_x_store_share_mean_3"] = (
                    combined["lag1_x_region_feb_mar_growth_2024"]
                    * combined["store_share_region_mean_3"].clip(0.50, 1.80)
                )
                combined["log_region_anchor_x_store_share_mean_3"] = _safe_log_external(
                    combined["region_anchor_x_store_share_mean_3"], eps=eps
                )
                _mark("region_anchor_x_store_share_mean_3")
                _mark("log_region_anchor_x_store_share_mean_3")

            if {"store_share_region_change_1_2_clipped", "day_scaled_lag1_prediction"}.issubset(combined.columns):
                combined["day_scaled_x_region_share_change"] = (
                    combined["day_scaled_lag1_prediction"]
                    * combined["store_share_region_change_1_2_clipped"]
                )
                combined["log_day_scaled_x_region_share_change"] = _safe_log_external(
                    combined["day_scaled_x_region_share_change"], eps=eps
                )
                _mark("day_scaled_x_region_share_change")
                _mark("log_day_scaled_x_region_share_change")

    # ------------------------------------------------------------
    # 5. Operational lag features
    # ------------------------------------------------------------
    if add_operational_lag_features:
        # lag_1, lag_3, lag_12 из исходных operational cols
        for col in lagged_monthly_cols:
            if col not in history_raw.columns:
                continue
            for lag in [1, 3, 12]:
                _merge_history_lag(history_raw, col, lag, f"{col}_lag_{lag}", [id_col])

            # rolling mean/median по прошлым 3 наблюдениям.
            raw_combined = pd.concat(
                [
                    history_raw.assign(__part_roll__="history"),
                    future_raw.assign(__part_roll__="future"),
                ],
                axis=0,
                ignore_index=True,
                sort=False,
            )
            raw_combined = _ensure_time_year_month_external(
                raw_combined,
                year_col=year_col,
                month_col=month_col,
                time_col=time_col,
                min_year=min_year,
                name=f"raw_combined_for_{col}",
            )
            raw_combined = raw_combined.sort_values([id_col, time_col], kind="mergesort")
            raw_combined["__x__"] = np.where(raw_combined["__part_roll__"].eq("history"), raw_combined[col], np.nan)
            shifted = raw_combined.groupby(id_col, sort=False)["__x__"].shift(1)
            raw_combined[f"{col}_roll_mean_3"] = (
                shifted.groupby(raw_combined[id_col], sort=False)
                .rolling(3, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )
            raw_combined[f"{col}_roll_median_3"] = (
                shifted.groupby(raw_combined[id_col], sort=False)
                .rolling(3, min_periods=1)
                .median()
                .reset_index(level=0, drop=True)
            )
            roll_map = raw_combined[[id_col, time_col, f"{col}_roll_mean_3", f"{col}_roll_median_3"]].drop_duplicates(
                [id_col, time_col],
                keep="last",
            )
            combined = combined.drop(columns=[f"{col}_roll_mean_3", f"{col}_roll_median_3"], errors="ignore")
            combined = combined.merge(roll_map, on=[id_col, time_col], how="left")
            _mark(f"{col}_roll_mean_3")
            _mark(f"{col}_roll_median_3")

        if {f"{target_col}_lag_1", "working_hours_per_day_lag_1"}.issubset(combined.columns):
            combined["pto_per_working_hour_lag1"] = _safe_divide_external(
                combined[f"{target_col}_lag_1"], combined["working_hours_per_day_lag_1"], fill=np.nan
            )
            combined["log_pto_per_working_hour_lag1"] = _safe_log_external(
                combined["pto_per_working_hour_lag1"], eps=eps
            )
            _mark("pto_per_working_hour_lag1")
            _mark("log_pto_per_working_hour_lag1")

        if {f"{target_col}_lag_1", "number_of_cash_registers"}.issubset(combined.columns):
            combined["pto_per_cash_register_lag1"] = _safe_divide_external(
                combined[f"{target_col}_lag_1"],
                combined["number_of_cash_registers"].astype(float),
                fill=np.nan,
            )
            combined["log_pto_per_cash_register_lag1"] = _safe_log_external(
                combined["pto_per_cash_register_lag1"], eps=eps
            )
            _mark("pto_per_cash_register_lag1")
            _mark("log_pto_per_cash_register_lag1")

        if {f"{target_col}_lag_1", "working_hours_per_day_lag_1", "number_of_cash_registers"}.issubset(combined.columns):
            denom = combined["working_hours_per_day_lag_1"] * combined["number_of_cash_registers"].astype(float)
            combined["pto_per_cash_register_hour_lag1"] = _safe_divide_external(
                combined[f"{target_col}_lag_1"], denom, fill=np.nan
            )
            combined["log_pto_per_cash_register_hour_lag1"] = _safe_log_external(
                combined["pto_per_cash_register_hour_lag1"], eps=eps
            )
            _mark("pto_per_cash_register_hour_lag1")
            _mark("log_pto_per_cash_register_hour_lag1")

    # ------------------------------------------------------------
    # Final cleanup and restore parts
    # ------------------------------------------------------------
    created = sorted(set(c for c in created if c in combined.columns))
    combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)

    if fill_history_na and created:
        combined[created] = combined[created].fillna(fill_value)

    combined = combined.drop(columns=["calendar_year"], errors="ignore")

    history_out = (
        combined[combined["__part__"] == "history"]
        .sort_values("__row_order__")
        .drop(columns=["__part__", "__row_order__"], errors="ignore")
        .reset_index(drop=True)
    )
    future_out = (
        combined[combined["__part__"] == "future"]
        .sort_values("__row_order__")
        .drop(columns=["__part__", "__row_order__"], errors="ignore")
        .reset_index(drop=True)
    )

    # Align columns
    ordered_cols = list(history_out.columns)
    for c in future_out.columns:
        if c not in ordered_cols:
            ordered_cols.append(c)
    for c in ordered_cols:
        if c not in history_out.columns:
            history_out[c] = np.nan
        if c not in future_out.columns:
            future_out[c] = np.nan
    history_out = history_out[ordered_cols]
    future_out = future_out[ordered_cols]

    if downcast_float_features:
        for df_ in (history_out, future_out):
            float_cols = [c for c in df_.select_dtypes(include=["float64"]).columns if c != target_col]
            if float_cols:
                df_[float_cols] = df_[float_cols].astype("float32")

    return history_out, future_out, created


def build_retail_features_with_split(
    df: pd.DataFrame,
    *,
    # split config
    split_mode: Literal["train_val", "train_val_test", "train_test"] = "train_val",
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    min_year: int = 2023,
    val_time_idx: Optional[int] = None,
    val_year: Optional[int] = None,
    val_month: Optional[int] = None,
    test_time_idx: Optional[int] = None,
    test_year: Optional[int] = None,
    test_month: Optional[int] = None,
    create_test_if_missing: bool = True,
    expected_test_size: Optional[int] = 18657,
    future_unknown_cols: Optional[Sequence[str]] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    # feature config
    use_val_as_test_history: bool = True,
    add_integrated_anchor_features: bool = True,
    add_calendar_anchor_features: bool = True,
    add_expanding_calendar_heuristic_features: bool = True,
    add_daily_anchor_features: bool = True,
    add_transition_anchor_features: bool = True,
    add_march_2024_features: bool = True,
    add_hierarchy_share_features: bool = True,
    add_operational_lag_features: bool = True,
    group_cols_for_transition: list | None = None,
    lagged_monthly_cols: tuple[str, ...] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    clip_growth_low: float = 0.70,
    clip_growth_high: float = 1.45,
    eps: float = 1e-9,
    # pass-through to base generator
    base_feature_kwargs: dict | None = None,
    verbose: bool = True,
) -> dict:
    """
    Единая функция:
    1) делает leakage-safe split;
    2) создаёт artificial test month, если нужно;
    3) запускает базовый generate_retail_forecast_features_leakfree;
    4) добавляет все новые идеи:
       - month_day_ratio_to_previous_month;
       - day_scaled_lag1_prediction;
       - expanding calendar heuristic factors/predictions;
       - daily_pto_lag_*;
       - recent daily baselines;
       - trend-corrected anchors;
       - same-transition-last-year anchors;
       - global/group transition anchors;
       - March 2024 growth coefficients;
       - hierarchical share features;
       - lagged operational features.
    """
    base_feature_kwargs = {} if base_feature_kwargs is None else dict(base_feature_kwargs)
    # Defaults chosen to cover the full feature-idea corpus. User can override them in base_feature_kwargs.
    base_feature_kwargs.setdefault("target_rolling_windows", (3, 6, 12))
    base_feature_kwargs.setdefault("monthly_cross_lags", (1, 3, 12))
    base_feature_kwargs.setdefault(
        "monthly_cross_feature_types",
        ("promo_share", "promo_x_items", "items_x_hours", "cancellations_per_item", "cancellations_per_hour"),
    )

    split_result = make_panel_time_split(
        df,
        group_col=id_col,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        target_col=target_col,
        split_mode=split_mode,
        val_time_idx=val_time_idx,
        val_year=val_year,
        val_month=val_month,
        test_time_idx=test_time_idx,
        test_year=test_year,
        test_month=test_month,
        min_year=min_year,
        create_test_if_missing=create_test_if_missing,
        future_unknown_cols=future_unknown_cols,
        expected_test_size=expected_test_size if split_mode in {"train_val_test", "train_test"} else None,
        verbose=verbose,
    )

    if split_mode == "train_val":
        train_df, val_df = split_result
        test_df = None
    elif split_mode == "train_val_test":
        train_df, val_df, test_df = split_result
    else:
        train_df, test_df = split_result
        val_df = None

    output = generate_retail_forecast_features_leakfree(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        id_col=id_col,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        target_col=target_col,
        min_year=min_year,
        use_val_as_test_history=use_val_as_test_history,
        verbose=verbose,
        **base_feature_kwargs,
    )

    output["raw_splits"] = {
        "train": train_df,
        "val": val_df,
        "test": test_df,
    }

    if add_integrated_anchor_features:
        # validation pair: history = train, future = val
        if output.get("train_features") is not None and output.get("val_features") is not None:
            train_feat, val_feat, created = _add_integrated_anchor_features_pair(
                output["train_features"],
                output["val_features"],
                train_df,
                val_df,
                id_col=id_col,
                year_col=year_col,
                month_col=month_col,
                time_col=time_col,
                target_col=target_col,
                min_year=min_year,
                add_calendar_anchor_features=add_calendar_anchor_features,
                add_expanding_calendar_heuristic_features=add_expanding_calendar_heuristic_features,
                add_daily_anchor_features=add_daily_anchor_features,
                add_transition_anchor_features=add_transition_anchor_features,
                add_march_2024_features=add_march_2024_features,
                add_hierarchy_share_features=add_hierarchy_share_features,
                add_operational_lag_features=add_operational_lag_features,
                group_cols_for_transition=group_cols_for_transition,
                lagged_monthly_cols=lagged_monthly_cols,
                clip_growth_low=clip_growth_low,
                clip_growth_high=clip_growth_high,
                eps=eps,
                fill_history_na=base_feature_kwargs.get("fill_history_na", False),
                fill_value=base_feature_kwargs.get("fill_value", 0.0),
                downcast_float_features=base_feature_kwargs.get("downcast_float_features", True),
            )
            if base_feature_kwargs.get("drop_year_col", True):
                train_feat = train_feat.drop(columns=[year_col], errors="ignore")
                val_feat = val_feat.drop(columns=[year_col], errors="ignore")
            output["train_features"] = train_feat
            output["val_features"] = val_feat
            output.setdefault("feature_info", {}).setdefault("integrated_anchors", {})["val_created"] = created
            if verbose:
                print("Added integrated anchor features to train/val:", len(created))

        # test pair: history = train + val if requested and val exists, otherwise train
        if output.get("test_train_features") is not None and output.get("test_features") is not None:
            if val_df is not None and use_val_as_test_history:
                test_history_raw = pd.concat([train_df, val_df], axis=0, ignore_index=True, sort=False)
            else:
                test_history_raw = train_df.copy()

            test_train_feat, test_feat, created = _add_integrated_anchor_features_pair(
                output["test_train_features"],
                output["test_features"],
                test_history_raw,
                test_df,
                id_col=id_col,
                year_col=year_col,
                month_col=month_col,
                time_col=time_col,
                target_col=target_col,
                min_year=min_year,
                add_calendar_anchor_features=add_calendar_anchor_features,
                add_expanding_calendar_heuristic_features=add_expanding_calendar_heuristic_features,
                add_daily_anchor_features=add_daily_anchor_features,
                add_transition_anchor_features=add_transition_anchor_features,
                add_march_2024_features=add_march_2024_features,
                add_hierarchy_share_features=add_hierarchy_share_features,
                add_operational_lag_features=add_operational_lag_features,
                group_cols_for_transition=group_cols_for_transition,
                lagged_monthly_cols=lagged_monthly_cols,
                clip_growth_low=clip_growth_low,
                clip_growth_high=clip_growth_high,
                eps=eps,
                fill_history_na=base_feature_kwargs.get("fill_history_na", False),
                fill_value=base_feature_kwargs.get("fill_value", 0.0),
                downcast_float_features=base_feature_kwargs.get("downcast_float_features", True),
            )
            if base_feature_kwargs.get("drop_year_col", True):
                test_train_feat = test_train_feat.drop(columns=[year_col], errors="ignore")
                test_feat = test_feat.drop(columns=[year_col], errors="ignore")
            output["test_train_features"] = test_train_feat
            output["test_features"] = test_feat
            output.setdefault("feature_info", {}).setdefault("integrated_anchors", {})["test_created"] = created
            if verbose:
                print("Added integrated anchor features to train/test:", len(created))

    # Recompute model columns after enhancement.
    ref = output["test_train_features"] if output.get("test_train_features") is not None else output.get("train_features")
    if ref is not None:
        exclude = {target_col}
        output["feature_cols"] = [c for c in ref.columns if c not in exclude]
        output["categorical_cols"] = [
            c for c in output["feature_cols"]
            if c in ref.columns and (
                pd.api.types.is_object_dtype(ref[c]) or pd.api.types.is_categorical_dtype(ref[c])
            )
        ]

    gc.collect()
    return output


# ============================================================
# Target engineering utilities
# ============================================================

def make_forecast_target(
    df: pd.DataFrame,
    *,
    mode: Literal[
        "original",
        "growth_log",
        "day_scaled_residual_log",
        "calendar_residual_log",
        "daily_rate_log",
        "direct_log1p",
    ] = "growth_log",
    target_col: str = "pto",
    eps: float = 1e-9,
) -> pd.Series:
    """
    Builds model target for the main target formulations used in the experiments.

    Required feature columns by mode:
    - growth_log: pto_lag_1;
    - day_scaled_residual_log: day_scaled_lag1_prediction;
    - calendar_residual_log: calendar_heuristic_prediction;
    - daily_rate_log: n_days_in_month or days_in_month;
    - direct_log1p: only target_col.
    """
    if target_col not in df.columns:
        raise ValueError(f"В df нет target_col='{target_col}'.")

    y = pd.to_numeric(df[target_col], errors="coerce").astype(float)

    if mode == "original":
        return y

    if mode == "direct_log1p":
        return np.log1p(y.clip(lower=0))

    if mode == "growth_log":
        anchor_col = f"{target_col}_lag_1"
        if anchor_col not in df.columns:
            raise ValueError(f"Для mode='growth_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(df[anchor_col], errors="coerce").astype(float)
        return np.log(np.clip(y, eps, None)) - np.log(np.clip(anchor, eps, None))

    if mode == "day_scaled_residual_log":
        anchor_col = "day_scaled_lag1_prediction"
        if anchor_col not in df.columns:
            raise ValueError(f"Для mode='day_scaled_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(df[anchor_col], errors="coerce").astype(float)
        return np.log(np.clip(y, eps, None)) - np.log(np.clip(anchor, eps, None))

    if mode == "calendar_residual_log":
        anchor_col = "calendar_heuristic_prediction"
        if anchor_col not in df.columns:
            raise ValueError(f"Для mode='calendar_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(df[anchor_col], errors="coerce").astype(float)
        return np.log(np.clip(y, eps, None)) - np.log(np.clip(anchor, eps, None))

    if mode == "daily_rate_log":
        day_col = "n_days_in_month" if "n_days_in_month" in df.columns else "days_in_month"
        if day_col not in df.columns:
            raise ValueError("Для mode='daily_rate_log' нужна колонка 'n_days_in_month' или 'days_in_month'.")
        days = pd.to_numeric(df[day_col], errors="coerce").astype(float)
        daily_rate = y / days.replace(0, np.nan)
        return np.log(np.clip(daily_rate, eps, None))

    raise ValueError(f"Неизвестный mode='{mode}'.")


def inverse_forecast_target(
    raw_prediction,
    features: pd.DataFrame,
    *,
    mode: Literal[
        "original",
        "growth_log",
        "day_scaled_residual_log",
        "calendar_residual_log",
        "daily_rate_log",
        "direct_log1p",
    ] = "growth_log",
    target_col: str = "pto",
    residual_shrinkage: float = 1.0,
    clip_min: float = 0.0,
) -> np.ndarray:
    """
    Converts model output back to RTO forecast.

    residual_shrinkage is useful for residual modes, e.g. 0.35 or 0.50.
    """
    pred = np.asarray(raw_prediction, dtype=float)

    if mode == "original":
        out = pred
    elif mode == "direct_log1p":
        out = np.expm1(pred)
    elif mode == "growth_log":
        anchor_col = f"{target_col}_lag_1"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='growth_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(pred)
    elif mode == "day_scaled_residual_log":
        anchor_col = "day_scaled_lag1_prediction"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='day_scaled_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(residual_shrinkage * pred)
    elif mode == "calendar_residual_log":
        anchor_col = "calendar_heuristic_prediction"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='calendar_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(residual_shrinkage * pred)
    elif mode == "daily_rate_log":
        day_col = "n_days_in_month" if "n_days_in_month" in features.columns else "days_in_month"
        if day_col not in features.columns:
            raise ValueError("Для mode='daily_rate_log' нужна колонка 'n_days_in_month' или 'days_in_month'.")
        days = pd.to_numeric(features[day_col], errors="coerce").astype(float).to_numpy()
        out = days * np.exp(pred)
    else:
        raise ValueError(f"Неизвестный mode='{mode}'.")

    out = np.asarray(out, dtype=float)
    out = np.where(np.isfinite(out), out, np.nan)
    if clip_min is not None:
        out = np.maximum(out, clip_min)
    return out

# Новая функция для отбора признаков. Используется

In [19]:
# -*- coding: utf-8 -*-
"""
Self-contained adaptive ranked-add feature selection for the current X5/RTO notebook.

This v6 module is synchronized with another-x5-time-series.ipynb:
- contains all helper functions that were previously only in a commented old cell;
- accepts both fold schemas: name and fold_name;
- keeps the same ideas: periodic proxy re-ranking, stability-aware ranking,
  enhanced categorical policy, and floating backward pruning;
- uses the exact X5 categorical source columns via make_x5_retail_categorical_source_cols();
- categorical source columns are processed and scored as candidates, but are NOT inserted into the baseline by default;
- numeric categorical source columns can be used both as numeric features and as CatBoost categorical copies;
- string/object categorical source columns are used as CatBoost categorical features;
- baseline defaults to vladimir_features only and can be pruned via CV, except explicitly hard-protected features;
- is intended to be pasted/imported after build_retail_features_with_split is defined.
"""

from __future__ import annotations

import ast
import contextlib
import gc
import json
import os
import warnings
from typing import Any, Callable, Optional, Sequence

import numpy as np
import pandas as pd


# ============================================================
# Metrics
# ============================================================

def mape_score(
    y_true,
    y_pred,
    *,
    eps: float = 1e-9,
    clip_pred_lower: float | None = 0.0,
) -> float:
    """
    MAPE in percent.

    Formula:
        MAPE = 100 * mean(abs(y_pred - y_true) / abs(y_true))

    Parameters
    ----------
    y_true, y_pred:
        True and predicted values on the original target scale.
    eps:
        Minimal denominator threshold.
    clip_pred_lower:
        If not None, predictions are clipped from below.
        For RTO forecasting, negative predictions are usually invalid,
        so default is 0.0.

    Returns
    -------
    float
        MAPE in percentage points, e.g. 5.23 means 5.23%.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if clip_pred_lower is not None:
        y_pred = np.maximum(y_pred, clip_pred_lower)

    denom = np.where(np.abs(y_true) > eps, np.abs(y_true), np.nan)
    ape = np.abs(y_pred - y_true) / denom

    if np.all(np.isnan(ape)):
        return np.nan

    return float(np.nanmean(ape) * 100.0)


def competition_score_from_mape(mape: float) -> float:
    """
    Competition score:
        score = 100 * ((100 - min(MAPE, 100)) / 100) ** 2
    """
    if pd.isna(mape):
        return np.nan
    return float(100.0 * ((100.0 - min(float(mape), 100.0)) / 100.0) ** 2)


# ============================================================
# Utility helpers
# ============================================================

def _unique_keep_order(items: Sequence[Any] | None) -> list:
    """Unique values preserving original order."""
    if items is None:
        return []
    seen = set()
    out = []
    for x in items:
        if x is None:
            continue
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def _month_index(year: int, month: int, min_year: int) -> int:
    """Convert year/month to global monthly integer index."""
    year = int(year)
    month = int(month)
    if not 1 <= month <= 12:
        raise ValueError(f"month должен быть от 1 до 12, получено: {month}")
    return (year - int(min_year)) * 12 + (month - 1)


def make_default_temporal_folds() -> list[dict]:
    """
    Default temporal folds for March 2025 RTO forecasting.

    selection_score =
        0.50 * MAPE_march_2024
      + 0.15 * MAPE_january_2025
      + 0.35 * MAPE_february_2025
    """
    return [
        {
            "name": "march_2024",
            "valid_year": 2024,
            "valid_month": 3,
            "weight": 0.50,
        },
        {
            "name": "january_2025",
            "valid_year": 2025,
            "valid_month": 1,
            "weight": 0.15,
        },
        {
            "name": "february_2025",
            "valid_year": 2025,
            "valid_month": 2,
            "weight": 0.35,
        },
    ]


def _normalize_folds(folds: Optional[Sequence[dict]]) -> list[dict]:
    """
    Validate and normalize fold config.

    Compatible with both schemas used in the notebook/project:
    - old schema: {"name", "valid_year", "valid_month", "weight"}
    - new schema: {"fold_name", "valid_year", "valid_month", "weight"}

    Output always contains both "name" and "fold_name" to avoid helper conflicts.
    Weights are not normalized here; the main selector normalizes them explicitly
    when normalize_fold_weights=True.
    """
    if folds is None:
        folds = make_default_temporal_folds()

    normalized = []
    for i, fold in enumerate(folds):
        fold = dict(fold)

        required = {"valid_year", "valid_month"}
        missing = required - set(fold.keys())
        if missing:
            raise ValueError(f"В fold #{i} отсутствуют ключи: {sorted(missing)}")

        name = fold.get("name", None)
        fold_name = fold.get("fold_name", None)

        if name is None and fold_name is None:
            name = f"{int(fold['valid_year'])}_{int(fold['valid_month']):02d}"
            fold_name = name
        elif name is None:
            name = str(fold_name)
        elif fold_name is None:
            fold_name = str(name)

        normalized.append(
            {
                "name": str(name),
                "fold_name": str(fold_name),
                "valid_year": int(fold["valid_year"]),
                "valid_month": int(fold["valid_month"]),
                "weight": float(fold.get("weight", 1.0)),
            }
        )

    total_weight = sum(f["weight"] for f in normalized)
    if total_weight <= 0:
        raise ValueError("Сумма весов folds должна быть положительной.")

    return normalized


def compute_weighted_score(fold_results: pd.DataFrame) -> dict:
    """
    Compute weighted MAPE and competition score from fold_results.

    Expected columns:
        - fold_name
        - weight
        - mape
    """
    if fold_results.empty:
        raise ValueError("fold_results пустой.")

    weights = fold_results["weight"].astype(float).to_numpy()
    mapes = fold_results["mape"].astype(float).to_numpy()

    if np.any(pd.isna(mapes)):
        weighted_mape = np.nan
    else:
        weighted_mape = float(np.sum(weights * mapes) / np.sum(weights))

    return {
        "weighted_mape": weighted_mape,
        "competition_score": competition_score_from_mape(weighted_mape),
    }


def _fold_mape_dict(results: dict) -> dict[str, float]:
    """Map fold_name -> mape."""
    fr = results["fold_results"]
    return dict(zip(fr["fold_name"], fr["mape"]))


def _extract_named_fold_values(results: dict, prefix: str = "mape") -> dict:
    """
    Flatten fold results into columns:
        mape_march_2024, mape_january_2025, ...
    """
    out = {}
    for _, row in results["fold_results"].iterrows():
        out[f"{prefix}_{row['fold_name']}"] = float(row["mape"])
    return out


def _extract_delta_values(
    previous_results: dict,
    new_results: dict,
    prefix: str = "delta",
) -> dict:
    """
    Fold-level delta:
        previous_mape - new_mape

    Positive delta means improvement.
    Negative delta means degradation.
    """
    prev = _fold_mape_dict(previous_results)
    new = _fold_mape_dict(new_results)

    out = {}
    for fold_name in prev:
        out[f"{prefix}_{fold_name}"] = float(prev[fold_name] - new[fold_name])
    return out


def _count_improved_folds(
    previous_results: dict,
    new_results: dict,
    *,
    min_fold_improvement: float = 0.0,
) -> int:
    """Count folds where MAPE improved by at least min_fold_improvement."""
    prev = _fold_mape_dict(previous_results)
    new = _fold_mape_dict(new_results)

    n = 0
    for fold_name in prev:
        if prev[fold_name] - new[fold_name] > min_fold_improvement:
            n += 1
    return n


def _max_fold_degradation(
    previous_results: dict,
    new_results: dict,
) -> float:
    """
    Max fold degradation in MAPE percentage points.

    Positive value means some fold became worse.
    If all folds improved, returns <= 0.
    """
    prev = _fold_mape_dict(previous_results)
    new = _fold_mape_dict(new_results)

    degradations = []
    for fold_name in prev:
        degradations.append(float(new[fold_name] - prev[fold_name]))

    return max(degradations)


# ============================================================
# Basic temporal split for already prepared feature DataFrame
# ============================================================

def get_train_valid_by_month(
    df: pd.DataFrame,
    *,
    valid_year: int,
    valid_month: int,
    target_col: str = "pto",
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    min_year: int | None = None,
    drop_target_na_from_train: bool = True,
    require_known_valid_target: bool = True,
    copy: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Time-based train/valid split for an already feature-engineered DataFrame.

    train:
        all rows strictly before valid month.

    valid:
        rows from exactly valid_year / valid_month.

    This helper does not generate features.
    It should be used only if df already contains leak-free features.
    """
    work = df.copy() if copy else df

    required = {target_col, id_col}
    if time_col not in work.columns:
        required.update({year_col, month_col})
    missing = required - set(work.columns)
    if missing:
        raise ValueError(f"В df отсутствуют обязательные колонки: {sorted(missing)}")

    if time_col not in work.columns:
        if min_year is None:
            min_year = int(work[year_col].min())
        work[year_col] = pd.to_numeric(work[year_col], errors="raise").astype(int)
        work[month_col] = pd.to_numeric(work[month_col], errors="raise").astype(int)
        work[time_col] = (
            (work[year_col] - int(min_year)) * 12 + (work[month_col] - 1)
        ).astype(int)
    else:
        work[time_col] = pd.to_numeric(work[time_col], errors="raise").astype(int)
        if min_year is None and {year_col, month_col}.issubset(work.columns):
            inferred = work[year_col] - ((work[time_col] - (work[month_col] - 1)) // 12)
            unique_min_year = sorted(pd.Series(inferred).dropna().unique())
            if len(unique_min_year) == 1:
                min_year = int(unique_min_year[0])
            else:
                raise ValueError(
                    "Не удалось однозначно восстановить min_year. "
                    "Передай min_year явно."
                )

    valid_idx = _month_index(valid_year, valid_month, min_year)

    train_mask = work[time_col] < valid_idx
    if drop_target_na_from_train:
        train_mask = train_mask & work[target_col].notna()

    valid_mask = work[time_col] == valid_idx

    train_df = work.loc[train_mask].copy()
    valid_df = work.loc[valid_mask].copy()

    if train_df.empty:
        raise ValueError(
            f"train пустой для valid_year={valid_year}, valid_month={valid_month}."
        )

    if valid_df.empty:
        available = sorted(work[time_col].dropna().unique())
        raise ValueError(
            f"valid пустой для {valid_year}-{valid_month:02d}. "
            f"Доступные time_idx: {available[:5]} ... {available[-5:]}"
        )

    if require_known_valid_target and valid_df[target_col].isna().any():
        n_nan = int(valid_df[target_col].isna().sum())
        raise ValueError(
            f"В valid есть NaN в '{target_col}': {n_nan}. "
            "Для отбора признаков validation target должен быть известен."
        )

    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True)


# ============================================================
# CatBoost default params and CPU/GPU handling
# ============================================================

def _default_catboost_params(
    *,
    random_seed: int = 42,
    thread_count: int | None = -1,
    task_type: str = "CPU",
    devices: str | None = None,
    iterations: int = 300,
    learning_rate: float = 0.05,
    depth: int = 5,
    loss_function: str = "MAE",
    eval_metric: str = "MAE",
) -> dict:
    """
    Default small proxy CatBoost config.

    task_type:
        "CPU" or "GPU"

    devices:
        Used only for GPU, e.g. "0", "0:1".
    """
    task_type = str(task_type).upper()

    if task_type not in {"CPU", "GPU"}:
        raise ValueError("task_type должен быть 'CPU' или 'GPU'.")

    params = {
        "iterations": int(iterations),
        "learning_rate": float(learning_rate),
        "depth": int(depth),
        "loss_function": loss_function,
        "eval_metric": eval_metric,
        "random_seed": int(random_seed),
        "verbose": False,
        "allow_writing_files": False,
        "task_type": task_type,
    }

    if task_type == "CPU":
        if thread_count is not None:
            params["thread_count"] = thread_count

    if task_type == "GPU":
        if devices is not None:
            params["devices"] = devices

    return params


def _normalize_catboost_params(
    catboost_params: dict | None,
    *,
    random_seed: int = 42,
    thread_count: int | None = -1,
    task_type: str = "CPU",
    devices: str | None = None,
) -> dict:
    """
    Normalize CatBoost params and make CPU/GPU handling explicit.
    """
    if catboost_params is None:
        return _default_catboost_params(
            random_seed=random_seed,
            thread_count=thread_count,
            task_type=task_type,
            devices=devices,
        )

    params = dict(catboost_params)
    params.setdefault("random_seed", random_seed)
    params.setdefault("allow_writing_files", False)
    params.setdefault("verbose", False)

    normalized_task_type = str(params.get("task_type", task_type)).upper()
    if normalized_task_type not in {"CPU", "GPU"}:
        raise ValueError("task_type должен быть 'CPU' или 'GPU'.")

    params["task_type"] = normalized_task_type

    if normalized_task_type == "CPU":
        if thread_count is not None:
            params.setdefault("thread_count", thread_count)
        params.pop("devices", None)

    if normalized_task_type == "GPU":
        if devices is not None:
            params.setdefault("devices", devices)
        params.pop("thread_count", None)

    return params


# ============================================================
# Fold feature cache
# ============================================================

def _build_fold_data_with_feature_builder(
    df: pd.DataFrame,
    fold: dict,
    *,
    feature_builder_func: Callable,
    feature_builder_kwargs: dict | None,
    id_col: str,
    year_col: str,
    month_col: str,
    time_col: str,
    target_col: str,
    min_year: int,
) -> dict:
    """
    Build fold train/valid features using existing project pipeline.

    Expected compatible output:
        {
            "train_features": pd.DataFrame,
            "val_features": pd.DataFrame,
            "feature_cols": list, optional,
            "categorical_cols": list, optional
        }
    """
    feature_builder_kwargs = {} if feature_builder_kwargs is None else dict(feature_builder_kwargs)

    reserved = {
        "df", "split_mode", "id_col", "year_col", "month_col", "time_col",
        "target_col", "min_year", "val_year", "val_month", "verbose"
    }
    safe_kwargs = {k: v for k, v in feature_builder_kwargs.items() if k not in reserved}

    built = feature_builder_func(
        df,
        split_mode="train_val",
        id_col=id_col,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        target_col=target_col,
        min_year=min_year,
        val_year=int(fold["valid_year"]),
        val_month=int(fold["valid_month"]),
        expected_test_size=None,
        verbose=False,
        **safe_kwargs,
    )

    if not isinstance(built, dict):
        raise TypeError("feature_builder_func должен возвращать dict.")

    train_features = built.get("train_features")
    valid_features = built.get("val_features")

    if train_features is None or valid_features is None:
        raise ValueError(
            "feature_builder_func должен вернуть ключи "
            "'train_features' и 'val_features'."
        )

    fold_cat_features = (
        built.get("categorical_cols")
        or built.get("cat_features")
        or built.get("categorical_features")
        or []
    )

    fold_feature_cols = built.get("feature_cols") or []

    return {
        "fold_name": fold["name"],
        "valid_year": fold["valid_year"],
        "valid_month": fold["valid_month"],
        "weight": fold["weight"],
        "train_df": train_features.reset_index(drop=True),
        "valid_df": valid_features.reset_index(drop=True),
        "fold_categorical_features": _unique_keep_order(fold_cat_features),
        "fold_feature_cols": _unique_keep_order(fold_feature_cols),
    }


def _build_fold_data_from_ready_features(
    df: pd.DataFrame,
    fold: dict,
    *,
    id_col: str,
    year_col: str,
    month_col: str,
    time_col: str,
    target_col: str,
    min_year: int,
) -> dict:
    """Build fold train/valid from an already prepared feature DataFrame."""
    train_df, valid_df = get_train_valid_by_month(
        df,
        valid_year=int(fold["valid_year"]),
        valid_month=int(fold["valid_month"]),
        target_col=target_col,
        id_col=id_col,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        min_year=min_year,
    )

    return {
        "fold_name": fold["name"],
        "valid_year": fold["valid_year"],
        "valid_month": fold["valid_month"],
        "weight": fold["weight"],
        "train_df": train_df,
        "valid_df": valid_df,
        "fold_categorical_features": [],
        "fold_feature_cols": [],
    }


def build_fold_cache(
    df: pd.DataFrame,
    *,
    folds: Sequence[dict],
    feature_builder_func: Callable | None = None,
    feature_builder_kwargs: dict | None = None,
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    min_year: int = 2023,
    defragment: bool = True,
    verbose: bool = True,
) -> list[dict]:
    """
    Build train/valid data for each fold once.

    This is important for speed:
    forward selection reuses these fold datasets and only changes feature subsets.
    """
    fold_cache = []

    for fold in folds:
        if verbose:
            print(
                f"[fold-build] {fold['name']}: "
                f"valid={fold['valid_year']}-{fold['valid_month']:02d}, "
                f"weight={fold['weight']}"
            )

        if feature_builder_func is None:
            fold_data = _build_fold_data_from_ready_features(
                df,
                fold,
                id_col=id_col,
                year_col=year_col,
                month_col=month_col,
                time_col=time_col,
                target_col=target_col,
                min_year=min_year,
            )
        else:
            fold_data = _build_fold_data_with_feature_builder(
                df,
                fold,
                feature_builder_func=feature_builder_func,
                feature_builder_kwargs=feature_builder_kwargs,
                id_col=id_col,
                year_col=year_col,
                month_col=month_col,
                time_col=time_col,
                target_col=target_col,
                min_year=min_year,
            )

        if defragment:
            fold_data["train_df"] = fold_data["train_df"].copy()
            fold_data["valid_df"] = fold_data["valid_df"].copy()

        if verbose:
            print(
                f"    train={fold_data['train_df'].shape}, "
                f"valid={fold_data['valid_df'].shape}"
            )

        fold_cache.append(fold_data)

    return fold_cache


def _feature_exists_in_any_fold(fold_cache: Sequence[dict], feature: str) -> bool:
    """Check whether feature exists in train or valid of any fold."""
    for fd in fold_cache:
        if feature in fd["train_df"].columns or feature in fd["valid_df"].columns:
            return True
    return False


def _drop_features_missing_everywhere(
    features: Sequence[str],
    fold_cache: Sequence[dict],
    *,
    feature_role: str,
    strict: bool,
) -> list[str]:
    """
    Remove or reject features absent in all folds.
    """
    features = _unique_keep_order(features)
    missing_everywhere = [f for f in features if not _feature_exists_in_any_fold(fold_cache, f)]

    if missing_everywhere:
        msg = (
            f"{feature_role}: признаки отсутствуют во всех folds и не могут быть использованы: "
            f"{missing_everywhere[:30]}"
        )
        if len(missing_everywhere) > 30:
            msg += f" ... всего {len(missing_everywhere)}"

        if strict:
            raise ValueError(msg)
        warnings.warn(msg)

    return [f for f in features if f not in set(missing_everywhere)]


# ============================================================
# Categorical dtype inference
# ============================================================

def _is_categorical_like_dtype(dtype) -> bool:
    """
    Pandas-safe categorical/object/string dtype check.
    Avoids deprecated pd.api.types.is_categorical_dtype.
    """
    return (
        pd.api.types.is_object_dtype(dtype)
        or isinstance(dtype, pd.CategoricalDtype)
        or pd.api.types.is_string_dtype(dtype)
    )


def _infer_object_cat_features(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    features: Sequence[str],
) -> list[str]:
    """
    Infer categorical features by dtype.
    Numeric columns are NOT treated as categorical automatically.
    """
    inferred = []

    for col in features:
        if col not in train_df.columns or col not in valid_df.columns:
            continue

        train_dtype = train_df[col].dtype
        valid_dtype = valid_df[col].dtype

        if _is_categorical_like_dtype(train_dtype) or _is_categorical_like_dtype(valid_dtype):
            inferred.append(col)

    return inferred


# ============================================================
# Target transformation
# ============================================================


_LOGDIFF_TARGET_MODES = {"growth_log", "log_diff", "log_diff_lag1"}


def _normalize_target_mode_name(target_mode: str) -> str:
    """Normalize aliases for supported model target modes."""
    target_mode = str(target_mode).lower()
    if target_mode in {"growth_log", "log_diff", "log_diff_lag1"}:
        return "growth_log"
    return target_mode


def _is_logdiff_target_mode(target_mode: str) -> bool:
    """True if model target is log(pto_t) - log(anchor_t)."""
    return _normalize_target_mode_name(target_mode) == "growth_log"


def _resolve_target_anchor_col(
    *,
    target_col: str = "pto",
    target_anchor_col: str | None = None,
) -> str:
    """
    Anchor column for log-diff target.

    Default:
        target_anchor_col = f"{target_col}_lag_1"

    For RTO:
        growth_log target = log(pto_t) - log(pto_lag_1)
        inverse prediction = pto_lag_1 * exp(pred_growth_log)
    """
    return str(target_anchor_col or f"{target_col}_lag_1")


def _check_supported_target_mode(target_mode: str) -> str:
    """Validate and return normalized target_mode."""
    target_mode = _normalize_target_mode_name(target_mode)
    allowed = {"original", "log_level", "growth_log"}
    if target_mode not in allowed:
        raise ValueError(
            "target_mode должен быть 'original', 'log_level' или 'growth_log' "
            "(алиасы: 'log_diff', 'log_diff_lag1'). "
            f"Получено: {target_mode}"
        )
    return target_mode


def _make_model_target(
    data,
    *,
    target_col: str = "pto",
    target_mode: str = "original",
    target_anchor_col: str | None = None,
    log_eps: float = 1e-9,
) -> np.ndarray:
    """
    Transform raw target into model-space target.

    target_mode:
        - "original": model learns pto directly;
        - "log_level": model learns log(pto);
        - "growth_log": model learns log(pto_t) - log(anchor_t),
          where anchor_t is target_anchor_col, by default pto_lag_1.

    Important:
        For "growth_log" this function requires a DataFrame, not a raw array,
        because the transformation must use a leak-free lag/anchor column.
    """
    target_mode = _check_supported_target_mode(target_mode)

    if isinstance(data, pd.DataFrame):
        if target_col not in data.columns:
            raise ValueError(f"В data нет target_col='{target_col}'.")
        y_raw = data[target_col].astype(float).to_numpy()
        df = data
    else:
        y_raw = np.asarray(data, dtype=float)
        df = None

    if target_mode == "original":
        return y_raw

    if target_mode == "log_level":
        return np.log(np.maximum(y_raw, log_eps))

    if target_mode == "growth_log":
        if df is None:
            raise ValueError(
                "Для target_mode='growth_log' нужно передавать DataFrame, "
                "потому что требуется leak-free anchor column."
            )

        anchor_col = _resolve_target_anchor_col(
            target_col=target_col,
            target_anchor_col=target_anchor_col,
        )
        if anchor_col not in df.columns:
            raise ValueError(
                f"Для target_mode='growth_log' нужна колонка '{anchor_col}'. "
                "Обычно это pto_lag_1, созданный feature builder'ом без лика."
            )

        anchor = df[anchor_col].astype(float).to_numpy()
        return (
            np.log(np.maximum(y_raw, log_eps))
            - np.log(np.maximum(anchor, log_eps))
        )

    raise AssertionError(f"Unexpected target_mode={target_mode}")

def _inverse_model_prediction(
    pred_model,
    *,
    target_mode: str = "original",
    inverse_anchor=None,
    valid_df: pd.DataFrame | None = None,
    target_col: str = "pto",
    target_anchor_col: str | None = None,
    log_eps: float = 1e-9,
    clip_pred_lower: float | None = 0.0,
) -> np.ndarray:
    """
    Convert model-space prediction back to raw pto-space.

    For target_mode="growth_log":
        pred_raw = anchor * exp(pred_model),
    where anchor is a leak-free known lag, usually pto_lag_1.
    """
    pred_model = np.asarray(pred_model, dtype=float)
    target_mode = _check_supported_target_mode(target_mode)

    if target_mode == "original":
        pred_raw = pred_model

    elif target_mode == "log_level":
        pred_raw = np.exp(pred_model)

    elif target_mode == "growth_log":
        if inverse_anchor is None:
            if valid_df is None:
                raise ValueError(
                    "Для inverse target_mode='growth_log' нужно передать "
                    "inverse_anchor или valid_df с anchor column."
                )

            anchor_col = _resolve_target_anchor_col(
                target_col=target_col,
                target_anchor_col=target_anchor_col,
            )
            if anchor_col not in valid_df.columns:
                raise ValueError(
                    f"Для inverse target_mode='growth_log' нужна колонка '{anchor_col}'."
                )
            inverse_anchor = valid_df[anchor_col].astype(float).to_numpy()

        inverse_anchor = np.asarray(inverse_anchor, dtype=float)
        if len(inverse_anchor) != len(pred_model):
            raise ValueError(
                "Длина inverse_anchor не совпадает с длиной pred_model: "
                f"{len(inverse_anchor)} vs {len(pred_model)}."
            )

        pred_raw = np.maximum(inverse_anchor, log_eps) * np.exp(pred_model)

    else:
        raise AssertionError(f"Unexpected target_mode={target_mode}")

    if clip_pred_lower is not None:
        pred_raw = np.maximum(pred_raw, clip_pred_lower)

    return pred_raw

def prepare_catboost_data(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    features: Sequence[str],
    categorical_features: Sequence[str] | None = None,
    *,
    target_col: str = "pto",
    target_mode: str = "original",
    target_anchor_col: str | None = None,
    drop_invalid_logdiff_rows: bool = True,
    allow_missing_features: bool = False,
    log_eps: float = 1e-9,
) -> tuple[
    pd.DataFrame | None,
    np.ndarray,
    pd.DataFrame | None,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    list[str],
    np.ndarray | None,
    dict,
]:
    """
    Prepare train/valid data for CatBoost.

    Returns
    -------
    X_train, y_train_model, X_valid, y_valid_model,
    y_train_raw, y_valid_raw, cat_features, inverse_anchor_valid, target_meta

    Notes
    -----
    - y_train_model is transformed according to target_mode.
    - y_valid_raw is kept for MAPE on original pto scale.
    - For target_mode="growth_log" target is:
          log(pto_t) - log(pto_lag_1)
      and predictions are restored through:
          pto_pred = pto_lag_1 * exp(pred).
    - For growth_log, rows without a valid lag anchor are dropped by default.
      This is not leakage: the anchor is a past known target feature.
    - categorical columns are converted to strings.
    - empty features are allowed and handled by a constant baseline.
    """
    target_mode = _check_supported_target_mode(target_mode)
    features = _unique_keep_order(features)
    categorical_features = _unique_keep_order(categorical_features)

    missing_train = [c for c in features if c not in train_df.columns]
    missing_valid = [c for c in features if c not in valid_df.columns]

    if missing_train or missing_valid:
        missing = sorted(set(missing_train + missing_valid))

        if not allow_missing_features:
            raise ValueError(f"Некоторые features отсутствуют в fold data: {missing}")

        train_df = train_df.copy()
        valid_df = valid_df.copy()

        for c in missing:
            if c not in train_df.columns:
                train_df[c] = np.nan
            if c not in valid_df.columns:
                valid_df[c] = np.nan

    if target_col not in train_df.columns or target_col not in valid_df.columns:
        raise ValueError(f"target_col='{target_col}' должен быть в train_df и valid_df.")

    train_target_na = int(train_df[target_col].isna().sum())
    valid_target_na = int(valid_df[target_col].isna().sum())

    if train_target_na > 0:
        raise ValueError(f"В train target есть NaN: {train_target_na}")
    if valid_target_na > 0:
        raise ValueError(f"В valid target есть NaN: {valid_target_na}")

    target_meta = {
        "target_mode": target_mode,
        "target_anchor_col": None,
        "n_train_before_target_filter": int(len(train_df)),
        "n_valid_before_target_filter": int(len(valid_df)),
        "n_train_dropped_invalid_logdiff": 0,
        "n_valid_dropped_invalid_logdiff": 0,
    }

    inverse_anchor_valid = None

    if target_mode == "growth_log":
        anchor_col = _resolve_target_anchor_col(
            target_col=target_col,
            target_anchor_col=target_anchor_col,
        )
        target_meta["target_anchor_col"] = anchor_col

        if anchor_col not in train_df.columns or anchor_col not in valid_df.columns:
            raise ValueError(
                f"Для target_mode='growth_log' нужна колонка '{anchor_col}' "
                "в train_df и valid_df. Проверь target_lags=(1, ...) "
                "и что feature builder не удаляет anchor."
            )

        train_anchor = pd.to_numeric(train_df[anchor_col], errors="coerce")
        valid_anchor = pd.to_numeric(valid_df[anchor_col], errors="coerce")
        train_target = pd.to_numeric(train_df[target_col], errors="coerce")
        valid_target = pd.to_numeric(valid_df[target_col], errors="coerce")

        train_ok = (
            train_target.notna()
            & train_anchor.notna()
            & np.isfinite(train_target)
            & np.isfinite(train_anchor)
            & (train_target > log_eps)
            & (train_anchor > log_eps)
        )
        valid_ok = (
            valid_target.notna()
            & valid_anchor.notna()
            & np.isfinite(valid_target)
            & np.isfinite(valid_anchor)
            & (valid_target > log_eps)
            & (valid_anchor > log_eps)
        )

        n_train_bad = int((~train_ok).sum())
        n_valid_bad = int((~valid_ok).sum())

        target_meta["n_train_dropped_invalid_logdiff"] = n_train_bad
        target_meta["n_valid_dropped_invalid_logdiff"] = n_valid_bad

        if n_train_bad or n_valid_bad:
            msg = (
                "Для target_mode='growth_log' найдены строки без валидного "
                f"target/anchor: train_bad={n_train_bad}, valid_bad={n_valid_bad}, "
                f"anchor_col='{anchor_col}'."
            )
            if not drop_invalid_logdiff_rows:
                raise ValueError(msg)
            warnings.warn(msg + " Эти строки будут исключены из оценки данного fold.")

            train_df = train_df.loc[train_ok].copy()
            valid_df = valid_df.loc[valid_ok].copy()

        if train_df.empty:
            raise ValueError(
                "После фильтрации invalid growth_log rows train_df стал пустым."
            )
        if valid_df.empty:
            raise ValueError(
                "После фильтрации invalid growth_log rows valid_df стал пустым."
            )

        inverse_anchor_valid = valid_df[anchor_col].astype(float).to_numpy()

    y_train_raw = train_df[target_col].astype(float).to_numpy()
    y_valid_raw = valid_df[target_col].astype(float).to_numpy()

    if target_mode == "growth_log":
        y_train_model = _make_model_target(
            train_df,
            target_col=target_col,
            target_mode=target_mode,
            target_anchor_col=target_anchor_col,
            log_eps=log_eps,
        )
        y_valid_model = _make_model_target(
            valid_df,
            target_col=target_col,
            target_mode=target_mode,
            target_anchor_col=target_anchor_col,
            log_eps=log_eps,
        )
    else:
        y_train_model = _make_model_target(
            y_train_raw,
            target_col=target_col,
            target_mode=target_mode,
            log_eps=log_eps,
        )
        y_valid_model = _make_model_target(
            y_valid_raw,
            target_col=target_col,
            target_mode=target_mode,
            log_eps=log_eps,
        )

    if not features:
        return (
            None,
            y_train_model,
            None,
            y_valid_model,
            y_train_raw,
            y_valid_raw,
            [],
            inverse_anchor_valid,
            target_meta,
        )

    X_train = train_df.loc[:, features].copy()
    X_valid = valid_df.loc[:, features].copy()

    inferred_cat = _infer_object_cat_features(X_train, X_valid, features)
    cat_features = _unique_keep_order(
        [c for c in categorical_features if c in features] + inferred_cat
    )

    for col in cat_features:
        X_train[col] = (
            X_train[col]
            .astype("object")
            .where(X_train[col].notna(), "missing")
            .astype(str)
        )
        X_valid[col] = (
            X_valid[col]
            .astype("object")
            .where(X_valid[col].notna(), "missing")
            .astype(str)
        )

    return (
        X_train,
        y_train_model,
        X_valid,
        y_valid_model,
        y_train_raw,
        y_valid_raw,
        cat_features,
        inverse_anchor_valid,
        target_meta,
    )

def _make_catboost_pool(
    X,
    y_model,
    cat_features,
):
    """Make CatBoost Pool."""
    from catboost import Pool

    return Pool(
        data=X,
        label=y_model,
        cat_features=list(cat_features),
    )


def _fit_catboost_regressor(
    X_train: pd.DataFrame,
    y_train_model,
    cat_features: Sequence[str],
    *,
    catboost_params: dict,
):
    """Fit CatBoostRegressor on model-space target."""
    try:
        from catboost import CatBoostRegressor
    except ImportError as e:
        raise ImportError(
            "CatBoost не установлен. Установи его через `pip install catboost`."
        ) from e

    train_pool = _make_catboost_pool(
        X_train,
        y_train_model,
        cat_features,
    )

    model = CatBoostRegressor(**catboost_params)
    model.fit(train_pool, verbose=False)

    return model, train_pool


# ============================================================
# Empty-feature baseline
# ============================================================

def _predict_constant_baseline(
    y_train_model,
    n_valid: int,
    *,
    constant_strategy: str = "median",
    target_mode: str = "original",
    inverse_anchor_valid=None,
    target_col: str = "pto",
    target_anchor_col: str | None = None,
    log_eps: float = 1e-9,
    clip_pred_lower: float | None = 0.0,
):
    """
    Constant prediction for empty feature set.

    This makes true forward selection from empty set possible.
    For growth_log, the constant is a median/mean log-growth and is restored
    via each validation row's leak-free anchor.
    """
    constant_strategy = str(constant_strategy).lower()

    if constant_strategy == "median":
        const_model = float(np.nanmedian(y_train_model))
    elif constant_strategy == "mean":
        const_model = float(np.nanmean(y_train_model))
    else:
        raise ValueError("constant_strategy должен быть 'median' или 'mean'.")

    pred_model = np.full(n_valid, const_model, dtype=float)

    return _inverse_model_prediction(
        pred_model,
        target_mode=target_mode,
        inverse_anchor=inverse_anchor_valid,
        target_col=target_col,
        target_anchor_col=target_anchor_col,
        log_eps=log_eps,
        clip_pred_lower=clip_pred_lower,
    )

def evaluate_feature_set(
    fold_cache: Sequence[dict],
    features: Sequence[str],
    *,
    categorical_features: Sequence[str] | None = None,
    target_col: str = "pto",
    target_mode: str = "original",
    target_anchor_col: str | None = None,
    drop_invalid_logdiff_rows: bool = True,
    catboost_params: dict | None = None,
    allow_missing_features: bool = False,
    clip_pred_lower: float | None = 0.0,
    log_eps: float = 1e-9,
    constant_strategy: str = "median",
    return_models: bool = False,
    verbose: bool = False,
) -> dict:
    """
    Evaluate one feature set by weighted temporal CV MAPE.

    For target_mode="log_level":
        model learns log(pto);
        prediction is exp(model_prediction);
        MAPE is calculated on original pto scale.

    Empty features are allowed:
        a constant baseline is used instead of CatBoost.
    """
    features = _unique_keep_order(features)
    categorical_features = _unique_keep_order(categorical_features)

    if catboost_params is None:
        catboost_params = _default_catboost_params()

    records = []
    models = {}

    for fd in fold_cache:
        fold_name = fd["fold_name"]
        train_df = fd["train_df"]
        valid_df = fd["valid_df"]

        fold_cat_features = _unique_keep_order(
            categorical_features + fd.get("fold_categorical_features", [])
        )

        (
            X_train,
            y_train_model,
            X_valid,
            y_valid_model,
            y_train_raw,
            y_valid_raw,
            cat_features,
            inverse_anchor_valid,
            target_meta,
        ) = prepare_catboost_data(
            train_df,
            valid_df,
            features,
            fold_cat_features,
            target_col=target_col,
            target_mode=target_mode,
            target_anchor_col=target_anchor_col,
            drop_invalid_logdiff_rows=drop_invalid_logdiff_rows,
            allow_missing_features=allow_missing_features,
            log_eps=log_eps,
        )

        if not features:
            pred_raw = _predict_constant_baseline(
                y_train_model,
                n_valid=len(y_valid_raw),
                constant_strategy=constant_strategy,
                target_mode=target_mode,
                inverse_anchor_valid=inverse_anchor_valid,
                target_col=target_col,
                target_anchor_col=target_anchor_col,
                log_eps=log_eps,
                clip_pred_lower=clip_pred_lower,
            )
            model = None

        else:
            model, _ = _fit_catboost_regressor(
                X_train,
                y_train_model,
                cat_features,
                catboost_params=catboost_params,
            )

            pred_model = model.predict(X_valid)

            pred_raw = _inverse_model_prediction(
                pred_model,
                target_mode=target_mode,
                inverse_anchor=inverse_anchor_valid,
                target_col=target_col,
                target_anchor_col=target_anchor_col,
                log_eps=log_eps,
                clip_pred_lower=clip_pred_lower,
            )

        mape = mape_score(
            y_valid_raw,
            pred_raw,
            clip_pred_lower=clip_pred_lower,
        )

        record = {
            "fold_name": fold_name,
            "valid_year": fd["valid_year"],
            "valid_month": fd["valid_month"],
            "weight": fd["weight"],
            "mape": mape,
            "competition_score": competition_score_from_mape(mape),
            "n_train": len(train_df),
            "n_valid": len(valid_df),
            "n_features": len(features),
            "n_cat_features": len(cat_features),
            "target_mode": target_mode,
            "target_anchor_col": target_meta.get("target_anchor_col"),
            "n_train_used_for_target": len(y_train_raw),
            "n_valid_used_for_target": len(y_valid_raw),
            "n_train_dropped_invalid_logdiff": target_meta.get("n_train_dropped_invalid_logdiff", 0),
            "n_valid_dropped_invalid_logdiff": target_meta.get("n_valid_dropped_invalid_logdiff", 0),
        }
        records.append(record)

        if return_models and model is not None:
            models[fold_name] = model

        if verbose:
            print(
                f"    [{fold_name}] "
                f"MAPE={mape:.5f}, "
                f"score={record['competition_score']:.5f}, "
                f"n_features={len(features)}"
            )

        del X_train, X_valid, y_train_model, y_valid_model, y_train_raw, y_valid_raw, model
        gc.collect()

    fold_results = pd.DataFrame(records)
    weighted = compute_weighted_score(fold_results)

    out = {
        "weighted_mape": weighted["weighted_mape"],
        "competition_score": weighted["competition_score"],
        "fold_results": fold_results,
    }

    if return_models:
        out["models"] = models

    return out


# ============================================================
# Proxy filter via LossFunctionChange
# ============================================================

def get_proxy_feature_importance(
    fold_cache: Sequence[dict],
    *,
    fixed_features: Sequence[str],
    candidate_features: Sequence[str],
    categorical_features: Sequence[str] | None = None,
    target_col: str = "pto",
    target_mode: str = "original",
    target_anchor_col: str | None = None,
    drop_invalid_logdiff_rows: bool = True,
    catboost_params: dict | None = None,
    allow_missing_features: bool = False,
    importance_type: str = "LossFunctionChange",
    importance_data: str = "valid",
    log_eps: float = 1e-9,
    strict_loss_importance: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Proxy feature filter via CatBoost LossFunctionChange.

    This is NOT split-based PredictionValuesChange.
    It estimates how CatBoost loss changes when the feature is removed.

    importance_data:
        - "valid": calculate importance on validation fold;
        - "train": calculate importance on train fold.
    """
    fixed_features = _unique_keep_order(fixed_features)
    candidate_features = _unique_keep_order(candidate_features)
    all_features = _unique_keep_order(fixed_features + candidate_features)
    categorical_features = _unique_keep_order(categorical_features)

    if not candidate_features:
        return pd.DataFrame(
            columns=[
                "feature",
                "importance_mean",
                "importance_std",
                "importance_by_fold",
                "is_fixed",
                "selected_for_forward",
            ]
        )

    if importance_type != "LossFunctionChange":
        raise ValueError(
            "В этой версии proxy importance намеренно ограничен "
            "importance_type='LossFunctionChange'. "
            "PredictionValuesChange здесь запрещён."
        )

    if catboost_params is None:
        catboost_params = _default_catboost_params()

    fold_importance_frames = []

    for fd in fold_cache:
        fold_name = fd["fold_name"]

        if verbose:
            print(
                f"[proxy-loss-importance] fold={fold_name}, "
                f"n_features={len(all_features)}, "
                f"target_mode={target_mode}"
            )

        fold_cat_features = _unique_keep_order(
            categorical_features + fd.get("fold_categorical_features", [])
        )

        (
            X_train,
            y_train_model,
            X_valid,
            y_valid_model,
            y_train_raw,
            y_valid_raw,
            cat_features,
            inverse_anchor_valid,
            target_meta,
        ) = prepare_catboost_data(
            fd["train_df"],
            fd["valid_df"],
            all_features,
            fold_cat_features,
            target_col=target_col,
            target_mode=target_mode,
            target_anchor_col=target_anchor_col,
            drop_invalid_logdiff_rows=drop_invalid_logdiff_rows,
            allow_missing_features=allow_missing_features,
            log_eps=log_eps,
        )

        model, train_pool = _fit_catboost_regressor(
            X_train,
            y_train_model,
            cat_features,
            catboost_params=catboost_params,
        )

        valid_pool = _make_catboost_pool(
            X_valid,
            y_valid_model,
            cat_features,
        )

        if importance_data == "valid":
            data_pool = valid_pool
        elif importance_data == "train":
            data_pool = train_pool
        else:
            raise ValueError("importance_data должен быть 'valid' или 'train'.")

        try:
            importances = model.get_feature_importance(
                data=data_pool,
                type="LossFunctionChange",
            )
        except Exception as e:
            if strict_loss_importance:
                raise RuntimeError(
                    "Не удалось посчитать CatBoost LossFunctionChange. "
                    "Fallback на PredictionValuesChange отключён специально."
                ) from e
            importances = np.full(len(all_features), np.nan)

        fold_imp = pd.DataFrame(
            {
                "feature": all_features,
                f"importance_{fold_name}": importances,
            }
        )
        fold_importance_frames.append(fold_imp)

        del (
            X_train, X_valid,
            y_train_model, y_valid_model,
            y_train_raw, y_valid_raw,
            model, train_pool, valid_pool, data_pool
        )
        gc.collect()

    importance_df = fold_importance_frames[0]

    for part in fold_importance_frames[1:]:
        importance_df = importance_df.merge(part, on="feature", how="outer")

    importance_cols = [c for c in importance_df.columns if c.startswith("importance_")]

    importance_df["importance_mean"] = importance_df[importance_cols].mean(axis=1)
    importance_df["importance_std"] = importance_df[importance_cols].std(axis=1)
    importance_df["importance_by_fold"] = importance_df.apply(
        lambda row: {
            c.replace("importance_", ""): row[c]
            for c in importance_cols
            if pd.notna(row[c])
        },
        axis=1,
    )

    fixed_set = set(fixed_features)
    importance_df["is_fixed"] = importance_df["feature"].isin(fixed_set)
    importance_df["selected_for_forward"] = False

    importance_df = importance_df.sort_values(
        ["is_fixed", "importance_mean"],
        ascending=[False, False],
    ).reset_index(drop=True)

    return importance_df


# ============================================================
# Forward selection
# ============================================================



# =============================================================================
# Adaptive v4 selector body
# =============================================================================

# =============================================================================
# Default folds
# =============================================================================


def make_march_2025_feature_selection_folds(normalize: bool = True) -> list[dict]:
    """
    Default fold scheme for final March-2025 forecasting.

    Raw priorities requested by user:
        march_2024    = 0.6
        february_2025 = 0.2
        january_2025  = 0.2
        april_2024    = 0.3
        may_2024      = 0.3

    Raw sum is 1.6, so normalized weights are:
        march_2024    = 0.3750
        february_2025 = 0.1250
        january_2025  = 0.1250
        april_2024    = 0.1875
        may_2024      = 0.1875
    """
    folds = [
        {"fold_name": "march_2024", "valid_year": 2024, "valid_month": 3, "weight": 0.6},
        {"fold_name": "february_2025", "valid_year": 2025, "valid_month": 2, "weight": 0.2},
        {"fold_name": "january_2025", "valid_year": 2025, "valid_month": 1, "weight": 0.2},
        {"fold_name": "april_2024", "valid_year": 2024, "valid_month": 4, "weight": 0.3},
        {"fold_name": "may_2024", "valid_year": 2024, "valid_month": 5, "weight": 0.3},
    ]
    return _fs_v2_normalize_fold_weights(folds) if normalize else folds


# =============================================================================
# Small internal helpers
# =============================================================================


def _fs_v2_unique_keep_order(values):
    if values is None:
        return []
    seen = set()
    out = []
    for x in values:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out


def _fs_v2_normalize_fold_weights(folds: Sequence[dict]) -> list[dict]:
    folds_out = [dict(f) for f in folds]
    weights = [float(f.get("weight", 1.0)) for f in folds_out]
    total_weight = float(np.sum(weights))

    if not np.isfinite(total_weight) or total_weight <= 0:
        raise ValueError("Сумма весов фолдов должна быть положительной и конечной.")

    for f, w in zip(folds_out, weights):
        f["weight"] = float(w / total_weight)

    return folds_out


def _fs_v2_normalize_fold_cache_weights(fold_cache: Sequence[dict]) -> list[dict]:
    fold_cache_out = [dict(fd) for fd in fold_cache]
    weights = [float(fd.get("weight", 1.0)) for fd in fold_cache_out]
    total_weight = float(np.sum(weights))

    if not np.isfinite(total_weight) or total_weight <= 0:
        raise ValueError("Сумма весов fold_cache должна быть положительной и конечной.")

    for fd, w in zip(fold_cache_out, weights):
        fd["weight"] = float(w / total_weight)

    return fold_cache_out


def _fs_v2_prepare_catboost_params(
    catboost_params: dict,
    *,
    force_silent_catboost: bool = True,
    catboost_metric_period: int | None = 1000,
) -> dict:
    """Make CatBoost quiet and suppress the usual GPU/MAE metric-period warning."""
    params = dict(catboost_params)

    if force_silent_catboost:
        # CatBoost may fail if logging aliases are set together.
        for k in ("verbose", "verbose_eval", "silent"):
            params.pop(k, None)

        params["logging_level"] = "Silent"
        params["allow_writing_files"] = False

        # Suppresses:
        # "Default metric period is 5 because MAE is/are not implemented for GPU".
        if catboost_metric_period is not None:
            params["metric_period"] = int(catboost_metric_period)

    return params


@contextlib.contextmanager
def _fs_v2_suppress_catboost_output(enabled: bool = True):
    """Redirect stdout/stderr around CatBoost calls if needed."""
    if not enabled:
        yield
        return

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*Default metric period.*")
        warnings.filterwarnings("ignore", message=".*MAE.*not implemented for GPU.*")
        warnings.filterwarnings("ignore", category=UserWarning)

        with open(os.devnull, "w") as devnull:
            with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
                yield




# =============================================================================
# Categorical feature policy helpers
# =============================================================================


def make_default_retail_categorical_source_cols(
    *,
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
) -> list[str]:
    """
    Source columns that are usually useful to expose to CatBoost as categorical.

    Design:
    - object/string columns are used directly as CatBoost categorical features;
    - numeric/discrete columns are duplicated as '<col>_cat' by default, so the
      original numeric representation can stay available as a numeric feature.
    """
    cols = [
        id_col,
        year_col,
        month_col,
        "month_name",
        "prev_month",
        "season",
        "holiday_month_type",
        "opening_date_category",
        "trading_area_category",
        "settlement",
        "region",
        "population_city_type",
        "lag1_rto_size_bin",
        "alcohol_license_flag",
        "marketplaces_deliveries_parcel_lockers_100m",
        "medical_institutions_and_pharmacies_300m",
        "schools_300m",
        "public_transport_stops_300m",
        "grocery_stores_500m",
        "pyaterochka_stores_500m",
        "number_of_cash_registers",
        "working_hours_per_day",
        "working_hours_per_day_lag_1",
        "working_hours_per_day_lag_3",
        "working_hours_per_day_lag_12",
    ]
    return _fs_v2_unique_keep_order(cols)




def make_x5_retail_categorical_source_cols(
    *,
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    include_current_working_hours_per_day: bool = False,
    include_working_hours_lag_categoricals: bool = True,
) -> list[str]:
    """
    Categorical source columns requested for the current X5/RTO notebook.

    Important v6 semantics:
    - these columns are made available to CatBoost as categorical candidates;
    - they are NOT automatically inserted into the baseline feature set;
    - numeric/discrete columns get '<col>_cat' copies while preserving numeric originals;
    - year is kept by the default builder kwargs, so year/year_cat can be scored;
    - current working_hours_per_day is future-unknown by default, so it is not
      requested unless include_current_working_hours_per_day=True;
    - safe lagged working-hours columns are requested by default if the builder creates them.
    """
    cols = [
        id_col,
        "opening_date_category",
        "trading_area_category",
        "settlement",
        "region",
        "alcohol_license_flag",
        year_col,
        month_col,
        "marketplaces_deliveries_parcel_lockers_100m",
        "medical_institutions_and_pharmacies_300m",
        "schools_300m",
        "public_transport_stops_300m",
        "grocery_stores_500m",
        "pyaterochka_stores_500m",
        "number_of_cash_registers",
    ]

    if include_current_working_hours_per_day:
        cols.append("working_hours_per_day")

    if include_working_hours_lag_categoricals:
        cols.extend([
            "working_hours_per_day_lag_1",
            "working_hours_per_day_lag_3",
            "working_hours_per_day_lag_12",
        ])

    return _fs_v2_unique_keep_order(cols)


def make_x5_string_categorical_features() -> list[str]:
    """String/object categorical columns from the original renamed dataset."""
    return [
        "opening_date_category",
        "trading_area_category",
        "settlement",
        "region",
    ]

def _fs_v3_all_fold_dfs(fold_cache: Sequence[dict]) -> list[pd.DataFrame]:
    dfs = []
    for fd in fold_cache:
        for key in ("train_df", "valid_df"):
            obj = fd.get(key)
            if isinstance(obj, pd.DataFrame):
                dfs.append(obj)
    return dfs


def _fs_v3_column_exists_anywhere(fold_cache: Sequence[dict], col: str) -> bool:
    return any(col in df_part.columns for df_part in _fs_v3_all_fold_dfs(fold_cache))


def _fs_v3_max_nunique(fold_cache: Sequence[dict], col: str, sample_limit: int | None = None) -> int:
    max_unique = 0
    for df_part in _fs_v3_all_fold_dfs(fold_cache):
        if col not in df_part.columns:
            continue
        s = df_part[col]
        if sample_limit is not None and len(s) > sample_limit:
            s = s.sample(sample_limit, random_state=17)
        try:
            n = int(s.nunique(dropna=True))
        except Exception:
            n = 0
        max_unique = max(max_unique, n)
    return max_unique


def _fs_v3_make_cat_copy_name(col: str, suffix: str = "_cat") -> str:
    return f"{col}{suffix}"


def _fs_v3_prepare_series_as_cat(s: pd.Series, missing_token: str = "__MISSING__") -> pd.Series:
    """Convert a Series to a safe CatBoost categorical representation."""
    out = s.astype("object")
    out = out.where(pd.notna(out), missing_token)
    return out.astype(str)


def _fs_v3_apply_categorical_policy_to_fold_cache(
    fold_cache: Sequence[dict],
    *,
    explicit_categorical_features: Sequence[str] | None,
    id_col: str,
    year_col: str,
    month_col: str,
    target_col: str,
    auto_detect_object_categoricals: bool = False,
    auto_add_default_retail_categoricals: bool = False,
    numeric_categorical_source_cols: Sequence[str] | None = None,
    extra_categorical_source_cols: Sequence[str] | None = None,
    create_numeric_categorical_copies: bool = True,
    categorical_copy_suffix: str = "_cat",
    keep_original_numeric_categoricals: bool = True,
    max_auto_numeric_cat_unique: int = 40,
    include_all_low_cardinality_integer_cols: bool = False,
    missing_token: str = "__MISSING__",
    verbose: bool = True,
) -> tuple[list[dict], list[str], pd.DataFrame]:
    """
    Enrich fold_cache with safe categorical representations and return CatBoost cat_features.

    Policy:
    - string/object/category columns are passed directly to CatBoost after filling NA;
    - numeric categorical source columns are duplicated as '<col>_cat' by default;
    - original numeric columns are preserved, so the model can use both numeric
      and categorical representations;
    - new_id is included by default through make_default_retail_categorical_source_cols.
    """
    explicit = _fs_v2_unique_keep_order(explicit_categorical_features)
    default_sources = (
        make_default_retail_categorical_source_cols(
            id_col=id_col,
            year_col=year_col,
            month_col=month_col,
        )
        if auto_add_default_retail_categoricals
        else []
    )
    numeric_sources = _fs_v2_unique_keep_order(numeric_categorical_source_cols)
    extra_sources = _fs_v2_unique_keep_order(extra_categorical_source_cols)

    explicit_existing = [c for c in explicit if _fs_v3_column_exists_anywhere(fold_cache, c)]
    source_candidates = _fs_v2_unique_keep_order(default_sources + numeric_sources + extra_sources)

    if auto_detect_object_categoricals:
        for df_part in _fs_v3_all_fold_dfs(fold_cache):
            for c in df_part.columns:
                if c == target_col or str(c).startswith("__"):
                    continue
                if pd.api.types.is_object_dtype(df_part[c]) or pd.api.types.is_categorical_dtype(df_part[c]):
                    source_candidates.append(c)
        source_candidates = _fs_v2_unique_keep_order(source_candidates)

    if include_all_low_cardinality_integer_cols:
        for df_part in _fs_v3_all_fold_dfs(fold_cache):
            for c in df_part.columns:
                if c == target_col or str(c).startswith("__"):
                    continue
                s = df_part[c]
                if pd.api.types.is_integer_dtype(s) or pd.api.types.is_bool_dtype(s):
                    if _fs_v3_max_nunique(fold_cache, c) <= int(max_auto_numeric_cat_unique):
                        source_candidates.append(c)
        source_candidates = _fs_v2_unique_keep_order(source_candidates)

    cat_features_used: list[str] = []
    records: list[dict] = []
    fold_cache_out = []

    for fd in fold_cache:
        fd_new = dict(fd)
        fold_cat_features = _fs_v2_unique_keep_order(fd_new.get("fold_categorical_features", []))

        for key in ("train_df", "valid_df"):
            if key not in fd_new or not isinstance(fd_new[key], pd.DataFrame):
                continue
            df_part = fd_new[key].copy()

            for src_col in source_candidates:
                if src_col not in df_part.columns or src_col == target_col:
                    continue

                is_obj_like = (
                    pd.api.types.is_object_dtype(df_part[src_col])
                    or pd.api.types.is_categorical_dtype(df_part[src_col])
                )

                if is_obj_like:
                    cat_col = src_col
                    df_part[cat_col] = _fs_v3_prepare_series_as_cat(
                        df_part[src_col], missing_token=missing_token
                    )
                    representation = "original_object"
                else:
                    if create_numeric_categorical_copies:
                        cat_col = _fs_v3_make_cat_copy_name(src_col, categorical_copy_suffix)
                        df_part[cat_col] = _fs_v3_prepare_series_as_cat(
                            df_part[src_col], missing_token=missing_token
                        )
                        representation = "numeric_copy"
                    else:
                        cat_col = src_col
                        df_part[cat_col] = _fs_v3_prepare_series_as_cat(
                            df_part[src_col], missing_token=missing_token
                        )
                        representation = "numeric_overwritten_as_string"

                cat_features_used.append(cat_col)
                fold_cat_features.append(cat_col)
                records.append(
                    {
                        "fold_name": fd_new.get("fold_name"),
                        "part": key,
                        "source_col": src_col,
                        "cat_feature": cat_col,
                        "representation": representation,
                        "n_unique_part": int(df_part[cat_col].nunique(dropna=False)),
                    }
                )

            for cat_col in explicit_existing:
                if cat_col in df_part.columns and cat_col != target_col:
                    df_part[cat_col] = _fs_v3_prepare_series_as_cat(
                        df_part[cat_col], missing_token=missing_token
                    )
                    cat_features_used.append(cat_col)
                    fold_cat_features.append(cat_col)

            fd_new[key] = df_part

        fd_new["fold_categorical_features"] = _fs_v2_unique_keep_order(fold_cat_features)
        fold_cache_out.append(fd_new)

    cat_features_used = _fs_v2_unique_keep_order(explicit_existing + cat_features_used)
    cat_features_used = [
        c for c in cat_features_used
        if c != target_col and _fs_v3_column_exists_anywhere(fold_cache_out, c)
    ]

    meta = pd.DataFrame(records)
    if not meta.empty:
        meta = meta.drop_duplicates(
            subset=["fold_name", "part", "source_col", "cat_feature"],
            keep="last",
        )

    if verbose:
        print()
        print("[categorical-policy]")
        print(f"cat_features used: {len(cat_features_used)}")
        if cat_features_used:
            print("first cat_features:", cat_features_used[:30])
        if not meta.empty:
            summary = (
                meta[["source_col", "cat_feature", "representation"]]
                .drop_duplicates()
                .head(50)
            )
            print(summary.to_string(index=False))
        print()

    return fold_cache_out, cat_features_used, meta


def _fs_v2_importance_values_from_cell(x) -> list[float]:
    """Parse importance_by_fold cell into a clean numeric list."""
    if x is None:
        return []

    if isinstance(x, dict):
        raw_values = list(x.values())
    elif isinstance(x, (list, tuple, np.ndarray, pd.Series)):
        raw_values = list(x)
    elif isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, dict):
                raw_values = list(parsed.values())
            elif isinstance(parsed, (list, tuple)):
                raw_values = list(parsed)
            else:
                raw_values = []
        except Exception:
            raw_values = []
    else:
        raw_values = []

    values = []
    for v in raw_values:
        try:
            fv = float(v)
            if np.isfinite(fv):
                values.append(fv)
        except Exception:
            continue

    return values


def _fs_v2_add_stability_columns(
    importance_df: pd.DataFrame,
    *,
    stability_std_penalty: float = 0.50,
    positive_fold_bonus: float = 0.25,
    eps: float = 1e-12,
) -> pd.DataFrame:
    """
    Add stable ranking columns to candidate importance table.

    Adds:
        importance_min
        importance_median
        importance_std
        n_positive_folds
        n_negative_folds
        n_folds_with_importance
        positive_fold_share
        stability_score
    """
    df_imp = importance_df.copy()

    if "feature" not in df_imp.columns:
        raise ValueError("importance_df должен содержать колонку 'feature'.")

    if "importance_by_fold" in df_imp.columns:
        values_list = df_imp["importance_by_fold"].apply(_fs_v2_importance_values_from_cell)
    else:
        values_list = pd.Series([[] for _ in range(len(df_imp))], index=df_imp.index)

    computed = []
    for vals in values_list:
        arr = np.asarray(vals, dtype=float)
        arr = arr[np.isfinite(arr)]

        if arr.size > 0:
            computed.append(
                {
                    "importance_min": float(np.min(arr)),
                    "importance_median": float(np.median(arr)),
                    "importance_std_from_folds": float(np.std(arr, ddof=0)),
                    "n_positive_folds": int(np.sum(arr > eps)),
                    "n_negative_folds": int(np.sum(arr < -eps)),
                    "n_folds_with_importance": int(arr.size),
                    "positive_fold_share": float(np.mean(arr > eps)),
                }
            )
        else:
            computed.append(
                {
                    "importance_min": np.nan,
                    "importance_median": np.nan,
                    "importance_std_from_folds": np.nan,
                    "n_positive_folds": 0,
                    "n_negative_folds": 0,
                    "n_folds_with_importance": 0,
                    "positive_fold_share": np.nan,
                }
            )

    computed_df = pd.DataFrame(computed, index=df_imp.index)

    for col in computed_df.columns:
        if col not in df_imp.columns:
            df_imp[col] = computed_df[col]
        else:
            df_imp[col] = df_imp[col].combine_first(computed_df[col])

    if "importance_mean" not in df_imp.columns:
        df_imp["importance_mean"] = np.nan

    if "importance_std" not in df_imp.columns:
        df_imp["importance_std"] = df_imp["importance_std_from_folds"]
    else:
        df_imp["importance_std"] = df_imp["importance_std"].combine_first(
            df_imp["importance_std_from_folds"]
        )

    df_imp["importance_mean"] = pd.to_numeric(df_imp["importance_mean"], errors="coerce")
    df_imp["importance_std"] = pd.to_numeric(df_imp["importance_std"], errors="coerce").fillna(0.0)
    df_imp["positive_fold_share"] = pd.to_numeric(
        df_imp["positive_fold_share"], errors="coerce"
    ).fillna(0.0)

    df_imp["stability_score"] = (
        df_imp["importance_mean"].fillna(-np.inf)
        * (1.0 + float(positive_fold_bonus) * df_imp["positive_fold_share"])
        - float(stability_std_penalty) * df_imp["importance_std"].fillna(0.0)
    )

    return df_imp


def _fs_v2_get_fold_mape_map(results: dict) -> dict:
    fold_results = results.get("fold_results")
    if fold_results is None or len(fold_results) == 0:
        return {}

    fr = fold_results.copy()
    if "fold_name" not in fr.columns or "mape" not in fr.columns:
        return {}

    return {
        str(row["fold_name"]): float(row["mape"])
        for _, row in fr[["fold_name", "mape"]].iterrows()
        if pd.notna(row["fold_name"]) and pd.notna(row["mape"])
    }


def _fs_v2_fold_degradation_info(previous_results: dict, new_results: dict) -> tuple[float, dict]:
    previous_map = _fs_v2_get_fold_mape_map(previous_results)
    new_map = _fs_v2_get_fold_mape_map(new_results)

    common_names = sorted(set(previous_map).intersection(new_map))
    deltas = {
        name: float(new_map[name] - previous_map[name])
        for name in common_names
    }

    positive_deltas = [v for v in deltas.values() if np.isfinite(v) and v > 0]
    max_degradation = float(max(positive_deltas)) if positive_deltas else 0.0

    return max_degradation, deltas


def _fs_v2_check_acceptance(
    previous_results: dict,
    trial_results: dict,
    *,
    min_improvement: float,
    min_improved_folds: int,
    max_allowed_fold_degradation: float | None,
    max_allowed_degradation_by_fold: dict | None,
) -> tuple[bool, list[str], dict]:
    previous_weighted_mape = float(previous_results["weighted_mape"])
    new_weighted_mape = float(trial_results["weighted_mape"])
    improvement = previous_weighted_mape - new_weighted_mape

    n_improved_folds = _count_improved_folds(previous_results, trial_results)
    max_degradation, degradation_by_fold = _fs_v2_fold_degradation_info(
        previous_results,
        trial_results,
    )

    eligible = True
    fail_reasons = []

    if not np.isfinite(improvement) or improvement < float(min_improvement):
        eligible = False
        fail_reasons.append("min_improvement")

    if int(n_improved_folds) < int(min_improved_folds):
        eligible = False
        fail_reasons.append("min_improved_folds")

    if max_allowed_fold_degradation is not None:
        if max_degradation > float(max_allowed_fold_degradation):
            eligible = False
            fail_reasons.append("max_allowed_fold_degradation")

    if max_allowed_degradation_by_fold:
        for fold_name, allowed in max_allowed_degradation_by_fold.items():
            fold_name = str(fold_name)
            if fold_name in degradation_by_fold:
                if degradation_by_fold[fold_name] > float(allowed):
                    eligible = False
                    fail_reasons.append(f"max_degradation_by_fold:{fold_name}")

    meta = {
        "previous_weighted_mape": previous_weighted_mape,
        "new_weighted_mape": new_weighted_mape,
        "improvement": float(improvement),
        "n_improved_folds": int(n_improved_folds),
        "max_fold_degradation": float(max_degradation),
        "degradation_by_fold": degradation_by_fold,
    }

    return eligible, fail_reasons, meta


# =============================================================================
# Proxy ranking
# =============================================================================


def _fs_v2_compute_proxy_ranking(
    *,
    fold_cache: Sequence[dict],
    current_features: list[str],
    candidate_pool: list[str],
    categorical_features: Sequence[str],
    target_col: str,
    target_mode: str,
    target_anchor_col: str | None,
    drop_invalid_logdiff_rows: bool,
    catboost_params: dict,
    allow_missing_fold_features: bool,
    proxy_importance_type: str,
    proxy_importance_data: str,
    log_eps: float,
    use_proxy_filter: bool,
    precomputed_candidate_importance: pd.DataFrame | None,
    ranking_round: int,
    top_k_candidates: int | None,
    min_proxy_importance: float | None,
    min_proxy_importance_on: str,
    ranking_score_col: str,
    stability_std_penalty: float,
    positive_fold_bonus: float,
    suppress_catboost_warnings: bool,
    verbose: bool,
) -> tuple[pd.DataFrame, list[str]]:
    if not candidate_pool:
        empty = pd.DataFrame(
            columns=[
                "feature",
                "importance_mean",
                "importance_std",
                "importance_min",
                "importance_median",
                "n_positive_folds",
                "positive_fold_share",
                "stability_score",
                "is_fixed",
                "selected_for_ranked_add",
            ]
        )
        return empty, []

    if not use_proxy_filter:
        candidate_importance = pd.DataFrame(
            {
                "feature": list(current_features) + list(candidate_pool),
                "importance_mean": np.nan,
                "importance_std": np.nan,
                "importance_by_fold": [{} for _ in range(len(current_features) + len(candidate_pool))],
                "is_fixed": [True] * len(current_features) + [False] * len(candidate_pool),
            }
        )
    elif precomputed_candidate_importance is not None and ranking_round == 1:
        candidate_importance = precomputed_candidate_importance.copy()

        if "feature" not in candidate_importance.columns:
            raise ValueError("precomputed_candidate_importance должен содержать колонку 'feature'.")

        if "importance_mean" not in candidate_importance.columns:
            raise ValueError("precomputed_candidate_importance должен содержать колонку 'importance_mean'.")

        if "is_fixed" not in candidate_importance.columns:
            candidate_importance["is_fixed"] = candidate_importance["feature"].isin(set(current_features))
    else:
        with _fs_v2_suppress_catboost_output(suppress_catboost_warnings):
            candidate_importance = get_proxy_feature_importance(
                fold_cache,
                fixed_features=current_features,
                candidate_features=candidate_pool,
                categorical_features=categorical_features,
                target_col=target_col,
                target_mode=target_mode,
                target_anchor_col=target_anchor_col,
                drop_invalid_logdiff_rows=drop_invalid_logdiff_rows,
                catboost_params=catboost_params,
                allow_missing_features=allow_missing_fold_features,
                importance_type=proxy_importance_type,
                importance_data=proxy_importance_data,
                log_eps=log_eps,
                strict_loss_importance=True,
                verbose=False,
            )

    candidate_importance = _fs_v2_add_stability_columns(
        candidate_importance,
        stability_std_penalty=stability_std_penalty,
        positive_fold_bonus=positive_fold_bonus,
    )

    candidate_importance["ranking_round"] = int(ranking_round)
    candidate_importance["n_fixed_features_at_ranking"] = int(len(current_features))

    candidate_rows = candidate_importance[
        candidate_importance["feature"].isin(candidate_pool)
    ].copy()

    if "is_fixed" in candidate_rows.columns:
        candidate_rows = candidate_rows[~candidate_rows["is_fixed"].fillna(False)].copy()

    if min_proxy_importance is not None:
        if min_proxy_importance_on not in candidate_rows.columns:
            raise ValueError(
                f"min_proxy_importance_on='{min_proxy_importance_on}' отсутствует в candidate_importance."
            )

        candidate_rows[min_proxy_importance_on] = pd.to_numeric(
            candidate_rows[min_proxy_importance_on],
            errors="coerce",
        )
        candidate_rows = candidate_rows[
            candidate_rows[min_proxy_importance_on] >= float(min_proxy_importance)
        ].copy()

    if ranking_score_col not in candidate_rows.columns:
        raise ValueError(f"ranking_score_col='{ranking_score_col}' отсутствует в candidate_importance.")

    candidate_rows[ranking_score_col] = pd.to_numeric(candidate_rows[ranking_score_col], errors="coerce")
    candidate_rows["importance_mean"] = pd.to_numeric(candidate_rows["importance_mean"], errors="coerce")

    candidate_rows = candidate_rows.sort_values(
        by=[ranking_score_col, "importance_mean"],
        ascending=[False, False],
        kind="mergesort",
    )

    if top_k_candidates is not None:
        candidate_rows = candidate_rows.head(int(top_k_candidates)).copy()

    ranked_features = candidate_rows["feature"].tolist()
    ranked_set = set(ranked_features)
    candidate_importance["selected_for_ranked_add"] = candidate_importance["feature"].isin(ranked_set)

    if verbose:
        print()
        print(f"[proxy-ranking round {ranking_round}]")
        print(f"current_features:       {len(current_features)}")
        print(f"candidate_pool:         {len(candidate_pool)}")
        print(f"ranked candidates:      {len(ranked_features)}")
        print(f"ranking_score_col:      {ranking_score_col}")

        display_cols = [
            "feature",
            "importance_mean",
            "importance_median",
            "importance_min",
            "importance_std",
            "n_positive_folds",
            "positive_fold_share",
            "stability_score",
            "selected_for_ranked_add",
        ]
        existing_display_cols = [c for c in display_cols if c in candidate_importance.columns]

        if ranked_features and existing_display_cols:
            print("Top proxy candidates:")
            print(
                candidate_importance
                .loc[candidate_importance["feature"].isin(ranked_features), existing_display_cols]
                .sort_values(by=[ranking_score_col, "importance_mean"], ascending=[False, False])
                .head(30)
                .to_string(index=False)
            )
        print()

    return candidate_importance, ranked_features


# =============================================================================
# Backward pruning
# =============================================================================


def _fs_v2_backward_prune_once(
    *,
    fold_cache: Sequence[dict],
    current_features: list[str],
    current_results: dict,
    added_features: list[str],
    categorical_features: Sequence[str],
    target_col: str,
    target_mode: str,
    target_anchor_col: str | None,
    drop_invalid_logdiff_rows: bool,
    catboost_params: dict,
    allow_missing_fold_features: bool,
    clip_pred_lower: float | None,
    log_eps: float,
    constant_strategy: str,
    protected_features: Sequence[str],
    initial_features: Sequence[str],
    pruning_min_improvement: float,
    pruning_allow_neutral: bool,
    pruning_neutral_tolerance: float,
    pruning_max_allowed_fold_degradation: float | None,
    pruning_max_allowed_degradation_by_fold: dict | None,
    pruning_max_features_to_check: int | None,
    pruning_order: str,
    prune_initial_features: bool = False,
    suppress_catboost_warnings: bool,
    verbose: bool,
) -> tuple[list[str], dict, list[str], list[dict]]:
    protected_set = set(protected_features)
    initial_set = set(initial_features)

    # By default, pruning touches only features added by forward selection.
    # In v5 we can also prune baseline/initial features if they are not hard-protected.
    if prune_initial_features:
        prunable_source = list(current_features)
    else:
        prunable_source = list(added_features)

    prunable = []
    for f in prunable_source:
        if f not in current_features:
            continue
        if f in protected_set:
            continue
        if (not prune_initial_features) and f in initial_set:
            continue
        if f not in prunable:
            prunable.append(f)

    if pruning_order == "reverse":
        prunable = list(reversed(prunable))
    elif pruning_order == "forward":
        prunable = list(prunable)
    else:
        raise ValueError("pruning_order должен быть 'reverse' или 'forward'.")

    if pruning_max_features_to_check is not None:
        prunable = prunable[:int(pruning_max_features_to_check)]

    pruning_records = []

    if verbose:
        print()
        print("[backward-pruning]")
        print(f"features before pruning: {len(current_features)}")
        print(f"prunable added features: {len(prunable)}")

    for feature in prunable:
        if feature not in current_features:
            continue

        trial_features = [f for f in current_features if f != feature]

        with _fs_v2_suppress_catboost_output(suppress_catboost_warnings):
            trial_results = evaluate_feature_set(
                fold_cache,
                trial_features,
                categorical_features=categorical_features,
                target_col=target_col,
                target_mode=target_mode,
                target_anchor_col=target_anchor_col,
                drop_invalid_logdiff_rows=drop_invalid_logdiff_rows,
                catboost_params=catboost_params,
                allow_missing_features=allow_missing_fold_features,
                clip_pred_lower=clip_pred_lower,
                log_eps=log_eps,
                constant_strategy=constant_strategy,
                return_models=False,
                verbose=False,
            )

        previous_weighted_mape = float(current_results["weighted_mape"])
        new_weighted_mape = float(trial_results["weighted_mape"])
        improvement = previous_weighted_mape - new_weighted_mape

        n_improved_folds = _count_improved_folds(current_results, trial_results)
        max_degradation, degradation_by_fold = _fs_v2_fold_degradation_info(current_results, trial_results)

        positive_accept = np.isfinite(improvement) and improvement >= float(pruning_min_improvement)
        neutral_accept = (
            bool(pruning_allow_neutral)
            and np.isfinite(improvement)
            and improvement >= -float(pruning_neutral_tolerance)
        )

        eligible = bool(positive_accept or neutral_accept)
        fail_reasons = []

        if not eligible:
            fail_reasons.append("pruning_improvement")

        if pruning_max_allowed_fold_degradation is not None:
            if max_degradation > float(pruning_max_allowed_fold_degradation):
                eligible = False
                fail_reasons.append("pruning_max_allowed_fold_degradation")

        if pruning_max_allowed_degradation_by_fold:
            for fold_name, allowed in pruning_max_allowed_degradation_by_fold.items():
                fold_name = str(fold_name)
                if fold_name in degradation_by_fold and degradation_by_fold[fold_name] > float(allowed):
                    eligible = False
                    fail_reasons.append(f"pruning_max_degradation_by_fold:{fold_name}")

        action = "removed" if eligible else "kept"

        record = {
            "pruned_feature": feature,
            "action": action,
            "previous_weighted_mape": previous_weighted_mape,
            "new_weighted_mape": new_weighted_mape,
            "improvement": float(improvement),
            "n_improved_folds": int(n_improved_folds),
            "max_fold_degradation": float(max_degradation),
            "features_count_before": len(current_features),
            "features_count_after_trial": len(trial_features),
            "fail_reasons": ",".join(fail_reasons),
            "positive_accept": bool(positive_accept),
            "neutral_accept": bool(neutral_accept),
        }

        record.update(_extract_named_fold_values(trial_results, prefix="mape"))
        record.update(_extract_delta_values(current_results, trial_results, prefix="delta"))
        pruning_records.append(record)

        if eligible:
            current_features = trial_features
            current_results = trial_results
            added_features = [f for f in added_features if f != feature]

            if verbose:
                print(
                    f"[backward-pruning] REMOVED: {feature}\n"
                    f"    weighted_mape: {previous_weighted_mape:.6f} -> {new_weighted_mape:.6f}\n"
                    f"    improvement: {improvement:.6f}\n"
                    f"    max_fold_degradation: {max_degradation:.6f}"
                )
        else:
            if verbose:
                print(
                    f"[backward-pruning] kept: {feature}; "
                    f"delta={-improvement:.6f}; fail_reasons={record['fail_reasons']}"
                )

    if verbose:
        print(f"[backward-pruning] features after pruning: {len(current_features)}")
        print()

    return current_features, current_results, added_features, pruning_records


# =============================================================================
# Main function
# =============================================================================


def ranked_add_features_with_proxy_catboost_v2(
    df: pd.DataFrame,
    *,
    base_features: Sequence[str] | None = None,
    protected_features: Sequence[str] | None = None,
    candidate_features: Sequence[str] | None = None,
    categorical_features: Sequence[str] | None = None,

    # Existing project feature pipeline.
    feature_builder_func: Callable | None = None,
    feature_builder_kwargs: dict | None = None,

    # Optional speed-up: reuse already built fold_cache if you have it.
    fold_cache: Sequence[dict] | None = None,

    # Columns.
    target_col: str = "pto",
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    min_year: int = 2023,

    # Target mode.
    target_mode: str = "log_level",
    target_anchor_col: str | None = None,
    drop_invalid_logdiff_rows: bool = True,
    log_eps: float = 1e-9,

    # Temporal CV.
    folds: Sequence[dict] | None = None,
    normalize_fold_weights: bool = True,

    # CatBoost.
    catboost_params: dict | None = None,
    random_seed: int = 42,
    thread_count: int | None = -1,
    task_type: str = "GPU",
    devices: str | None = None,
    force_silent_catboost: bool = True,
    suppress_catboost_warnings: bool = True,
    catboost_metric_period: int | None = 1000,

    # Start logic.
    start_from_empty: bool = True,
    force_include_protected: bool = False,

    # Proxy ranking.
    use_proxy_filter: bool = True,
    precomputed_candidate_importance: pd.DataFrame | None = None,
    top_k_candidates: int | None = 150,
    min_proxy_importance: float | None = None,
    min_proxy_importance_on: str = "importance_mean",
    proxy_importance_type: str = "LossFunctionChange",
    proxy_importance_data: str = "valid",

    # Stability ranking.
    ranking_score_col: str = "stability_score",
    stability_std_penalty: float = 0.50,
    positive_fold_bonus: float = 0.25,

    # Adaptive re-ranking.
    enable_periodic_reranking: bool = True,
    rerank_after_n_added: int = 5,
    rerank_after_n_checked: int | None = None,

    # Ranked-add stopping rules.
    min_improvement: float = 0.01,
    min_improved_folds: int = 2,
    max_features_to_add: int = 30,
    max_candidates_to_check: int | None = None,
    patience_no_improvement: int = 10,
    max_allowed_fold_degradation: float | None = 0.03,
    max_allowed_degradation_by_fold: dict | None = None,

    # Backward floating pruning.
    enable_backward_pruning: bool = True,
    backward_pruning_every_n_added: int = 5,
    backward_pruning_min_improvement: float = 0.0,
    backward_pruning_allow_neutral: bool = True,
    backward_pruning_neutral_tolerance: float = 0.002,
    backward_pruning_max_allowed_fold_degradation: float | None = 0.02,
    backward_pruning_max_allowed_degradation_by_fold: dict | None = None,
    backward_pruning_max_features_to_check: int | None = None,
    backward_pruning_order: str = "reverse",
    backward_pruning_include_initial_features: bool = False,

    # Categorical feature policy.
    use_enhanced_categorical_policy: bool = True,
    force_include_all_categorical_features: bool = True,
    protect_forced_categorical_features: bool = False,
    auto_detect_object_categoricals: bool = False,
    auto_add_default_retail_categoricals: bool = False,
    numeric_categorical_source_cols: Sequence[str] | None = None,
    extra_categorical_source_cols: Sequence[str] | None = None,
    create_numeric_categorical_copies: bool = True,
    categorical_copy_suffix: str = "_cat",
    keep_original_numeric_categoricals: bool = True,
    max_auto_numeric_cat_unique: int = 40,
    include_all_low_cardinality_integer_cols: bool = False,
    categorical_missing_token: str = "__MISSING__",

    # Evaluation behavior.
    allow_missing_fold_features: bool | None = None,
    clip_pred_lower: float | None = 0.0,
    constant_strategy: str = "median",
    forbidden_features: Sequence[str] | None = None,

    # Logging.
    verbose: bool = True,
    suppress_pandas_warnings: bool = True,
    gc_after_each_trial: bool = True,
    return_fold_cache: bool = False,
) -> dict:
    """
    Adaptive ranked-add feature selection with CatBoost proxy importance.

    Main improvements over ranked_add_features_with_proxy_catboost:
    ---------------------------------------------------------------
    1. Periodic proxy re-ranking:
       importance is recomputed after every N added features and/or every N checked
       candidates, using the current selected feature set as fixed_features.

    2. Stability-aware ranking:
       candidate_importance receives additional columns:
           importance_min
           importance_median
           importance_std
           n_positive_folds
           positive_fold_share
           stability_score

       By default candidates are sorted by stability_score instead of raw
       importance_mean.

    3. Floating backward pruning:
       after every N accepted features, the function tries remove-one pruning on
       added features. A feature is removed if deleting it improves weighted MAPE
       or almost does not hurt it under fold-degradation constraints.

    4. Fold weights are normalized:
       if folds=None, the March-2025-oriented scheme is used.
    """

    if suppress_pandas_warnings:
        try:
            from pandas.errors import PerformanceWarning
            warnings.filterwarnings("ignore", category=PerformanceWarning)
        except Exception:
            pass

        warnings.filterwarnings(
            "ignore",
            category=DeprecationWarning,
            message=".*is_categorical_dtype.*",
        )

    if folds is None:
        folds = make_march_2025_feature_selection_folds(normalize=False)

    folds = _normalize_folds(folds)

    if normalize_fold_weights:
        folds = _fs_v2_normalize_fold_weights(folds)

    base_features = _unique_keep_order(base_features)
    protected_features = _unique_keep_order(protected_features)
    candidate_features_list = _unique_keep_order(candidate_features)
    categorical_features = _unique_keep_order(categorical_features)

    forbidden_features = set(_fs_v2_unique_keep_order(forbidden_features))
    forbidden_features.add(target_col)
    forbidden_features.update({"__part__", "__row_order__"})

    target_mode = _check_supported_target_mode(target_mode)
    if target_anchor_col is None and target_mode == "growth_log":
        target_anchor_col = _resolve_target_anchor_col(target_col=target_col)

    if proxy_importance_type != "LossFunctionChange":
        raise ValueError(
            "proxy_importance_type должен быть только 'LossFunctionChange'. "
            "PredictionValuesChange здесь намеренно не используется."
        )

    catboost_params = _normalize_catboost_params(
        catboost_params,
        random_seed=random_seed,
        thread_count=thread_count,
        task_type=task_type,
        devices=devices,
    )
    catboost_params = _fs_v2_prepare_catboost_params(
        catboost_params,
        force_silent_catboost=force_silent_catboost,
        catboost_metric_period=catboost_metric_period,
    )

    if allow_missing_fold_features is None:
        allow_missing_fold_features = feature_builder_func is not None

    if verbose:
        print("=" * 100)
        print("ADAPTIVE RANKED-ADD FEATURE SELECTION WITH STABILITY RANKING + FLOATING PRUNING")
        print("=" * 100)
        print(f"feature_builder_mode:        {feature_builder_func is not None}")
        print(f"external_fold_cache:         {fold_cache is not None}")
        print(f"target_mode:                 {target_mode}")
        print(f"target_anchor_col:           {target_anchor_col}")
        print(f"drop_invalid_logdiff_rows:   {drop_invalid_logdiff_rows}")
        print(f"task_type:                   {catboost_params.get('task_type')}")
        print(f"force_silent_catboost:       {force_silent_catboost}")
        print(f"metric_period:               {catboost_params.get('metric_period')}")
        print(f"start_from_empty:            {start_from_empty}")
        print(f"force_include_protected:     {force_include_protected}")
        print(f"ranking_score_col:           {ranking_score_col}")
        print(f"periodic_reranking:          {enable_periodic_reranking}")
        print(f"rerank_after_n_added:        {rerank_after_n_added}")
        print(f"rerank_after_n_checked:      {rerank_after_n_checked}")
        print(f"backward_pruning:            {enable_backward_pruning}")
        print(f"backward_every_n_added:      {backward_pruning_every_n_added}")
        print(f"patience_no_improvement:     {patience_no_improvement}")
        print(f"raw base_features:           {len(base_features)}")
        print(f"raw protected_features:      {len(protected_features)}")
        print(f"raw candidate_features:      {None if candidate_features is None else len(candidate_features_list)}")
        print(f"enhanced_cat_policy:         {use_enhanced_categorical_policy}")
        print(f"force_include_all_cats:      {force_include_all_categorical_features}")
        print(f"protect_forced_cats:         {protect_forced_categorical_features}")
        print(f"prune_initial_features:      {backward_pruning_include_initial_features}")
        print(f"numeric_cat_copies:          {create_numeric_categorical_copies}")
        print()
        print("[folds]")
        print(pd.DataFrame(folds)[["fold_name", "valid_year", "valid_month", "weight"]].to_string(index=False))
        print()

    # -------------------------------------------------------------------------
    # 1. Build or reuse fold cache.
    # -------------------------------------------------------------------------
    if fold_cache is None:
        fold_cache = build_fold_cache(
            df,
            folds=folds,
            feature_builder_func=feature_builder_func,
            feature_builder_kwargs=feature_builder_kwargs,
            id_col=id_col,
            year_col=year_col,
            month_col=month_col,
            time_col=time_col,
            target_col=target_col,
            min_year=min_year,
            defragment=True,
            verbose=verbose,
        )
    else:
        fold_cache = list(fold_cache)

    if normalize_fold_weights:
        fold_cache = _fs_v2_normalize_fold_cache_weights(fold_cache)

    categorical_policy_meta = pd.DataFrame()
    enhanced_categorical_features = []
    forced_categorical_candidate_features = []

    if use_enhanced_categorical_policy:
        (
            fold_cache,
            enhanced_categorical_features,
            categorical_policy_meta,
        ) = _fs_v3_apply_categorical_policy_to_fold_cache(
            fold_cache,
            explicit_categorical_features=categorical_features,
            id_col=id_col,
            year_col=year_col,
            month_col=month_col,
            target_col=target_col,
            auto_detect_object_categoricals=auto_detect_object_categoricals,
            auto_add_default_retail_categoricals=auto_add_default_retail_categoricals,
            numeric_categorical_source_cols=numeric_categorical_source_cols,
            extra_categorical_source_cols=extra_categorical_source_cols,
            create_numeric_categorical_copies=create_numeric_categorical_copies,
            categorical_copy_suffix=categorical_copy_suffix,
            keep_original_numeric_categoricals=keep_original_numeric_categoricals,
            max_auto_numeric_cat_unique=max_auto_numeric_cat_unique,
            include_all_low_cardinality_integer_cols=include_all_low_cardinality_integer_cols,
            missing_token=categorical_missing_token,
            verbose=verbose,
        )

        categorical_features = _unique_keep_order(
            list(categorical_features) + list(enhanced_categorical_features)
        )

        forced_categorical_candidate_features = []

        if force_include_all_categorical_features:
            # v6 semantics:
            # categorical representations are scored as candidates, not forced into baseline.
            # This keeps the initial set small and lets CV decide whether each categorical
            # original/copy is useful. Numeric categorical source columns can therefore be
            # tested both as raw numeric features and as '<col>_cat' CatBoost categorical copies.
            source_categorical_candidates = _unique_keep_order(
                list(extra_categorical_source_cols or [])
                + list(numeric_categorical_source_cols or [])
                + list(categorical_features)
            )
            source_categorical_candidates = [
                f for f in source_categorical_candidates
                if f != target_col and not str(f).startswith("__")
            ]
            source_categorical_candidates = _drop_features_missing_everywhere(
                source_categorical_candidates,
                fold_cache,
                feature_role="categorical_source_candidates",
                strict=False,
            )

            forced_categorical_candidate_features = _unique_keep_order(
                list(source_categorical_candidates) + list(enhanced_categorical_features)
            )

            if protect_forced_categorical_features:
                protected_features = _unique_keep_order(
                    list(protected_features) + list(forced_categorical_candidate_features)
                )

    # -------------------------------------------------------------------------
    # 2. Initial feature set and candidate pool.
    # -------------------------------------------------------------------------
    if start_from_empty:
        if force_include_protected:
            initial_features = list(protected_features)
            candidate_pool_raw = _unique_keep_order(base_features + candidate_features_list)
        else:
            initial_features = []
            candidate_pool_raw = _unique_keep_order(protected_features + base_features + candidate_features_list)
    else:
        initial_features = _unique_keep_order(protected_features + base_features)
        candidate_pool_raw = _unique_keep_order(candidate_features_list)

    if target_col in initial_features:
        raise ValueError(f"target_col='{target_col}' попал в initial_features. Это leakage.")

    initial_features = [f for f in initial_features if f not in forbidden_features]

    initial_features = _drop_features_missing_everywhere(
        initial_features,
        fold_cache,
        feature_role="initial_features",
        strict=False,
    )

    if not candidate_pool_raw and candidate_features is None:
        all_fold_cols = []
        for fd in fold_cache:
            all_fold_cols.extend(list(fd["train_df"].columns))
            all_fold_cols.extend(list(fd["valid_df"].columns))

        candidate_pool_raw = [c for c in _unique_keep_order(all_fold_cols) if c not in forbidden_features]

        warnings.warn(
            "candidate_features=None: кандидаты автоматически взяты из fold data. "
            "Лучше передавать candidate_features явно."
        )

    if force_include_all_categorical_features and forced_categorical_candidate_features:
        # Add categorical originals/copies to the scoring pool after the optional
        # automatic candidate discovery. This avoids suppressing auto-discovery when
        # candidate_features=None.
        candidate_pool_raw = _unique_keep_order(
            list(candidate_pool_raw) + list(forced_categorical_candidate_features)
        )

    candidate_pool = _unique_keep_order(candidate_pool_raw)

    leaked = sorted(set(candidate_pool).intersection(forbidden_features))
    if leaked:
        warnings.warn(
            f"Из candidate_pool удалены forbidden_features: {leaked[:20]}"
            + ("..." if len(leaked) > 20 else "")
        )
        candidate_pool = [c for c in candidate_pool if c not in forbidden_features]

    initial_set = set(initial_features)
    candidate_pool = [c for c in candidate_pool if c not in initial_set]

    candidate_pool = _drop_features_missing_everywhere(
        candidate_pool,
        fold_cache,
        feature_role="candidate_pool",
        strict=False,
    )

    if verbose:
        print()
        print("[feature-lists]")
        print(f"initial_features: {len(initial_features)}")
        print(f"candidate_pool:   {len(candidate_pool)}")
        print()

    # -------------------------------------------------------------------------
    # 3. Baseline evaluation.
    # -------------------------------------------------------------------------
    if verbose:
        print("[baseline] evaluating initial feature set...")

    with _fs_v2_suppress_catboost_output(suppress_catboost_warnings):
        baseline_results = evaluate_feature_set(
            fold_cache,
            initial_features,
            categorical_features=categorical_features,
            target_col=target_col,
            target_mode=target_mode,
            target_anchor_col=target_anchor_col,
            drop_invalid_logdiff_rows=drop_invalid_logdiff_rows,
            catboost_params=catboost_params,
            allow_missing_features=allow_missing_fold_features,
            clip_pred_lower=clip_pred_lower,
            log_eps=log_eps,
            constant_strategy=constant_strategy,
            return_models=False,
            verbose=False,
        )

    current_features = list(initial_features)
    current_results = baseline_results
    added_features = []

    if verbose:
        print(
            f"[baseline] weighted_mape={baseline_results['weighted_mape']:.6f}, "
            f"score={baseline_results['competition_score']:.6f}"
        )
        print(
            baseline_results["fold_results"][["fold_name", "mape", "competition_score", "n_features"]]
            .to_string(index=False)
        )
        print()

    # -------------------------------------------------------------------------
    # 4. Adaptive ranked-add with periodic re-ranking.
    # -------------------------------------------------------------------------
    remaining_candidates = list(candidate_pool)

    selection_records = []
    trial_records = []
    pruning_records_all = []
    candidate_importance_history = []

    no_improvement_streak = 0
    checked_candidates = 0
    ranking_round = 0
    last_candidate_importance = pd.DataFrame()
    stop_reason = None

    while remaining_candidates:
        if len(added_features) >= int(max_features_to_add):
            stop_reason = "max_features_to_add"
            if verbose:
                print("[ranked-add] max_features_to_add reached. Stop.")
            break

        if no_improvement_streak >= int(patience_no_improvement):
            stop_reason = "patience_no_improvement"
            if verbose:
                print(f"[ranked-add] patience_no_improvement={patience_no_improvement} reached. Stop.")
            break

        if max_candidates_to_check is not None and checked_candidates >= int(max_candidates_to_check):
            stop_reason = "max_candidates_to_check"
            if verbose:
                print("[ranked-add] max_candidates_to_check reached. Stop.")
            break

        ranking_round += 1

        last_candidate_importance, ranked_features_round = _fs_v2_compute_proxy_ranking(
            fold_cache=fold_cache,
            current_features=current_features,
            candidate_pool=remaining_candidates,
            categorical_features=categorical_features,
            target_col=target_col,
            target_mode=target_mode,
            target_anchor_col=target_anchor_col,
            drop_invalid_logdiff_rows=drop_invalid_logdiff_rows,
            catboost_params=catboost_params,
            allow_missing_fold_features=allow_missing_fold_features,
            proxy_importance_type=proxy_importance_type,
            proxy_importance_data=proxy_importance_data,
            log_eps=log_eps,
            use_proxy_filter=use_proxy_filter,
            precomputed_candidate_importance=precomputed_candidate_importance,
            ranking_round=ranking_round,
            top_k_candidates=top_k_candidates,
            min_proxy_importance=min_proxy_importance,
            min_proxy_importance_on=min_proxy_importance_on,
            ranking_score_col=ranking_score_col,
            stability_std_penalty=stability_std_penalty,
            positive_fold_bonus=positive_fold_bonus,
            suppress_catboost_warnings=suppress_catboost_warnings,
            verbose=verbose,
        )

        candidate_importance_history.append(last_candidate_importance.copy())

        if not ranked_features_round:
            stop_reason = "no_ranked_candidates"
            if verbose:
                print("[ranked-add] no ranked candidates left. Stop.")
            break

        added_this_round = 0
        checked_this_round = 0

        for rank_in_round, feature in enumerate(ranked_features_round, start=1):
            if feature not in remaining_candidates:
                continue

            if len(added_features) >= int(max_features_to_add):
                stop_reason = "max_features_to_add"
                break

            if no_improvement_streak >= int(patience_no_improvement):
                stop_reason = "patience_no_improvement"
                break

            if max_candidates_to_check is not None and checked_candidates >= int(max_candidates_to_check):
                stop_reason = "max_candidates_to_check"
                break

            checked_candidates += 1
            checked_this_round += 1

            previous_weighted_mape = float(current_results["weighted_mape"])
            trial_features = current_features + [feature]

            if verbose:
                print("-" * 100)
                print(
                    f"[ranked-add] round={ranking_round}, rank_in_round={rank_in_round}, "
                    f"global_checked={checked_candidates}, feature={feature}"
                )
                print(
                    f"current_features={len(current_features)}, "
                    f"added_features={len(added_features)}, "
                    f"current_weighted_mape={previous_weighted_mape:.6f}, "
                    f"no_improvement_streak={no_improvement_streak}"
                )

            try:
                with _fs_v2_suppress_catboost_output(suppress_catboost_warnings):
                    trial_results = evaluate_feature_set(
                        fold_cache,
                        trial_features,
                        categorical_features=categorical_features,
                        target_col=target_col,
                        target_mode=target_mode,
                        target_anchor_col=target_anchor_col,
                        drop_invalid_logdiff_rows=drop_invalid_logdiff_rows,
                        catboost_params=catboost_params,
                        allow_missing_features=allow_missing_fold_features,
                        clip_pred_lower=clip_pred_lower,
                        log_eps=log_eps,
                        constant_strategy=constant_strategy,
                        return_models=False,
                        verbose=False,
                    )

                eligible, fail_reasons, meta = _fs_v2_check_acceptance(
                    current_results,
                    trial_results,
                    min_improvement=min_improvement,
                    min_improved_folds=min_improved_folds,
                    max_allowed_fold_degradation=max_allowed_fold_degradation,
                    max_allowed_degradation_by_fold=max_allowed_degradation_by_fold,
                )

                action = "added" if eligible else "skipped"

                record = {
                    "ranking_round": ranking_round,
                    "rank_in_round": rank_in_round,
                    "checked_candidate_number": checked_candidates,
                    "trial_feature": feature,
                    "action": action,
                    "previous_weighted_mape": meta["previous_weighted_mape"],
                    "new_weighted_mape": meta["new_weighted_mape"],
                    "improvement": meta["improvement"],
                    "n_improved_folds": meta["n_improved_folds"],
                    "max_fold_degradation": meta["max_fold_degradation"],
                    "eligible": eligible,
                    "fail_reasons": ",".join(fail_reasons),
                    "features_count_before": len(current_features),
                    "features_count_after_trial": len(trial_features),
                    "added_features_count_before": len(added_features),
                    "target_mode": target_mode,
                    "no_improvement_streak_before": no_improvement_streak,
                }

                record.update(_extract_named_fold_values(trial_results, prefix="mape"))
                record.update(_extract_delta_values(current_results, trial_results, prefix="delta"))

                if eligible:
                    current_features.append(feature)
                    added_features.append(feature)
                    current_results = trial_results

                    remaining_candidates = [f for f in remaining_candidates if f != feature]

                    added_this_round += 1
                    no_improvement_streak = 0

                    history_record = {
                        "add_order": len(added_features),
                        "ranking_round": ranking_round,
                        "rank_in_round": rank_in_round,
                        "added_feature": feature,
                        "features_count": len(current_features),
                        "previous_weighted_mape": meta["previous_weighted_mape"],
                        "new_weighted_mape": meta["new_weighted_mape"],
                        "improvement": meta["improvement"],
                        "n_improved_folds": meta["n_improved_folds"],
                        "max_fold_degradation": meta["max_fold_degradation"],
                        "target_mode": target_mode,
                    }

                    for k, v in record.items():
                        if k.startswith("mape_") or k.startswith("delta_"):
                            history_record[k] = v

                    selection_records.append(history_record)

                    if verbose:
                        print(
                            f"[ranked-add] ADDED: {feature}\n"
                            f"    weighted_mape: {meta['previous_weighted_mape']:.6f} -> {meta['new_weighted_mape']:.6f}\n"
                            f"    improvement: {meta['improvement']:.6f}\n"
                            f"    improved_folds: {meta['n_improved_folds']}\n"
                            f"    max_fold_degradation: {meta['max_fold_degradation']:.6f}"
                        )

                    if (
                        enable_backward_pruning
                        and backward_pruning_every_n_added is not None
                        and int(backward_pruning_every_n_added) > 0
                        and len(added_features) > 0
                        and len(added_features) % int(backward_pruning_every_n_added) == 0
                    ):
                        (
                            current_features,
                            current_results,
                            added_features,
                            pruning_records,
                        ) = _fs_v2_backward_prune_once(
                            fold_cache=fold_cache,
                            current_features=current_features,
                            current_results=current_results,
                            added_features=added_features,
                            categorical_features=categorical_features,
                            target_col=target_col,
                            target_mode=target_mode,
                            target_anchor_col=target_anchor_col,
                            drop_invalid_logdiff_rows=drop_invalid_logdiff_rows,
                            catboost_params=catboost_params,
                            allow_missing_fold_features=allow_missing_fold_features,
                            clip_pred_lower=clip_pred_lower,
                            log_eps=log_eps,
                            constant_strategy=constant_strategy,
                            protected_features=protected_features,
                            initial_features=initial_features,
                            pruning_min_improvement=backward_pruning_min_improvement,
                            pruning_allow_neutral=backward_pruning_allow_neutral,
                            pruning_neutral_tolerance=backward_pruning_neutral_tolerance,
                            pruning_max_allowed_fold_degradation=backward_pruning_max_allowed_fold_degradation,
                            pruning_max_allowed_degradation_by_fold=backward_pruning_max_allowed_degradation_by_fold,
                            pruning_max_features_to_check=backward_pruning_max_features_to_check,
                            pruning_order=backward_pruning_order,
                            prune_initial_features=backward_pruning_include_initial_features,
                            suppress_catboost_warnings=suppress_catboost_warnings,
                            verbose=verbose,
                        )

                        for pr in pruning_records:
                            pr["trigger_add_order"] = len(added_features)
                            pr["ranking_round"] = ranking_round
                            pruning_records_all.append(pr)

                else:
                    remaining_candidates = [f for f in remaining_candidates if f != feature]
                    no_improvement_streak += 1

                    if verbose:
                        print(
                            f"[ranked-add] SKIPPED: {feature}\n"
                            f"    weighted_mape: {meta['previous_weighted_mape']:.6f} -> {meta['new_weighted_mape']:.6f}\n"
                            f"    improvement: {meta['improvement']:.6f}\n"
                            f"    improved_folds: {meta['n_improved_folds']}\n"
                            f"    max_fold_degradation: {meta['max_fold_degradation']:.6f}\n"
                            f"    fail_reasons: {record['fail_reasons']}\n"
                            f"    no_improvement_streak: {no_improvement_streak}"
                        )

                record["no_improvement_streak_after"] = no_improvement_streak
                trial_records.append(record)

            except Exception as e:
                remaining_candidates = [f for f in remaining_candidates if f != feature]
                no_improvement_streak += 1

                record = {
                    "ranking_round": ranking_round,
                    "rank_in_round": rank_in_round,
                    "checked_candidate_number": checked_candidates,
                    "trial_feature": feature,
                    "action": "error",
                    "previous_weighted_mape": previous_weighted_mape,
                    "new_weighted_mape": np.nan,
                    "improvement": np.nan,
                    "n_improved_folds": 0,
                    "max_fold_degradation": np.nan,
                    "eligible": False,
                    "fail_reasons": f"error: {repr(e)}",
                    "features_count_before": len(current_features),
                    "features_count_after_trial": len(trial_features),
                    "added_features_count_before": len(added_features),
                    "target_mode": target_mode,
                    "no_improvement_streak_before": no_improvement_streak - 1,
                    "no_improvement_streak_after": no_improvement_streak,
                }

                trial_records.append(record)

                if verbose:
                    print(f"[ranked-add] ERROR: {feature}; error={repr(e)}")
                    print(f"    no_improvement_streak: {no_improvement_streak}")

            if gc_after_each_trial:
                gc.collect()

            if not enable_periodic_reranking:
                continue

            need_rerank_by_added = (
                rerank_after_n_added is not None
                and int(rerank_after_n_added) > 0
                and added_this_round >= int(rerank_after_n_added)
            )

            need_rerank_by_checked = (
                rerank_after_n_checked is not None
                and int(rerank_after_n_checked) > 0
                and checked_this_round >= int(rerank_after_n_checked)
            )

            if need_rerank_by_added or need_rerank_by_checked:
                if verbose:
                    print(
                        f"[ranked-add] trigger re-ranking: "
                        f"added_this_round={added_this_round}, "
                        f"checked_this_round={checked_this_round}"
                    )
                break

        if stop_reason is not None:
            break

        if checked_this_round == 0:
            stop_reason = "no_candidates_checked_in_round"
            break

        if not enable_periodic_reranking:
            stop_reason = "single_pass_finished"
            break

    if stop_reason is None:
        stop_reason = "candidate_pool_exhausted"

    # Final backward pruning pass.
    if enable_backward_pruning and added_features:
        if verbose:
            print("[final-pruning] running final backward pruning pass...")

        (
            current_features,
            current_results,
            added_features,
            pruning_records,
        ) = _fs_v2_backward_prune_once(
            fold_cache=fold_cache,
            current_features=current_features,
            current_results=current_results,
            added_features=added_features,
            categorical_features=categorical_features,
            target_col=target_col,
            target_mode=target_mode,
            target_anchor_col=target_anchor_col,
            drop_invalid_logdiff_rows=drop_invalid_logdiff_rows,
            catboost_params=catboost_params,
            allow_missing_fold_features=allow_missing_fold_features,
            clip_pred_lower=clip_pred_lower,
            log_eps=log_eps,
            constant_strategy=constant_strategy,
            protected_features=protected_features,
            initial_features=initial_features,
            pruning_min_improvement=backward_pruning_min_improvement,
            pruning_allow_neutral=backward_pruning_allow_neutral,
            pruning_neutral_tolerance=backward_pruning_neutral_tolerance,
            pruning_max_allowed_fold_degradation=backward_pruning_max_allowed_fold_degradation,
            pruning_max_allowed_degradation_by_fold=backward_pruning_max_allowed_degradation_by_fold,
            pruning_max_features_to_check=backward_pruning_max_features_to_check,
            pruning_order=backward_pruning_order,
            suppress_catboost_warnings=suppress_catboost_warnings,
            verbose=verbose,
        )

        for pr in pruning_records:
            pr["trigger_add_order"] = len(added_features)
            pr["ranking_round"] = ranking_round
            pr["is_final_pruning"] = True
            pruning_records_all.append(pr)

    selection_history = pd.DataFrame(selection_records)
    trial_history = pd.DataFrame(trial_records)
    pruning_history = pd.DataFrame(pruning_records_all)

    if candidate_importance_history:
        candidate_importance_history_df = pd.concat(
            candidate_importance_history,
            ignore_index=True,
            sort=False,
        )
    else:
        candidate_importance_history_df = pd.DataFrame()

    final_results = current_results

    checked_set = set(trial_history["trial_feature"]) if not trial_history.empty else set()
    added_set = set(added_features)

    remaining_features = [
        f for f in remaining_candidates
        if f not in checked_set and f not in added_set and f not in set(current_features)
    ]

    fold_cache_meta = pd.DataFrame(
        [
            {
                "fold_name": fd["fold_name"],
                "valid_year": fd["valid_year"],
                "valid_month": fd["valid_month"],
                "weight": fd["weight"],
                "train_shape": fd["train_df"].shape,
                "valid_shape": fd["valid_df"].shape,
                "n_fold_categorical_features": len(fd.get("fold_categorical_features", [])),
            }
            for fd in fold_cache
        ]
    )

    if verbose:
        print("=" * 100)
        print("[done]")
        print(f"stop_reason:            {stop_reason}")
        print(f"ranking_rounds:         {ranking_round}")
        print(f"initial features:       {len(initial_features)}")
        print(f"initial candidates:     {len(candidate_pool)}")
        print(f"checked candidates:     {checked_candidates}")
        print(f"added features:         {len(added_features)}")
        print(f"final features:         {len(current_features)}")
        print(f"remaining candidates:   {len(remaining_features)}")
        print(
            f"weighted_mape:          "
            f"{baseline_results['weighted_mape']:.6f} -> {final_results['weighted_mape']:.6f}"
        )
        print(
            f"score:                  "
            f"{baseline_results['competition_score']:.6f} -> {final_results['competition_score']:.6f}"
        )
        print("=" * 100)

    if not last_candidate_importance.empty and "selected_for_ranked_add" in last_candidate_importance.columns:
        ranked_features_last_round = last_candidate_importance.loc[
            last_candidate_importance["selected_for_ranked_add"].fillna(False),
            "feature",
        ].tolist()
    else:
        ranked_features_last_round = []

    result = {
        "selected_features": current_features,
        "initial_features": initial_features,
        "added_features": added_features,
        "ranked_features_last_round": ranked_features_last_round,
        "remaining_features": remaining_features,
        "selection_history": selection_history,
        "trial_history": trial_history,
        "pruning_history": pruning_history,
        "candidate_importance": candidate_importance_history_df,
        "last_candidate_importance": last_candidate_importance,
        "baseline_results": baseline_results,
        "final_results": final_results,
        "fold_cache_meta": fold_cache_meta,
        "stop_reason": stop_reason,
        "ranking_rounds": ranking_round,
        "catboost_params_used": catboost_params,
        "target_mode": target_mode,
        "target_anchor_col": target_anchor_col,
        "drop_invalid_logdiff_rows": drop_invalid_logdiff_rows,
        "categorical_features_used": categorical_features,
        "enhanced_categorical_features": enhanced_categorical_features,
        "forced_categorical_candidate_features": forced_categorical_candidate_features,
        "categorical_policy_meta": categorical_policy_meta,
    }

    if return_fold_cache:
        result["fold_cache"] = fold_cache

    return result



# =============================================================================
# Current-notebook convenience wrappers
# =============================================================================


def make_x5_feature_selection_builder_kwargs() -> dict:
    """
    Feature-builder kwargs synchronized with the final holdout pipeline cell
    in another-x5-time-series.ipynb.
    """
    return {
        "base_feature_kwargs": {
            "target_history_scale": "log",
            # Explicitly keep lag_1 because growth_log target uses it as a leak-free anchor.
            "target_lags": (1, 2, 3, 6, 12, 13),
            "monthly_history_scale": "raw",
            "target_rolling_windows": (3, 6, 12),
            "monthly_cross_lags": (1, 3, 12),
            "monthly_cross_feature_types": (
                "promo_share",
                "promo_x_items",
                "items_x_hours",
                "cancellations_per_item",
                "cancellations_per_hour",
            ),
            # Keep year so year/year_cat can be scored by the selector.
            # Current monthly operational columns are still dropped by default
            # because they are unknown for the future month; their lagged versions
            # remain available if the feature builder creates them.
            "drop_current_monthly_cols": True,
            "drop_current_monthly_derived_cols": True,
            "drop_year_col": False,
            "drop_obvious_duplicates": False,
            "fill_history_na": False,
            "downcast_float_features": True,
        }
    }



def run_x5_ranked_add_feature_selection_v6(
    *,
    df: pd.DataFrame,
    feature_builder_func: Callable,
    vladimir_features: Sequence[str] | None = None,
    hard_protected_features: Sequence[str] | None = None,
    candidate_features: Sequence[str] | None = None,
    feature_builder_kwargs: dict | None = None,
    catboost_params: dict | None = None,
    include_current_working_hours_per_day: bool = False,
    include_working_hours_lag_categoricals: bool = True,
    target_mode: str = "log_level",
    target_anchor_col: str | None = None,
    drop_invalid_logdiff_rows: bool = True,
    task_type: str = "GPU",
    devices: str | None = "0",
    verbose: bool = True,
    return_fold_cache: bool = True,
    **selector_kwargs,
) -> dict:
    """
    Safe launch wrapper for the current notebook.

    It uses the exact categorical source columns requested by the user and the
    same feature_builder_kwargs as the final holdout pipeline unless overridden.
    Any selector parameter can still be overridden via **selector_kwargs.
    """
    if feature_builder_kwargs is None:
        feature_builder_kwargs = make_x5_feature_selection_builder_kwargs()

    target_mode = _check_supported_target_mode(target_mode)
    if target_anchor_col is None and target_mode == "growth_log":
        target_anchor_col = _resolve_target_anchor_col(target_col="pto")

    if catboost_params is None:
        # Для log-diff таргета MAPE в model-space некорректен:
        # y_model может быть около нуля/отрицательным. Поэтому для proxy-CatBoost
        # используем MAE, а итоговую оценку всё равно считаем как MAPE на pto scale.
        default_eval_metric = "MAE" if target_mode == "growth_log" else "MAPE"
        catboost_params = {
            "iterations": 500,
            "learning_rate": 0.05,
            "depth": 5,
            "loss_function": "MAE",
            "eval_metric": default_eval_metric,
            "random_seed": 42,
            "task_type": task_type,
            "devices": devices,
            "allow_writing_files": False,
        }

    retail_categorical_source_cols = make_x5_retail_categorical_source_cols(
        include_current_working_hours_per_day=include_current_working_hours_per_day,
        include_working_hours_lag_categoricals=include_working_hours_lag_categoricals,
    )
    explicit_string_cat_features = make_x5_string_categorical_features()

    # Baseline features are included in the initial model, but are not hard-protected.
    # v6 baseline is deliberately small: only vladimir_features.
    # Categorical originals/copies are scored as candidates via force_include_all_categorical_features.
    baseline_features_for_fs = _unique_keep_order(
        list(vladimir_features or [])
    )

    protected_features_for_fs = _unique_keep_order(
        list(hard_protected_features or [])
    )

    call_kwargs = dict(
        df=df,
        base_features=baseline_features_for_fs,
        protected_features=protected_features_for_fs,
        candidate_features=candidate_features,
        categorical_features=explicit_string_cat_features,
        feature_builder_func=feature_builder_func,
        feature_builder_kwargs=feature_builder_kwargs,
        target_col="pto",
        id_col="new_id",
        year_col="year",
        month_col="month",
        time_col="year_month_index",
        min_year=2023,
        target_mode=target_mode,
        target_anchor_col=target_anchor_col,
        drop_invalid_logdiff_rows=drop_invalid_logdiff_rows,
        log_eps=1e-9,
        folds=None,
        normalize_fold_weights=True,
        catboost_params=catboost_params,
        random_seed=42,
        thread_count=-1,
        task_type=task_type,
        devices=devices,
        force_silent_catboost=True,
        suppress_catboost_warnings=True,
        catboost_metric_period=1000,
        start_from_empty=False,
        force_include_protected=False,
        use_proxy_filter=True,
        top_k_candidates=150,
        min_proxy_importance=None,
        proxy_importance_type="LossFunctionChange",
        proxy_importance_data="valid",
        ranking_score_col="stability_score",
        stability_std_penalty=0.50,
        positive_fold_bonus=0.25,
        enable_periodic_reranking=True,
        rerank_after_n_added=5,
        rerank_after_n_checked=None,
        min_improvement=0.01,
        min_improved_folds=2,
        max_features_to_add=30,
        max_candidates_to_check=None,
        patience_no_improvement=10,
        max_allowed_fold_degradation=0.03,
        enable_backward_pruning=True,
        backward_pruning_every_n_added=5,
        backward_pruning_min_improvement=0.0,
        backward_pruning_allow_neutral=True,
        backward_pruning_neutral_tolerance=0.002,
        backward_pruning_max_allowed_fold_degradation=0.02,
        backward_pruning_max_features_to_check=None,
        backward_pruning_order="reverse",
        use_enhanced_categorical_policy=True,
        force_include_all_categorical_features=True,
        protect_forced_categorical_features=False,
        backward_pruning_include_initial_features=True,
        auto_add_default_retail_categoricals=False,
        auto_detect_object_categoricals=False,
        extra_categorical_source_cols=retail_categorical_source_cols,
        create_numeric_categorical_copies=True,
        keep_original_numeric_categoricals=True,
        max_auto_numeric_cat_unique=40,
        include_all_low_cardinality_integer_cols=False,
        categorical_missing_token="__MISSING__",
        allow_missing_fold_features=True,
        clip_pred_lower=0.0,
        constant_strategy="median",
        verbose=verbose,
        return_fold_cache=return_fold_cache,
    )
    call_kwargs.update(selector_kwargs)

    return ranked_add_features_with_proxy_catboost_v2(**call_kwargs)


# Backward-compatible alias. The implementation is v6 semantics.
run_x5_ranked_add_feature_selection_v5 = run_x5_ranked_add_feature_selection_v6
run_x5_ranked_add_feature_selection_v4 = run_x5_ranked_add_feature_selection_v6

# Запуск

In [ ]:
# ============================================================
# CatBoost params для CPU
# ============================================================

catboost_params_cpu_logdiff = {
    "iterations": 500,
    "learning_rate": 0.05,
    "depth": 5,
    "loss_function": "MAE",
    "eval_metric": "MAE",   # для log_diff лучше MAE, не MAPE
    "random_seed": 42,
    "task_type": "CPU",
    "thread_count": -1,
    "allow_writing_files": False,
}

# ============================================================
# CatBoost params для GPU
# ============================================================

catboost_params_gpu_logdiff = {
    "iterations": 500,
    "learning_rate": 0.05,
    "depth": 5,

    "loss_function": "MAE",
    "eval_metric": "MAE",   # для log_diff лучше MAE, не MAPE

    "random_seed": 42,

    # GPU
    "task_type": "GPU",
    "devices": "0",         # одна GPU, для Kaggle P100 обычно так

    # служебное
    "allow_writing_files": False,
    "verbose": 100,
}


# ============================================================
# Feature builder kwargs
# Важно: нужен pto_lag_1 для восстановления прогноза
# ============================================================

feature_builder_kwargs_logdiff = make_x5_feature_selection_builder_kwargs()

# На всякий случай явно фиксируем лаги таргета.
# Если в твоём build_retail_features_with_split это поддерживается.
feature_builder_kwargs_logdiff["base_feature_kwargs"]["target_lags"] = (1, 2, 3, 6, 12, 13)
feature_builder_kwargs_logdiff["base_feature_kwargs"]["target_history_scale"] = "log"


# ============================================================
# Ranked-add feature selection: log-diff target + CPU
# ============================================================

result_ranked_add_logdiff_gpu = run_x5_ranked_add_feature_selection_v6(
    df=df,
    feature_builder_func=build_retail_features_with_split,
    feature_builder_kwargs=feature_builder_kwargs_logdiff,

    vladimir_features=globals().get("vladimir_features", []),
    candidate_features=globals().get("candidate_features", None),

    # CatBoost / GPU
    catboost_params=catboost_params_gpu_logdiff,
    task_type="GPU",
    devices="0",

    # Target mode
    target_mode="log_diff_lag1",
    target_anchor_col="pto_lag_1",
    drop_invalid_logdiff_rows=True,

    # Forward selection
    top_k_candidates=150,
    min_improvement=0.0001,
    min_improved_folds=2,
    max_features_to_add=50,
    patience_no_improvement=150,
    max_allowed_fold_degradation=2.0,

    # Adaptive reranking
    enable_periodic_reranking=True,
    rerank_after_n_added=5,

    # Backward pruning
    enable_backward_pruning=False,
    backward_pruning_every_n_added=10,
    backward_pruning_min_improvement=0.0,
    backward_pruning_allow_neutral=True,
    backward_pruning_neutral_tolerance=0.003,
    backward_pruning_max_allowed_fold_degradation=0.05,
    backward_pruning_include_initial_features=True,

    verbose=True,
    return_fold_cache=True,
)

selected_features_logdiff_gpu = result_ranked_add_logdiff_gpu["selected_features"]

# ============================================================
# Быстрый просмотр результата
# ============================================================

print("n selected:", len(selected_features_logdiff_gpu))
print("n initial:", len(result_ranked_add_logdiff_gpu.get("initial_features", [])))
print("n added:", len(result_ranked_add_logdiff_gpu.get("added_features", [])))
print("stop_reason:", result_ranked_add_logdiff_gpu.get("stop_reason"))
print("ranking_rounds:", result_ranked_add_logdiff_gpu.get("ranking_rounds"))

print("\nSelected features:")
print(selected_features_logdiff_gpu)

print("\nFinal fold results:")
display(result_ranked_add_logdiff_gpu["final_results"]["fold_results"])

print("\nSelection history:")
display(result_ranked_add_logdiff_gpu["selection_history"])

print("\nTrial history tail:")
display(result_ranked_add_logdiff_gpu["trial_history"].tail(20))

print("\nPruning history:")
display(result_ranked_add_logdiff_gpu["pruning_history"])

print("\nCategorical features used:")
print(result_ranked_add_logdiff_gpu["categorical_features_used"])

print("\nCategorical policy meta:")
display(
    result_ranked_add_logdiff_gpu["categorical_policy_meta"][
        ["source_col", "cat_feature", "representation"]
    ].drop_duplicates()
)

ADAPTIVE RANKED-ADD FEATURE SELECTION WITH STABILITY RANKING + FLOATING PRUNING
feature_builder_mode:        True
external_fold_cache:         False
target_mode:                 growth_log
target_anchor_col:           pto_lag_1
drop_invalid_logdiff_rows:   True
task_type:                   GPU
force_silent_catboost:       True
metric_period:               1000
start_from_empty:            False
force_include_protected:     False
ranking_score_col:           stability_score
periodic_reranking:          True
rerank_after_n_added:        5
rerank_after_n_checked:      None
backward_pruning:            False
backward_every_n_added:      10
patience_no_improvement:     150
raw base_features:           0
raw protected_features:      0
raw candidate_features:      None
enhanced_cat_policy:         True
force_include_all_cats:      True
protect_forced_cats:         False
prune_initial_features:      True
numeric_cat_copies:          True

[folds]
    fold_name  valid_year  valid_month  weight


/tmp/ipykernel_58/4040559098.py:2968: UserWarning:

candidate_features=None: кандидаты автоматически взяты из fold data. Лучше передавать candidate_features явно.



[baseline] weighted_mape=9.019855, score=82.773868
    fold_name   mape  competition_score  n_features
   march_2024  9.745             81.460           0
february_2025  5.994             88.372           0
 january_2025 18.220             66.880           0
   april_2024  6.616             87.206           0
     may_2024  5.858             88.627           0


[proxy-ranking round 1]
current_features:       0
candidate_pool:         396
ranked candidates:      150
ranking_score_col:      stability_score
Top proxy candidates:
                                       feature  importance_mean  importance_median  importance_min  importance_std  n_positive_folds  positive_fold_share  stability_score  selected_for_ranked_add
                      daily_growth_1_2_clipped            0.000              0.000          -0.000           0.000                 4                0.800            0.000                     True
          last_year_anchor_to_day_scaled_ratio            0.001            